# Crawl corpus ViBioMIR trên Kaggle — tự upload Hugging Face

Notebook tự chứa code crawl, không cần clone repo hoặc upload file Python.
Phần chẩn đoán đã tích hợp: Output tự tạo `<run>_diagnostics.zip` để debug.
Bật **Internet** trong cài đặt notebook; dùng CPU, không cần GPU.
Chỉnh ô **1. Cấu hình**, rồi chạy các ô theo thứ tự.

ID bắt đầu và kết thúc đều được tính vào khoảng chọn. Đây là **ID gốc**,
không phải số thứ tự dòng. Giữ nguyên ID/URL nguồn, không dịch hoặc sinh lại text.
Trích HTML/PDF có text; Selenium thử lại URL lỗi, trang JS hoặc text rỗng/QC lỗi và render JavaScript. Chưa có OCR.
Dùng chung engine với `crawl_domain_samples.ipynb`: header trình duyệt,
timeout/retry có ngân sách, giữ lỗi theo từng lần thử và kiểm tra challenge.
Nguồn mới: `links_corpus_vi_en_zh.parquet`, **1.220.598 URL**, cùng snapshot với domain samples.
ID gốc từ 583 đến 7.466.161 và không liên tục; số URL không phải ID cuối.
Tài liệu nguồn: [ViBioMIR](https://huggingface.co/datasets/AIGuruTinix/ViBioMIR).

**Bản tự upload:** `crawl_corpus_kaggle_hf_upload.ipynb`.
Mục 1–9 chạy quy trình crawl/xuất; mục 10 tự upload output đã hoàn tất
lên [AncientWings_R2AI_STAGE3_ds](https://huggingface.co/datasets/AncientWings/AncientWings_R2AI_STAGE3_ds).
Trước Run All: mở **Add-ons > Secrets**, tạo secret **HF_TOKEN** với token Hugging Face
có quyền ghi vào dataset này và bật secret cho notebook. Token không nằm trong code/output.

PubMed dùng EFetch GET theo PMID; các website khác dùng Requests/Selenium.


## 1. Cấu hình — chỉnh ID tại đây

Để cứu run đã crawl xong nhưng audit/export lỗi, đặt POSTPROCESS_ONLY = True,
chỉnh đúng khoảng ID và RESUME_RUN_PATH tới thư mục run đầy đủ trong Kaggle Input
(hoặc giữ None nếu run vẫn ở session hiện tại). Chọn SOURCE_PARQUET nếu nguồn nằm trong Input.
Chạy các ô từ trên xuống: mục 6 tự bỏ qua crawl; mục 7 chạy lại audit/export.
Snapshot crawler/runner/probe, cấu hình crawl và manifest nguồn được giữ nguyên.
ZIP diagnostics chỉ có một số mẫu, không đủ để hậu xử lý toàn run.

Mặc định chọn ID **583–1582**. Để thử trước, đặt `MAX_NEW = 10`;
sau đó đổi thành `None` và chạy lại ô crawl để làm tiếp.
Giữ nguyên cấu hình tải trang/browser và phiên bản code khi resume cùng run.
Với khoảng lớn, chia thành nhiều run nhỏ: crawler hiện tại nạp danh sách công việc vào RAM.

`SOURCE_PARQUET` để trống sẽ tải nguồn đã chốt từ Hugging Face.
Nếu đã thêm Parquet vào Kaggle Input, điền đường dẫn `/kaggle/input/.../links_corpus_vi_en_zh.parquet`.
Run mới dùng nguồn 1.220.598 URL. Resume phải khớp nguồn; nâng cấp từ run nguồn cũ dùng `RECOVERY_RUN_PATH`.
`POSTPROCESS_ONLY` nhận diện hash nguồn cũ trong manifest để audit/export đúng snapshot của run đó.
`RESUME_RUN_PATH` tùy chọn trỏ tới **thư mục run đã giải nén** từ notebook này trong Kaggle Input.
Để nâng cấp run cũ hoặc thử lại lỗi, đặt `RECOVERY_RUN_PATH` và chọn `RUN_TAG` mới.
Recovery dùng lại raw 2xx đã kiểm tra hash, chạy lại extractor/QC, rồi crawl URL còn lỗi.
`RESPECT_ROBOTS = False` tải trực tiếp giống `domain_samples`; đặt `True` để
kiểm tra robots trước các request trang. Trạng thái robots không được chuyển sang Selenium.


In [ ]:
# ===== CHỈNH CẤU HÌNH TẠI ĐÂY =====
ID_FROM = 583
ID_TO = 1582              # Bao gồm ID này
WORKERS = 4
DELAY = 1.0
LOG_EVERY = 300          # Tiến độ mỗi 300 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
MAX_NEW = None            # None: toàn bộ còn lại; 10: thêm tối đa 10 URL
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
RESPECT_ROBOTS = False    # False: giống domain_samples; True: kiểm tra robots
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
LONGCHAU_USE_CURL = True
LONGCHAU_WORKERS = 1
LONGCHAU_DELAY = 1.75
LONGCHAU_ATTEMPTS = 3
LONGCHAU_CONNECT_TIMEOUT = 10.0
LONGCHAU_READ_TIMEOUT = 45.0
LONGCHAU_REQUEST_BUDGET = 180.0
LONGCHAU_RETRY_FAILED_ON_RESUME = True

BROWSER_TEST_MODE = 'failed_only'  # 'off', 'failed_only', 'all'
PUBMED_USE_EFETCH = True  # Mặc định: PubMed đi EFetch GET, ghép theo PMID
PUBMED_EFETCH_BATCH_SIZE = 100
PUBMED_EFETCH_WORKERS = 1
PUBMED_EFETCH_GAP_SECONDS = 0.5
PUBMED_EFETCH_ATTEMPTS = 3
PUBMED_EFETCH_CONNECT_TIMEOUT = 10.0
PUBMED_EFETCH_READ_TIMEOUT = 60.0
PUBMED_EFETCH_BUDGET = 180.0
PUBMED_SELENIUM_FIRST = True  # Chỉ dùng khi PUBMED_USE_EFETCH = False
BROWSER_USER_AGENT = None  # Dùng User-Agent thực của Chrome
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_WAIT_SECONDS = 15.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None      # Selenium Manager tự tìm/cài Chrome; hoặc đường dẫn có sẵn

POSTPROCESS_ONLY = False  # True: chỉ audit/export run cũ, bỏ qua toàn bộ crawl
SOURCE_PARQUET = None     # None: tự tải; hoặc '/kaggle/input/.../links_corpus_vi_en_zh.parquet'
RESUME_RUN_PATH = None    # None: mới/tiếp tục session; hoặc thư mục run trong Kaggle Input
RECOVERY_RUN_PATH = None  # Run cũ để reuse raw thành công, thử lại URL lỗi trong run mới
RUN_TAG = None            # None: ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh; đổi khi muốn run mới

from pathlib import Path
import json
import os
import shutil
import sys

NCBI_EMAIL = os.environ.get('NCBI_EMAIL', '')  # Email liên hệ; có thể điền ở đây
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '')  # Tùy chọn; không ghi vào settings/log

if sys.version_info < (3, 11):
    raise RuntimeError('Code crawl yêu cầu Python 3.11 trở lên.')
WORKDIR = Path('/kaggle/working')
SOURCE_FILENAME = 'links_corpus_vi_en_zh.parquet'
DATASET_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
LEGACY_SOURCE = {
    'filename': 'links_corpus.parquet',
    'revision': '0148f6f80ffafed5c005af6d506ccfd9d3fb47a7',
    'sha256': 'dee1153f5b9d9010dea18e1b7724b4915ff847b711d036bec8b933486802a971',
}

if type(ID_FROM) is not int or type(ID_TO) is not int or ID_FROM < 0 or ID_TO < ID_FROM:
    raise ValueError('ID_FROM và ID_TO phải là số nguyên; ID_TO >= ID_FROM >= 0.')
if type(WORKERS) is not int or WORKERS < 1 or type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('WORKERS và ATTEMPTS phải là số nguyên dương.')
if not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if type(POSTPROCESS_ONLY) is not bool:
    raise ValueError('POSTPROCESS_ONLY phải là bool.')
if POSTPROCESS_ONLY and RECOVERY_RUN_PATH:
    raise ValueError('Hậu xử lý dùng RESUME_RUN_PATH hoặc run hiện tại, không dùng RECOVERY_RUN_PATH.')
if RECOVERY_RUN_PATH and RESUME_RUN_PATH:
    raise ValueError('Chỉ chọn RECOVERY_RUN_PATH hoặc RESUME_RUN_PATH.')
source_tag = f'ids_{ID_FROM}_{ID_TO}_synced_vi_en_zh'
previous_tag = source_tag + ('_efetch' if PUBMED_USE_EFETCH else '')
default_tag = previous_tag + ('_longchau' if LONGCHAU_USE_CURL else '')
if POSTPROCESS_ONLY:
    if RESUME_RUN_PATH:
        default_tag = Path(RESUME_RUN_PATH).name
    elif not (WORKDIR / 'runs' / default_tag).exists():
        default_tag = previous_tag if (WORKDIR / 'runs' / previous_tag).exists() else (source_tag if (WORKDIR / 'runs' / source_tag).exists() else f'ids_{ID_FROM}_{ID_TO}_synced')
tag = RUN_TAG or default_tag + ('_recovery' if RECOVERY_RUN_PATH else '')
if not isinstance(tag, str) or tag in ('.', '..') or any(separator in tag for separator in ('/', chr(92))):
    raise ValueError('RUN_TAG chỉ là tên thư mục, không phải đường dẫn.')
RUN_DIR = WORKDIR / 'runs' / tag
RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if RESUME_RUN_PATH:
    previous = Path(RESUME_RUN_PATH)
    if not (previous / 'manifest.json').is_file():
        raise ValueError('RESUME_RUN_PATH phải chứa manifest.json, records/ và raw/.')
    previous_manifest = json.loads((previous / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and previous_manifest.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Run resume dùng nguồn khác. Nguồn 1.220.598 URL cần run mới; dùng RECOVERY_RUN_PATH để nâng cấp.')
    if not POSTPROCESS_ONLY and PUBMED_USE_EFETCH and (previous / 'corpus_config.json').exists():
        old_settings = json.loads((previous / 'corpus_config.json').read_text(encoding='utf-8')).get('settings', {})
        if old_settings.get('pubmed_use_efetch') is not True:
            raise ValueError('Run cũ chưa dùng EFetch. Chọn RUN_TAG mới và RECOVERY_RUN_PATH.')
    if not RUN_DIR.exists():
        shutil.copytree(previous, RUN_DIR)
    else:
        print('Đã có output trong session; dùng run hiện tại, không copy đè.')
if (RUN_DIR / 'manifest.json').exists():
    if not POSTPROCESS_ONLY and (RUN_DIR / 'crawl_config.json').exists() and not (RUN_DIR / 'corpus_config.json').exists():
        raise ValueError('Run cũ cần nâng cấp: chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    saved = json.loads((RUN_DIR / 'manifest.json').read_text(encoding='utf-8'))
    if not POSTPROCESS_ONLY and saved.get('source_sha256') != SOURCE_SHA256:
        raise ValueError('Nguồn của run cũ khác bản 1.220.598 URL. Chọn RUN_TAG mới và dùng RECOVERY_RUN_PATH.')
    if saved.get('id_start_inclusive') != ID_FROM or saved.get('id_stop_exclusive') != ID_TO + 1:
        raise ValueError('Khoảng ID không khớp run đã có. Chọn RUN_TAG mới.')
    if POSTPROCESS_ONLY and saved.get('source_sha256') == LEGACY_SOURCE['sha256']:
        SOURCE_FILENAME = LEGACY_SOURCE['filename']
        DATASET_REVISION = LEGACY_SOURCE['revision']
        SOURCE_SHA256 = LEGACY_SOURCE['sha256']
        print('Hậu xử lý run cũ: dùng đúng snapshot nguồn gốc trong manifest.')
if POSTPROCESS_ONLY and not all((RUN_DIR / name).exists() for name in (
        'manifest.json', 'sample.jsonl', 'records', 'crawler_snapshot.py')):
    raise ValueError('Hậu xử lý cần thư mục run đầy đủ; ZIP diagnostics không chứa toàn bộ checkpoint/raw.')
SOURCE_URL = f'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/resolve/{DATASET_REVISION}/{SOURCE_FILENAME}'
SOURCE_PATH = WORKDIR / 'data' / SOURCE_FILENAME
print(f'File nguồn: {SOURCE_FILENAME}; revision: {DATASET_REVISION}')
print(f'ID: {ID_FROM} đến {ID_TO}, bao gồm cả hai đầu')
print(f'Output: {RUN_DIR}')


## 2. Cài dependencies

Lượt mới dùng phiên bản tương thích mà pip chọn trong môi trường Kaggle,
Phiên bản thực tế được lưu cùng run.
Khi khôi phục run, notebook cài lại dependencies trực tiếp đã ghi nhận.
Nếu bật Selenium, bước crawl tự kiểm tra thư viện Chrome trên Linux, cài phần thiếu
bằng apt và mở trang thử trước khi tải URL. Kaggle cần Internet để cài package/driver.


In [ ]:
import subprocess
import importlib.metadata

package_names = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'curl_cffi']
if BROWSER_TEST_MODE != 'off' and not POSTPROCESS_ONLY:
    package_names.append('selenium')
requirements_path = RUN_DIR / 'notebook_requirements.txt'
if requirements_path.exists():
    install_args = ['-r', str(requirements_path)]
else:
    install_args = package_names
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'logs').mkdir(exist_ok=True)
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
)
with (RUN_DIR / 'logs' / 'pip_install.log').open('a', encoding='utf-8') as log:
    log.write(installation.stdout or '')
if installation.stdout:
    print(installation.stdout)
if installation.returncode:
    raise subprocess.CalledProcessError(installation.returncode, installation.args, output=installation.stdout)
versions = {name: importlib.metadata.version(name) for name in package_names}
print(json.dumps(versions, indent=2))


## 3. Chuẩn bị code crawl

Ô dưới giải nén snapshot nhúng trong notebook vào output; không tải code từ mạng.
Snapshot crawler/runner/probe đang có được giữ nguyên để resume đúng phiên bản.
Auditor, exporter và diagnostics được cập nhật từ notebook; không thay cấu hình/hash crawl.


In [ ]:
import base64
import hashlib
import zlib

EMBEDDED_FILES = {'audit_snapshot.py': 'c-pmG-EZ93b${1iaaY@doMc4VUT4vcG}?G=Z{ud|HEi!9fL7pAoI5j&I3&#vNfSvxi#+tHKo=<x1Vw;#f?|;dK~l6pQTw3}TIByw|C0XBx%ZNn!<mu1*b49rhrIWkd(YQBU-ahY<|pHE93@gjNhv3JRL-}<6qiXUMo}<H(xQyQLIlYO|3sC<!J&*}ITE=%j^v3brc#z-oaQ1BVIG{sBCir}b8~Yenq_HTiXfk4L0-s>ah}dZm|}8RMrl$AJ>Y&?0bg#$Pkt6<=6V_w(>OY;f0naY1O<QlvIv7rnn&sjW4)@3VsrPWMVctQSx^!rgYzkVnn5}Cmd@&5HOOLKn9C}O!gM4@K^d4^r?c3TWiF-nFqSir>jkc9aSvZ8jN>W^%QTJaea3m^he0^il&JEUmdms;LvXLi;;7tELt-Ei*4_>7Y#!u!dO~wwR-itxW-m83Hb!!c^#>!DcH0|l2>j);tn!2@;l*h%D)^yC!~9a7maa^&wJ4bkovIvf{gu<}X-Hu@%Yv|UX;d4SBOLFiX_UAYUA!dCXF(kOOuEjKkM28t!Sg)ck6QyRrh!Yt+W<A6Jv_bhgZI?{Zz@km(L@%QZT1$N_{@8vyL*_9=6zw7uK|uEFGvgpJ@(YkdXIhK<T8tcP&z#x9*sqkmS&b$lzEi78kDuZCut%vSI)9>?$atpX;=}{9UfWc^Uf4nr$<~Qn90P|D0+Pc#-rtFD6>+yJXFcjQ<8@KA-J3O#b-fWNxojWqAFqe2@Yequ&#}bAJzir&VthdoEhHj)pUgRjzAV^%L<VUlp_sf%FVJ|7KKa7qJ}B4Ih>cWAeLI|QOqFj(p7IQ`m8#f$&nA?_H$s#A(K2wi$n^B453Cg^%PLeF&g2O!9=_>6wXryb}xsOisf@zIu@yd2&&ak(WSwUAslA!A}W~u4?mJ+I2Av6{ITblUGciGj<XBCK?Y9UgrVdWG<f1P49KEkv;**DP47n~-*R}pJdamATwJjuWQ;{RAExQigO*UyUp9+Klr?~Hl#EzqYlPny&i2#Cj~+nV9KCXD^<E)M&^xH&(j`cJv47C>f-IBC$W>!nJk-;)vIW&BD*c96dPZMZyn>nkOy<#eE{e3uLC`WPW5}gEmib%+hu|J8h$xiB=R<_g57@+HBxxPkZaf$1Sd>$#&JURjmg$@i5sM1OIO>J$JGkLsLu}8K<~rMrUTdOFWa&D37X--Zk=W4H8bKLvYDppDZh!jhV=;@0SpXp!h%<v8unsOZ6+V6@NnTb7H!q~-!kR?}S>zXxo~po7NnU`ufDWkDH;cw>gSD1T1&Cw>a~JNutYDpCzFM$@kW7%Y;jSl$UJMha(W7gqV;>z5)iawPzJ}-w#EZO=7jHwEQ3ZK`DoB3RrB5sqSJS8`cEz3lAClCo_-=^OY)WmYO_tV{Ajr2D_hggym~=CeB=g$R5v-faJx{<14;KR}YGAF{jZ{t$6hN<T|JDIN2(=}|6Fr9QnA@?GJ=&fbvd&hv!L$sgT1%jY1EWE%#~L^{MrG!!Q=V*_pWqY+WR0T@_yvv7i{R08B4}5sne9dJcG-^icp(;C%_TbA^|_$Vf+Y0|_+kVv4rYg=fSs!W81EI;p_@CO7vJYIpf9d>$t=%DK~-yGajI{%><Mp+kC@&~gonp(<21ca1#&r-Iq6E@91&ze_j&#S*-6DteT?ukH^DmY={sN~oy!Rw$u>iRICiZ;3t&7DflfdM^$E_Md>Oo!=rF*Ap}pEKz;OjZ>O&KwNu(BIa5dz@Ahm;o`WEA2wkH}!cl9sWmbXn2anX{jMVV@i<>R(5$ev8noV+_8&V2oto+-W>&$n+qeiF~_>uK#lEd-+ohjK9Y84=4TePd^CjT53)Uz5xB>63fUpFjGWCl4PO@5gDwG{Rtfl{0mu4S45R3gf^}y*Vd|XTHIJ+vc+00^0d=6u&S*T`=HeQUw!<dzvAdQ2I8L0UzZ&TO9L(Zt?Sr_GU=KvFvVa(UJ_aRkuY?FV}v}0HO(JZy?yAY}4G35Cd4h?Gun;4;I&>1(mE`Td2E>GZZ1uW+g>BvuedQ+fsOHy>PHrckBrF%)r0s@y=>=)~0Cl8_S_<P3B#yF-4i?((;*kwFa$k1j}hTVRWXdd29puryj^&RFKTw)&goP37BvpEU!x$$(ALRrx}1rm_8nH(B6v%Zvi@za|Te)!;J&B6}TW%;cEF`6#RZI(Y7QTX<e`joUmb}{TdSiN9d<%(b(jhS^Y(CzqrW>puz^<ShUZR&l7FcGG$sh+A{oRi8hMVJ;p3xYYwx9O>ea;ns<{OA(i3RlZMRP!m!>fowr6YP(xxn8_@zmG1Is4S=Vey4;Cj;IZdmw-s8$snw?aOx|X*}aZc@KDqj`EH4b}6?*?t{T4SLN44W93ZvVW{kt(IN{5VY};WVgR>_~H#0z<HX(-(ACY(p>fFuzEGte7H^kj)*N_#92j)2Jv5+gRJMe19ClWAiEWB$H^vt-%%0DNJbRB|W<9Q(i-XFo8Tw6i3?z3Ivq%Eh@H5M-{u03$WCE1&!dk^wF!4bG8prv^)h$HP%&MEw-v#t(d<ir5*A2byVI&{?FtVWPyd$NC0J)SggpVAc~1f5mq_TXe%SDeuz}>D99sO^eNy^RE{y1NS4O2IwnofzYqe};!X~PM%$KmJCv%qJT1$tPIdWd63^Ggne{ZaRTAt%g(mL|O;X~shtHlr`sB%C*rk`0<`_(3M0UC*+zoK;sUlZ5^N(+{u{Muzu*;xW`lw2ZDkJ_=h^QX-TEXsj7}$O+CqXzD9|O0z59C7c4zyUb5j&#7>_8+a1t9TXn=4KLl_{=kA%Sdot76}Jn=zyI8-y)>A-ySTPUtZ@4{UEAUUU%qlpeCOASPdA7!O#+E)#o~Cs5QNj(@tNRJ6C&<TfO%C48H7WYr<EV-t)52}M{#P%f=Bfd^DB9x0bVTjxVqlp{Z+iV8{35g5LBPmi1;lS^5p_Sl8HOU!efj$*Kd1ySAbRSv%;yV~CSaA1n%K<lBZs)VngP<~qT=rD!J%;ESVHBv+qu$Z3*6;83-;nY6Om6V`%)FBi{C{N)W;tf_Af@1ZntvO&abCk*1TVS)gdA-WijAr)AGN1^&<}*B926WL|amc=v0<)H}Xs)8$!40Ld_60_=$Phzn-O|Oq^$vlhKrI!f>H=yRwFY<mgN-iRand8_6&o3{W*FuyljU2zb+EK7uYtCx11sRQ1>OSEQH2%&trn=O=BW=zxp9B$`Hg}ES*o2<3xJwQH5i#6o8|8>%sNmWvf`39XO>JajP9wximOMRm6lvaQpB)!x#TJB)OcUB0N%Q(-B^Vsk83|ZPP!o1IfHI)eAyr+fy26R?A^;JoZeH#F;vou5N<Sy*3hSl7d$a*rWcH$<9dqgY;D2ybLf0{Sw#qIh9rV($1B<2qKXxz%n&>cBTDb+Qp)i5ty^p7D|%t-RL0rR`ONr6x<v9+7j1LN-X9)ttrx}hG={&1kf2abht6eAiss1UFG5ymVv+l<y;8dCk(;Xv5=k^h?UW;7RD@|D+*EVhqk3>tE1|lb8BzqZ40$vffb+9m@gv)4K<w+0#^A3rInHw)%5~n_CTjHTF0MeHB6-j)P*c4d5r7P44N5hU@@rP5oAS#l*wP&idKw4)xMSk613MFNmMd$gN`5?vVP7=K0sg)2;CCL>H37Y_M-K|U-#XexZBe5_B3FAS9i(=YY{wwXZ7orlVlhQYU*qT0EVq?e^UT_VDzyXrLM^^=*qBk)QWSTb{BSu(jFZB~k#<BWN>`<?2YNJ1mwWrxox9?Dg1(SkJ9L~SNiwxOPo3c|!)ztG^mD6cU_n+}`#!klJa?YAb#<BCYT`pTU+QiOGLWuCpgtV57@IDJ`!4ahPbM9o@Jh-$E!69wgjvGO&-r#cDaEjAxvQrT$JfdZ&%3a*NH<W<Q7TpWL3u1~bJeMf@2f_WUX!LNI5?^8h#m?pp2X>)>wHh!9Fvac<kK}upkdU(nx16si%3DV(IdOGTDS3M6=mxw290IOdPJ>HIL``{QECiO-5qffM7qlfZaj{x@F+u;sx(R+B5Jd;`&tO-9I<)`7Q#zBs9Ql6cigO+AQK3OYk`_p<jv->(~t}|7F<R=Tkc9?=r|50G-ChY!qy?>KJY5o!=zIouX9&yhNV6P^zEDK(|D8W7;QBg)SR~~qT~lJMEt{6#Lp`uDb>Ts<sfBquJ_R&>)JJW=*COAZWXJ*>!1@)K>|8t?UouCH?0MLT|+qNbqWJ!zM`o9wS=H6HuMY|xdBbvJCX@f#M01pOr0-F^8U62?`xO`<O4xHVv^}@dx-7~>3|{$HilczZx50&R_8YD9d&2N6wQZ8Fk{W6`cs^q;0Wn;lPk3+n@mCJDh=y&9F*HT=8X8s$%(h>@sb0FOz?ufqB=_oJ6MXktK8kR<hP}AlaMpHxCLzAmguBM+$RTRLkWdWc=yox%cl=M+FC{v2seqkk|Q*T#Yo+aeB+;~jEQ#AI&-PLP3I7OvOij9mbCNrmhqyQxSag7-+gV^4oualnbcc4X({=s4&jn!<yUWhnM}5irkDQ-qoMjVzxnddf4X|}+fY3C<nfL;l{4!s5XYBav2B@Lz4<cqdR-b&?6rLP*nFf85Xol&JFDCg$X=5(ijGy+f}@*rw-Dqa+qjiMMc8D!x@rri6oqDmQ@39pEXB$J=G$RK%VM5=dyEw5Q7ee)EzLVcnbixmrqvCo(&J-DFHcuicpGYOE8r<<SrVWV_sM<Wd+X*d#^y9pt&(~Nc$c7?x>JYT%lBr6(ZQy!bOM>d3|5v%mWsSy7$EqRahzLA?5#qtgl{#Xrk_TQG_5_?Qr1ox7|NPHjZ-=kUB}zr=25FS-{sZ}g@kKiMzhH}hzS0USndNQe^Zngb@NAHH<Kte2$T}E#$HE(pe&Kupp*dRp_O3mSB#+|3~B|O^+TIxkcU(1;_=*UG(Ja4<Me~`)3`YOpyz%t_>1%X58gfKQEv~WF}z1>kn6yKC5KwxZvWsG%hhTU4%RkFtdKDVz?!GiDuthhxlTeltLgr(c+~WL>d3IE?s6W43@D5%8d0c;R7_+d^CnBeU81aATr66Nh|NH3v$By5l|4GAt8yMox`@U(+zh(H`wskg+40vl<=!YyGj(*W-tfv3rml0Bt)g(bD1EA@hnq;+h3}i6E$dIZAFJany!;Pjp6JyKtm4C~*RK!(J`SQCaTHy>`8f!m2hG_=Y)FN1Z0(ay7xFu1y<MnkB>r&}!b+#(@)5lPlF0HQ>W1h($j@g*x~2F-SEah#edqIi2v0OV-{l*7=euw{_jG5Q){Zt6mP>Ft9!FFV%>7xAA5|IZPQj70ECA?a;@??aN^9fN%_$3T)Kl(1>1`~X2>R(z8vWaD>(RoLx@(Aj=s=XTnoOI1tHvF(T!nm|aFEJ{BmUWdRY7Yt12N_tY${RUoU@hj>XRd&f@Sl>#NAC9XofXWQha6B-}vezKd?>8j!+mQizTfD%}YqW30slg2VHoh-3|iByaG}9<j$(o9h0y-S0}U=&+cBU@3Kv<8zN|B>90dYJ9}$1_K{k&k*rA2i#>7cdNkdfyv+m)%d&=#zW4}LIGtm?37Sqv$a-YU8n<rU0NX+>YTdSOP`0}HE^koJT84Vn=(pVLy295wwB7MHLtKO@w?R7hsxqDNAq^sK<Mg&)9o%8nqHG$y+=m*6)lDF@RrI=5VP{$)du!^Nw%NI*l2#_qs6u3IymXGyJHd;L%0qFv9x$qAS>ftI6lE4wjzUTiM$wR&+vhh@l5%*b*Su3!R~(igRQA^tDlB}aE!V=EFT=>Kjc@~++@GfN3iNfUwmu%*ypuOvcW|{z$UuW?{7`q3=XGY$q<K{}Ipy@FdO^~A#$WE;JBac4Ja7%PJ`zxgDoN;>R6Oe|-&~LB>9b3{FFF);mYBz*rlx6W_hRwbCCyl%^^D#3>$9S&Nuy4N>Yk0hXwq_nZmMr-ivs8iA|vIc`h)cgct$W2RPFp*J`(wNtcT>OQlY8yY-wYKR>!289inK^I#M)1-QkcocR(gS4jkvg7KWOAO{k!V`JD(V(|@I)W~yCy-`RZ!XBgE_`DMtxU4DI+8dHax$I<5AE_F!=j=N<!+++~A%PiWvM@j~7CeNy3cU#?CFcOaT0V#2GHbw?uaQpr2lwRA#^0umI2fB__=ituW>~x2?ur-y@WQt+l`xiPVc6JD@k@?^@O-y6FXZCIwhC3N*y4cs?_8m;sVCWjQ2Q}CzKhqu91H3Rj+M$mv;N;enExh4e&5~kpdz=gW+nEFzFX00GJUf|!c3a$)I>7I(lRU_FsJCZ~+$R4Tpkwp|z0z_=ZRxJPr3;UIpfjLI^TE4+{r-CwyDEt3(q?*l@98wUdi_6@?ZTIrUqy&LuipGa)BCZzjj_Ag-s7v+zpq3xQD4*O^53XG6$a<-9Z%f9dh;tZ!RD8*FduoYavD(XOCcz)-u#|S|9~R5?Z17F_PB4pynKzY;*ju7)2rA271HaKLbagoh>*c$)2r9NRqzz2J=1e``I}0NF8`YY6Z(8vp_}aL^&dlV$m>clKf;?=cyxUA=3gRRAcs-Y**XQ8kDG74#NbfGSFit&dE%RYy8O)?EC2JK0yF9en<#+-Kk(q{^}o(&eP5&ZYC_q*Uq`f?SIJaNqRUs_?sm5494uCCYK3d86xtN1oZ7sr?&(+7o1W51-5@{S+|?qvx2un-y*igK{`c?xS)5_En|I&HAra6Mz(h65WY8~YvAam)2~N@t9THv#7?e8&m3(iHj%)OaU%mMSQhFsQpD@?@nnnotelZg|@7IH@yazP7REX@bl2_JP;-AP_6ELb}(ljjA;I<cWpV^O;qQXP3GSn+sMB2?6c42R0&-AHDy4hH!Is^n%vPuhwyLBGX<AtNW=}rfvl?HG&V)OT|_;I6nss3LUl$`5#wZ@xULcd>nG=&&vh7qXa*umAs_L&uhU3i<evO8OXvelG)=IFalQCn*N{73h;P*!TYS%m$eruPJ6|Hzv1H98L};@Yn=!PZI{D{-cUZ2!9W=K!l+JC+2R?E>W-G{f6lqLqDUMjS#dq0yL#&mX;j5ow6D3Qs#&I=JY9H@1<TxtiRy7WXcen#vum6tu3|I4F|=6vR`TG50TDg|IP~zsqc1B$r>!#VI{XF!2v~^()5#kOY(gBPL?PE0jtJF}kzaP8QH1*snLgq<oCd*Fau=hi^DuQ{vm}ujiV5mT3g({vBUz1vBVBgtqXU&N|P@y8l|7i|2&L-uF2TJ{MbCTY^6axO1Ro+~haL7;pw(@&ZH7IQqnuV_sP4tBi!zAD{wvk9`Q&c!dqPeNqgz9$$1E?JszXKW8#NdaxtDSkPAUoId|CtgYH{eZIi1za~SDI$*VfVc6y0C$@E`v1tpgLaeBE(6$x#K7H`$h5zi~(@&ng@Mfb`wqiC~?JD6vH@HEB!Yym`e=5K)_I;{|`aZk!zE9nEzOVC8@Dq?E9ASALnT})}O#sAQVWqflZ{z;}v#z60',
 'corpus_runner_snapshot.py': 'c-rkeU6b3$aqs*UbL$6yG8emh`mnPy9%M<ryE>g#S*`9W#js$I9Fp*Y01gA}YQ5V3kUXUF8rN0IRY|#$%A@p<2f6$|_MfDCzA!TYNUqLy`60ThTXJT)d#0zmr>DDzdwYAQbw;WcC2SR!bWY~^I!-cDl=+krnk|!zG7@KVQq<ElVJq_0v;Cr^jFuNkwj{H>ENVt(D>^$b@+7NRu(!82NY+JORwOQ$MO-pESd{r1#-=HqRY{&PA_qLp>#U;Xh|K9CuG4Cs%qj&uV;Ab9_>(5npstcsx8~~8Z&;qG&+E8a33>&50vJW>6?Vwa?1~Kr-#$Nm{ng1!5)w}=W52AkjF$TskG#RN=f8UL=Ed1-=)E2g_~+HI<|tm$tn!YD_f7uWBu(RkhruE7ze}=te#ywoGjeno96lz{@aVx~^7he#A$d|1DgBO4zfP)yhxdOL+<!#;ufII|=EaDl$vGuo0PFH0dAfo*(}Sb?fB^p^uj55rCKALOiAl_98eakZqhLh5xT@&7s2F^|4_~WfP4gOOH~#z(`X4+DCeS>s=SzyMM?Z&Ffd)3PfUSsNmvavPt2kv;b@Ik2FZfnL=e25b2+G!34qO9d4gI5sl2FG#1y#~`Zh+j!88>}Zc@^o2|DD;lh?BG~X;f9I0K1XXo|do$utn>9PH~5DAe~2fmR@<IrrX-amw85^Z^n5aO_MAxuT-<Z3Sz$oER3c!OX_vBNJ@r~XJzdS0mKnq&}z2o?ABe^-ZX|!5leoH^E^5{gc)4s<vA^pu}AQ285cki_&Udy;Gw(MJ-js~;0Ss4=&&sTid|`*EoZB^=JTG_Wopk_b@K_^vukAVXAd_@;n8L(wC4_*2_*6?uK4bWEy0Zw!k~8(goD*<Drf6%ewtfS1;mZ5zl@X0;OmjqX)NRO(coq<7=TS8TuNEI1{G%htc)*Hv{rz8c+3f}<1ATFR)Lw2y#b#BS|lHakptqXF#$GuLk^cL^!NoVELz2ErN#?#e01s!$!DRaGM0T280*g;KJq5VnvrFkFiO6S)0#dn%e?fx*L)~J%wV76Y)R*Vq<)!HE7DCa$P1eJG@Ipfu<xN)SBw3h!z^M(SOwfzHzXxnVCiE7<_l+j3Ma;pEb@}zCjw`QfWswYO8{S!!XHi)A3(AE>vHTRb8o`oN)9BD-|+S05ZN%?wB%=XktQ>cAPEkK6;o>lF{(?A*WBI`B%+*KyE}r3q;vct#!ISrqQG(m<HQL3uy-v+i8b7CR_>|gWqHNn0{S*#74yvnh$*49kyFHpbczyU(30befdMF#k0gg9Vz!_z4>{j_b6#pKV?JQAiIEHMsXowwhX-#?Uyyae)@ZlD9^hndY_<yelB?2HXfc*M6_UReU*jTyRrha(vZ11rc^X`C1Q8^R^BTEnvRJYdJqsU(jYvG3QN|)x#Z}Eh@7pIQ(X%%vPd$l8@KE%E+=t*D(6?0?&ni0i2!&&kcwc{c^74yFLk=}LwQ-kaQc=qYsHykLi`v7XD;|=yqUHNq7@G~el7m=3w-g%Sn9D6&Af;{GOivlmZR9!_%g&RcXrHd!-&C|WO_GqGkW-iidZKZP4N?eVa5ixcuxDaV3=cIBBFCj*BwgGQ^yiFbuu?9hffN$J4Tz{OmoQSZ8V=gRB`}yhL0!zj82Pru*_1;8CYU3dJk>YsI_UNGN_3sv9W}#NRaJD3H9EY_lqed7=p39}8$;7>QRK}XBezfyJd4n{go`~_iVwPUMvIu-7FX}p1}G7gY>;1FmZ?!D$E=}SIcDgL<bnmSxIk=`QA>6dsdL4BR2vP7({vio&e6XHfY9f9(y&O(M3Vvmr%D-ZCG7Lxd;f9MD6~hRcP69MXgJgxCj{N1@Ad7Xve<?>cGqQDmy%|1Kx?Zl2aeUCNgpr4Kdj(-5LH(N4ZRA;9jvN#`WU?^xXf0e@QM1smE_tQpjtt>#}m{Rew%T|h}&4NXu-rwST?w-Ev=XB{jlJ(7mS-)?*4|b^V!_DmDC;rGd&!VpV|!W8Z#pI<%a&*j=wIiLLim4w4`{C-{e{-PqE00I*motVS|&-^17VS2=JgkE?xDp2gbRA3ob+{Kwt(!)2h1Igh?A494O3yM8|!%t)sM~D8%;qMsNg%!?}RtxW_%QVkY2qgIe){=s`>Gw{fF89HNaLMAu1f__Jcb@xaxZB&%sdDZj+v$vM3`Hr-SBF0Kdg0cYqB>}%j0?|kT?;!TqWqwte;gkpm}D!r!7w9mF4c;DA?np9T?0@boe<E6mJt+V<djy2Q0;il5`uevF99lREpjoxl{k=X~AIUNVPb?SJA$E;>8zk`EoK*`2pjhBcUbC&_#Sn+tm_Z+v%lR*p&nZOoBv&AC8yCJx1Tr3dCnev49@*aB+CwM_Iyl))FmBAJ(fb2QNyHv;<Ip;6$8_T_&3{Bx2dw?~D0TEvc>Nrd)Gz|t9SY@oFFiz0qnRr+bgA<F!G`}3Sc(`%V_ELp%k6n6`7Jg2Yq5T6L3e$K!oyVj+CjNfexhSx$=Je%VTiWcROmeWSiMYsLt8vFN5{z2d!g$<bk{W#2eE_Lo9H3xP0x_H#Dd4q!2@+cL011ZRRM5o9Sg~KWolk;MgXUmwq6h-JA==G99HVB<hv9fZF@S?+%~Xh@VbCY@T(TPCR%-|_qE+PH4)<Y>t>^}7mb$sfF5GQwM;U;NjsrmUjX~}*rSv7!6ys$;-Y?q{8Cq(nS=WQnbW$(4ccl?)b76UNLjSjr-PzgX$hv*l&)Poh&Z~hvUW?H0Uu`WLRuX=~O57-O*E(e9R$MwXEW^TGxElMv6`jz1iKo$d4(Dp#1qPu&^-$oY%@9h&q}(^!D7CuZ=Ty5Sv`%<eBc+7f9M-yqFfz@6{cq(A6ar%g!r@A2I_Ji-&v|K^had0J+`*^OzJ_)Vw)-@FevCVZ9le@Q_hbg$a=yD?Gtpjg7O&xofSZ8=H*W4^(FM~Xr~uNA-l#{E)j!`mUxAK97jYiV>wG4G#-*(b+zHs?0!)TR5MM^hLr84~uOUpHEx$VPGkS?}^UyQmj9gwZE(e5zXK*@x#~XZsU?gM`*ULg7o-U}6_!Zm%0IItaSKuSJ+TqQ(%Gb#(nqF0)t3qJ}a|M00YFv?hxXfTCf%+nku+NgEClAAvXO>ZGenOsp{N0RP(kTP7WcKlYGLqhY_}7F~Wqn0vx9|T@;I6vD;Qi^pYEmU1|AQy6b5h-Y_!kHSVz7F4`{DO7!OI)Y3-G{h5vhi7+$Xp1e^`_G?fc(nK^Nx5-XH(z;}3xK_QM~@wLV{O4zA6k_XbHlCKn(7Ehm5c=Rf`T?S~&`<XYO7v3d4RZXU}OHQmxG!u{6`>Ud{**$rNx0`Z62_y1mzls93>6DSlI;!L4)6({8U_Wl1><og<yM@nSO5^~UVP5%DPlNVo|{X=y6{ME_nS+JgWuG)J?o{8-LJ95_WQnC#i@tfKhWGHxX(E5V6oJw6H?>f-#@9z`-|6}V9jBT_NeJ;W)50rUr^n%NZa)TFp@5nWdZv3ko(loVQF{8J`4Hexyfi<t!1@py7>3dEv7RxZ-9J5)Hggj3UJO6Ial;WSF7$Zxcu7EcQ--R#F&R&fqyVkewe+T@+J$x<g3<jK0V*gWc`I!8=ZBl+MwrMihqI=$V&wm*`KRrD;eGPkyF#ni&!EYeg&9iaL(L$wxDkCN&Qf0*ItUIa#_xnY%9Fcn?BGbX)gArsV-$r1dLXk+9P9YpgJend~(|j3GY!2@q9?GZ;PCVo#8Mp>{7;_5f9pAgB$}fQ4J<)!n^QQdk*v|JGchr2b@J6llKX;=d8bjh;NKFI|m?|dVhX%wdy^Yz$H`#fXUuMK495ne{3>W%pa!!H8WT-L|(>zZ%5wQWux`sRrRN^35AFb{Zh22SZ!rz6mQTcFZstxQ;6x&q`yHjkzc2ZQMKn`^t%#gZ{2^ci<T#I!S@x*>K44?!G!pV9Nh)oS0GMCIj4h)~0We{Oq<yBBzd2Zaz0BzFATM$}!jK%S|r2zG1{ZpiiK1L=|<NSA+z~d#F5FGgl25KTP%x{ktnB;bM^S48!?#N$s1sz^km!pjwR=^$E`dx5aDD^tKSZ&|eMjCUQ+4e}}xTMv`ZwH_W3#*80cePAkp^U)-C5FhLge{T`&PqiVkWY9Z#`sQt5t2hEp4YB5z!33`)<uC;lpN;Z5GEJBMz&SD)7SzmTxjOfF_Qd(9BncdE`*+6tfz%U=n{@(IlZPHroU=PtYZ%H3imV~2!ex=K^P6Lx3V4)FMzbdoHpKh<*_?xm(lTBMO6uO+)+-XHKYStj@|^WS~j8K4#FlRA|I$x=tEoxry*IAC1Y1tWucRiaA}ZBJ<!;7MUgZ(UFaJ-y9u+lX*O<NH>F0y27m(1&i$qutWn#@Diu;nFZx6v!Bi>(rpNFY^<quJ$pGWwFSW|!)<(7E`>8D41s~=>I$p9!>@fl33t|p#9XWmTGCF(m1=q&Y=TA>SMBx?zjjx`ZeHj=RQkojMEk09gu|`sh7lyIQcQb>-V43Dq-@7LvhFk20_ku3t*%dmh$Ggs$R5_`OWf?b)TTc|oVC_irwGHv+35t(2GDc<Tq&4#N<d-LB(UUKpLyd4ZNWh?Y4WfYuV2O{0<HH6=0V5)#+K!1NpON2FmRTM-X!#^vyaW48)qL^TgV3-bn@y7-OV>e`tb%2JVN9AA?O3k7y%|EQ8AY$L=GWXsokl90;-nMJ*|)9;D(xH1lQL|VF7*hQM&guK_9zb&3Gtdy5D^`==9gF0XGDo~6AZV&EtC<c>my=zC@&@DSK>9#KFkA~4>4l{lS&aM!1_4LGblq$as36Lj~5uMi8Fp+lx58etKlrEE-4ko_p%J4DN<@m?q@v9($L+K-{sTVz5Y(a*>!t-8)TW`@b1V}DqjOa1#jDtLUjfIz_oMk(a;8wcAt}WgiXs?k2BQVSmoc*uBiw2ZtDv2u6TF_GqT4>BJIS}p?Q<mpN6)T<iZMq9Hq%RsicpMT$RgJ<#eL9KsD|w=DK^)K1to1M2}svOqPKN+{VZ@Ns|_8W4K1avn9^rK5IFbt;q>_(Zn5zG1==<9CciF`uDi8<6)O+M~yJA^<iI_pr3;MD5F7Oztu40+u>F}=D(;wS|^IVeP+FGt8+iJHnD|s!;$EU9unDA{9w2Rj$+pnZ^DxlLrrZ7v}@qtvD6|>maEFKR2?GI@kC{^vt1$9BNmO*e%f*ebmrZ#HACy-I2k(50nepIG>akjjipA0XJRt6CvLS1)4K?cG-aXnSR&OrT%4)wW>R_xh<%3r#Q;R)tj6zT^d`?p@+)-scSnuh)zIv0PzHU6;SNrlQYo>bPR3@HZ^YiOo4mJT0|TcBBPS|>+&LfIR=2XwDLJ$YhnxFIV>blBo~&O}7t`xM-+)0}q%~WG_%dnOV;98bNKSGmDN)k*qb$a5*5NK%-%ZC19kmcxaV4Opf-7NvoTi%!e}bl=#Dg@z_alz79zSA-;|-sUWLN&5Qv|!c8gT`N_qw<R@)G>2o_+&1R%ey)Mgo{r4bfxx%oU&X)tgE)h}WaQbDc@9*mpQ)9n-b#togCmRCAK9zuVz$G=6W(+b}Krpyjui6#7nxMz!@dcn95fSlp!6<lj-Bq_;F@o5pPQP+g^O^qh#dn~jo+r<^^6n$yZ)+vsoMTr%y0bJaz|htCx1Ok{X>x@vK(s1<j7jAbaiv;UJ`3PxdxY(>d5eP|n2tGm`+6|=#z!^r9OTXHC=jCvfg))C*gpB>luG|ex$TiAZ|X{?Rc6I+iuFcm%w_*(~tfgAN+BJzfV-RkW0>kyqQPBNZEiDb!Eow0ZUss8yK*1^1vy3saYO`GWU!aAhU$6mSe){Vn~>WJRT-I-oRiOD#)Hnn)yp0cbPjhe@N8>YO=^+4{<ryX{a9h|^!2dw?X$#e~QZJ@o<K2J_OMrzd+@APy@?7>*BZZf>fD=A>PrACnluWP#X>n>=U6WLjQ$NFzN%US*+ZmT0$|L?y3jk&LoX9W>=E@e9emp?VYyd}CTW4a;*BVKp8iG-fKM~@2`ZgL@qcMo)a%G5T5os4Hz_GEN=QM_<dKcN%6#A?nWo>8>BA?6%yxaM)eogc~#dQGuJdIzh~SyW~gK!`$7p0gCpMu!1QUajBw39vQ7pYah1Sa8pH9>fG6G}Sb1cv-)~F(57<|0nbGS4?)ZF`~m1u=uX$!;`GA3G^+@LQY)#u}bG8lvpHQDSZ7y5v4e4_4Yj}mvF_(s#m<npHnu2pTWQ-g%MPw%~Xn6KnCD~M6m?Hni1e8udIfu!36d-9E&8YlH^XP{e4l^A+lQ#S!#BwXS}dTcLGsA4Kw1U86n7zewY|O*a!(1g?+dv7?2x>g9+Q`CajYbRu*o8R@s_;k*$$B2P3#^EMWK_gkrvWXSJo=NNQD2i4Xjtyp%j|vf_9k8s$go2^Rj?R!^M}_8C}YdF)jrOD|f;x(YaPO@XN5Y!wmk)$svh_wnh4x`py)brl`BH$2{}(rB&x81HRmdz&MP_Z#QZU;qX)icle=h+FR{;&qyl46&oyvT6>?;AnBKV3OKDRU?0w2bFQ~e<aCu=l',
 'crawler_snapshot.py': 'c-qZ<>vJ5}b?0~d6)mc~Jwpu!1CZ285Jxsm$uY|krBIZ2QxGy~%=BPdn3uX|2*4mdI4&o)H{Oajk8*aCiWF}uDLamw#G8%9?D_*$`af7@O7f@u7xs4^_x8R07yz`q8<h?Abl-dK>)i9c=U%#W>8r^Xli91c?gafbIvlUhrsLyao{VRUc5r(Zyt9nc<6s=^#pA3MOlAZAcacWPG@0%PvouJigZK!~?Ms&~tqs%JB<S^q%f&K{d%Yl;%xCE$h^EuoB3dM~X|}dz-lhBVD9vK2)1QsUai2?=N;hZA=^{>B!5|(+%kg56^cSY^w7*Q#c)DmyOSaD4Q5r{s*Jrcw%SUm4xtO7Pl=b?v$$T6y;(@Kdh>sTIWY5&oH<M@@?Z>H<8bphDkxXJ!#{9<c@%L-9sdhb{M9H|lTqI-L4IUVk?=5CMQ_l>1|7*$IJddJmgl?Fh@(*6LC-EXe#mIh{M}q+dnwb}GXS1n!o<xh0^kt4GXIQV(2YpVX>0maozifB15p~`En)MIjh5bc~mMxOLZ8kf$j|-k_th{~U401V5`al*Q0{3j0j%i#xGp2iZ_x3EAw&)?7kCTO0G@b3u7F<3x74FchFD2vnb$)GeX?`4~>Ff|ojTtDsY}4W|7cqbA7e4|%P<+83-^nKj@3KWc(li<-;|Q3hgISCRrc3+rqBk55*4DoI@|`zsy>>h32I0NUPPq2w4SF=Y>rR}-QQ9Bfq)is58=dxMxOV5YFTQrScjGr+zI}H^*>G)b4O<{sF8Yn;_F91d(inZ2T8wN@XNL`gl)%J(vz=tK;VcDWG@4p{knCej4O$T&isE298%0~!o|OXaQGBEoOnuBi+@rD$LNWz$m@Gz&W%9B;o5#~enC^v5n$e*Ys%VRWX#>Mn5K7x&RlCt*Hc9%uz2ilkaVxFB83Vn*+~r-cZ)s_XLQD<@N!pMfS@&+b#0Cbz$`-xZ0soXLi+C~zAg)QH>P@3b%plwJ6g(3Q!}f{A`8KV4G62j*^R&H~%)=Z^`!G$g^`*z$*Zfl&&&N?e*1pOJr8sS99p7y4Z5>|4;Hg}Du$;`Z2FhV9(~M|sl=YLO`=w}{VSGfBr;F}ZvmH-q8#cn_Vz~ZX*fdR!XVD;U(Z>gFfC^*{XJu*JJ))5f28k)?7J*X5vGx*sE2|$v7`AESH=MDgRU>I)+QYY|VE|kXh}8((x>1d0z#C8xe>@FCUHDU*3h{~pWC)BK&kp0%V%wB9%{cj5+@P^4AbJFDI5e%DmkV!y>1NoHG78cxa`;w?&bN37-dpp|g`URkY`ND+!=1Ml!CT8tC+h4zLj?nc-gF0LOTRTexzainfVMH2-J^E4NRxS^$<ud4(?@XHG8x=~X490>@z2pqY?TJMvsu5{a$$BDy=~*SbR`)E4f?ndgkYS>5WBWbD~hr52Fnb&;(kX72>(~3@iKlHls9dJuPqm#%Rw|I)H){85JZ6i^a4h>Bu>GT+8@nk_z?xu_%K*sN26_>yll2i`?1$Qd8LG4NPGI7_Ul6CiFGvG?-;^-*v|SRF!7#n1j3Chc&Z!hgvo&B6Ic`OQZFFGfIhSr$MZM?f$b%b86Yh*iV%iLJRUTtc9RRQoQ0Q<=8$`0@DqU+Bm;;`gMgL=0u~x4*=RYLW@gCAAd^AKmW=l7m|CW2m;1iklz+@fQL#N;j>iy(AX(9f=KVH5gA{|yPy34o$~4=MnC8bo8%)9&^qk(U8p5kdmZ>l}SOT~DU`u9UZoOpt%`H*aRj_Nw>tT|D7T0`yI(YNWD?zkejAkH`$3c<>*>X5c`U&V8qxC$UftjHB7lM1(ki-&$AqIII?8W$wAMv6;B8n}op~_x_H?$kDaG<<tk6(Z|<+I0Y?hKl$(Ndri7j8?@Mh7cA!(@#U;#hfkopHjL1akv9YKol?2rXLJrokrc;t7GnKK|{|VlmG?zrC@s^V=J|@pD4v{#Tm*VR&oGBR5ly8QTs{Q2VL$RHxqj2LTX*JCU+<Y{<X{n!3{5+1j;~(5)fsvLh9DgJ-(IW^s_&a?)@Yv3-v^ht1y%t_Pc)PNzHww|E&gV9G`Yc@17<_$A1gz`P8aS%sf=ej?HW0;*zg^6B8^cb3t3otOeB(RkKpEk)#-G^TBlCVNYMM>s&`GWj|LDI#tpc=gT_3JxZgl$18fhzId;hHaT*YisYDt)L^V9*z>~2gb$Kya}_T@{vzh4*%mSB~0VDb}FwI*NrL4dD}Dp`7rJBq}b|RRVUha_#Y*_aXc_g3ex$#8CIF5Bv;=Wv`>_9$MLjG3s&vUW^qRP5WreyTSBR7tk{|@f8TC;9}C;;;=6<fQ@=;%P)C`;n@KT6y;9?3k`SZyX1%z~M2-m@tsGY<uNr-BM{@|`gT@Kb|7|-}OxCsoCPSgLz+dU%)E^Pl{0fX<U7v;ma!Lq*|L=qOIHPZt>kPG8ZI4Z_cevax;9sQQf=#luyMgYwMk>sKW}c)*JM|X#bCHtgvlU>&cE{0VZx99Pc91Fu73hm7T@dA-k%|DxKUyqO#(DH78T9x&5ac%0FF>j)Fq@n2R&?T8#L@UQG1~^WVBV}EX>UMiNQU5XGf<-S^<=Ov&}g6QPe5wKNB!|KOYX%@D?AJX+b(e+YAJ6hi^n32)X5iMwb+v2kPQ`%UhIY~frx>dm)Ap?tmdw_&GKf7t95eua_TNE3-YO9!>9RLnGUXAFK4qzQ#`vhl&*@QzQod43S42a^u&R5%0!=3NR`{x*O>>MSn#K9hwB=?a3W*d4mt&LE`<8juh3r3Nswt6>WI<l-0V7YAsuQOA^p6XuBd4$xC;;#+sCwO0#mqDu9lj@j0HUrxug>6oY=mHG&NX0P;O(tg1|ZH$&ArPnDFvK?8Jj7YwG9dNIyqM{^z_6D`J8)P-7ibWwwlGS|`hF`K<~W_$ChyAQ3`HK<W7<UFoD{rl?Scmq~z~)=ihM6nPWX$KC?+^r8jPH8SiF`eVMbnBIgpZvDnGUEYO}^=QMGg=lBNPB~2WsYHL4&X*azV-7{H*eV-3Bb4>RCrB~6g|Vr#+4x=zrq{9ys@9in$Ht>dk_(Kc0dh38grzbYB&;MU%s+xKCgDnE&XQoagNAkjni4AREK^%UH+z<d@weHW`VC0~zpT{3uY@uv7APVY`(HWMs;%J6&MKsK5dia0p!}@D_R{6l(EgnAK+nFH%$Av_x!Uqg$8LsKM2~lNJ?V$6ot6gp1Z_5E`fVE6pnd>buvzdtymW%9LM%^tte2{k7Qm>`;d7;63T<&2`*f5WLCd*q2~e6v=u%8NKvdcGhD3VvQB-d8m4BT>6AJIJ$>7#Y8>BQ&l8kk$c39H~GU35i86Eb>C`A@aFwd<{C%E4AN|0hkXu9C-)*;dBN|jOrNdCN=(by(-OLg{UuzYmcCD0zkWT_?_e*-my`-;)5St(Vios>W@e{-8tEZ?mbw(4e%MnF5FcJ4$v?xV-BBc)MaT5s8=*^L5o0$Dx6-TeJ9nSu@oH!shd%(b#SfIq7U5fvB%xM;cTa2)OTNOeOc2oYSfTnaP>i;IEC=0C?$NsS9yl|3ba9WBs6=Ep>PXpvhA^{f#K73h&(*(F+Tkj$gzu-v7yKG28xN|(7x8E~0RIn;r+J=wp^%2oA?dSdhocQfZ@tC}FfR^o@2!Y-%Eq$&p%>hOi%%MYcwa~R1Mt{M5HjuF%e3Fx7=O?J&wyc&YrjC2jx*81Zp%K~=A$X<E*s2|VCAfSYitfoCMuw>EeK@}PgSsn(lAjjnbuE8!$!k&l?b0HTX0hk~1kIOmfmTe19AP{Mkl6jB5L}slxt?dIwLme1L$KA~~WF+t<_#c3)S6abkvd=?x$a?lIv8Ps<SZFcq>nSF+M?NM1?rvTqkor;71pRWQgD<skV7yL`dwa_PaK8I|#{n|LFv<A8Sd1A)amIK!{KR9@XVZPdMQI6j)cJ^gK!t&$rhO#NQ2464n20llkFKgg1g}vAziFhRL@chAeemp8)dad&WwM09YL86f^I4WGARQS4vP-kw)<kq<<+=cj@V}y4txDdJ{})yx2W`>H59fvF;!_UHeS3aLIl8%!^0()O<e+B`$a#6{31?7ETLBvyRA((L0@<}DS%tmWu@!T5SgqLDqvqoB=LQy?DAp`mG;J{o2Y`QKNc2EU@;B)bq?^vU%*rPi#;HX#fyOeD!MHk1(^UehHEJ-SYEi3Ud5h(Jab}+^QdPyvCH*$`mk~>>xmWiUgg-P2J%Lg9Ak(&|kbAH}{GPZ4<Q46*)0Xr9%CW(Okfg_4w$x%Y_WH$d$Vq3YJ+;{bt|_%o<9#w!AG2A+!7H{@=VL8U8RJTny3nK+YO8H91{bHrX+DFP0T#%uJI$%~02z|ipnNc_pmCM3fc1Hqww7R1sRUrO-_UlNXVgL5Ks111$_Pv+;hRu5*Kh1YJi~^4^9DL_qfd;R6=kgA#%M7aw;;!`F##Yp=7Zr2?{quu=UbOIF7wm1@U&TsU?{gOv+@OwmFYu89L)xHo{USX;X89AK1eLgxog^;MkyhF#aJ0gAi4<Y{Ol|Wo4Hq#OS`UZeFa%5ExejTd^ntykbwz+Bj<#PxJ_7Wuj(6zznGq}{17|`hJN+xmF7i2V>GBtWNkpZk)S%wJssS@5*-JNnQ?Cs1&Z%cRGJa;5vWEl#M2k>+H62DyhpA;NU3x9!WZhMF%+5zFw<rZcOpHH!mE<$aMmcyWy>2bsx~md@(OjkW>G{|c&^i|aR-_}u68o->0^VsvvYNKr?X3uJH7q!Y%dzu5ovcKm(wg7#s*H<Dq@1OWpz*WX2d>(YOq0_jq*lRlx4guA#Nnk$%_GY7i>Rqw_wHK&EVm9{)xCn*Mvuj>yrp^FN5wgU5t-K!Vc7_=2s@_M$)k%zn0QjVhrdWPMRJXiOrups@0DA?<|2Y4lRktx+vavGjy^+cJi?Y>pUs#$qY4S2n7K_JQLBeDsvz=Vx1HUq5B<Ki9pyC)W%T+CJTCS7=f?CyXIJksb|i10Galiq$-#x?W@b9*myzn#5RkH&KI%MokDBJqus`Byx3he6X(Izi_OK}v6+HCM4-X!a9XfMa2K+19M2on!NSMlJXtpmvDHH7E0kQ>D!CwIo@oC+*MgO(+9FdigEbZy3L51IT5npphm{rBYGvwFi*TRO%vYFvYUTun__#rad=*CX4sHZn;$X8vaG`pIUBXH#Mz&}&1~cI>Te1L!gDK)@qV^Z^-yd4jPi8{M;J`5xP(5k=(Q-;eWdW1^m`$zNGL<EV9E*g~>QrQ);-yy8SL4}h*RDL<tX~~z<Qa*qVOY)DvZPpY@fTOQPV3iw2ChgRDRRmU!&fO9W-*GU@MjUU(;&Av@`i({F<jl5!p1Qei~e1`#(SfdhLoy0^rU&wah378zT2|6#$dH#!dFqiRlWbUZNxyOZ;NO(qcqkg6@gpEsBH053iK#;v6YL~*gDHT2hX=}!tHbmC8wg-y_B%wRBJ0(il8uIn=+@>y)o2UnUZU=;eQBCI<+gHhB1;TuWmix3a)jotiTSTF3w1e7QlA<JERM)-xy+d(lmEgAW0>=STQQ|Q&n%(vFqW-Y%_G!>(#BD*`1YLsyuw4U2M}cCV4V%Sebx7ujsrpfJQ!oR&W<KFMWIsga$Kl$UmEB_nlObYFu^%m>m*cb}RTbf`ZLxYgm<NYE}DU%@0F^TVh-#DO7dCqH~pi)vD^{iVcuYcV#D7AWdP(#?JKBei%%|d|>RnX|kV8-Cz;=N}3fpH~1q3QEy14_Tp#}u6Y6x>!GzGktK;;Tl$$=15G|A(!^k|=YTi~@()j>GpE}p7VCOb7{N|M6VBai?rd*9n|og13P2b;NB5qDQXAU4JlLQhJ6ycsW{X~R(-(62>SMLlFEpePB}oJb)etI9e~mQY;ZjHpezSa$n{MVam=dtcF>M0$SyU%uK?KE|k`U|K69f{9G9*ty)wWsA;2^8bR<-T;nP9W5XkLt8nRzzG6COY}yd$P8?q^v_)N%F>5VX||T`cIW5+&VHlhjuBh=S`KtW|Yyr?4Qqx$S<ft_go)y}{0B^G2AD!3)oFtPeVVMt%gS)3K*maS(!;n)~2yZjtAgLL7)=`}PUQAThSHA6F~q*No+o5cq5bMj}w>Ia1UKnrNS4pXesh)^ePt;0bwRoR)qfP-D))S930C_Rwif6J}NHw%IjO7B^qbQ1;?|4y`;>2RoAw)`>l`OI%}z9sOo!>uPYB|7o4cqt5)#C?yr9Z^vR{h1}W(bpWo}^9<R{h-CHh1}%Wu&QAP7*G!u0sPB4G(@puZPTZZurm%U=sj{-Ig6p0fYzOzm9|CU(A+2dh-5*4$4oo~PzCY=bG)&yV`a(Nx;<!UYvDYIb2GSPAQ}gtk!y>HGaEh)QanW2M_8%I5q!RxPyn?ZI!!Sc3A358jxSa6BaHrE=bfTN{w3nV>JD#@pzZTZE=bf)I&QgkLp!fogEU|GV60I>IEbVrhXW)+;4Z-X`<+v1z$b$pus7_>5QPH}|N&m*X?eih4@E&Rlbs!I1>vVQo3KPJ3?ELPsyOj#SQ%+{iUaz@Y3#-=V>%_mpG(c*bQ~W^&b$N;ygk7fEOAxvWg_sE(#fq<_nr66#ZE53Vx|U{nJ5@VW2(wwcC>|>q=M@L*=xTRMtA8GK?Mep{%9YL*{kcMa{yqJ9PNJ$j3iFEDoGTQCoIUCp;hcyoteqUr?h5U{DpIXp)TobquhHjy*^ZQXBQz=Nm#BWi-*`|d5MdTI=1eEECc4ZyDM;(a(PH@#NTzdqyfCa?tnfOyyh3vf5VAHxU+sAYKD&7RDa2GUR@IbRwO5Qh;K)?8Q>?RxTfd00q%^r-zWeGc0lC0|0>R8gYp_n!tT|OcA~`xyI5^(0Lf5K#a#+2QpP3{+p4HMlI{{RIhfWU__*JlTbgsf_w%KWJ|N4foQ?W(L{lEskT5E5qUEiSuNS+9&)9_+j2QU-0PpT~a<-2!Z4=l9iMfRs7_<I9`#U9oqM)7Nfz4mOU^Z#<VIWnJrykcT;2Pc|kS+}XtZz@%r3+qBP%Gr6jz0r#C(S*PG0WbK_a;B`QA>?)Hb9K}V18G<}T1~6Acd%5wA<&y=qfny|a>KAKXe*K+Na2lF1nAj}n3!Rs7G|-#&2vh`M7rH5W#g5v($RL-H+RpZU<I5ni(A#yWYuc2+df}=@|Tyb;WebEyaUSnsnVFRnlaTFE!NI}V5Q+)LalQ2=iPP8Io)oh6=(Y#JFVbQxAM>`r@9vMmwPYjAIyeyFn%kzd^u>CzOBQO1ap!e>_oByN+3dB<@VOKW}VYnf&GfE=O^+oyVy+&3HvD4C)UNisTSf5GhC6uGRI4qmJM9th3rxOt}^|g!Bpr9^tz-M)H<<?ih{+au6=@9<SQoKTBSryx72*Sw$P7Yr|ON8g*r})kx$mXsa;bGpc7%j5{VScBTU?|r`m`oHB=LOJ(+7v%_+{p<Hs~!K%I9Ot%;IYVMNNE$$s>RAFx+4og2%0PZ7LgCbjXk(!>pNbGsk&S?xf)UJ(h%+s2r;C5Ur6Y*x`0X|cv{Aux9Y&~6?^lY0(y;A_<~V`+thZWZ7vPAtVGLd)BnY%#AnjZz?NW&&#H+T6xARjWuG4Qd@P<p|JUg;LcAaXPbI;RkX9Ppc`3>7eXEl&>;2Q0K2)L?Es}F_QGm3|Gr|&gl)un4}Ds4v94CtXOa}Y5tcSg_>)h4T3qoAWkHZL&(XN?21;tg$snyzNcxdjVKi==Co%X*NTXKM(~%4EtZnA@xut@u}GoC$XV6e`q8w<=D>!NBCIMSNWET_WE>8|x|K(2#R=K9O;Oz*zjPbpz1DF~pz|KpI>x$AZ@o(HaJUMQ0(vl23X+cL3+-3+k#cEWH#TBAl}jv1>(~{ttVkx$iE^aVYHpy3XJE?TjwJCz;jm>d_qaS)Sb$0wg&B&81<#S&aK}o)Om(SzHS1*_Ck=rYWbGp_ov}KtAoHc@<I3(BRQK>#-LOrPWQkY1L9oKG<&ELmHHIu&jU7W%cB?j7z4^9WSsvGluCiJm>I&!S#l2^%9A&1f=k<`)I=r69*`=;7<M9%-Y_6@abh}FAZuxjxF&g9?diF}|zp<}ZXjNB=T9t#5MP;g9i>fD+z)!@$AM^}8_VqagFLGwjqyAc*=TD_}R-v>5bpcT|Cg`I1O{`QD^VFd~TO9pFf3Ax5Y=(XPJfnU(cx46|S-f2~Tn54@VgQYzL_zfta=XGa#^*i7X}z4b&m*Qix&G;n{{o^If()O%Skx<pK(YNDDw0(WIpEF)hFPtUGv10Gs+wT)*SPd$rIIPdDvxc2YbcL&Wxn;9d3AqvPi}=v7Tju|U9(ZPSVtb=WGHsu+*jK?E$KR+xqrzX`+hRF63=s<T#1N;#~&6m{FFAx)hvW`TZd7>iE}ALDxEje0t-K0I9pY$sYZ4176@$`*x5wWc#grcrzXWEk~e<^NNo<;u&0S;c8`x@?2-H&rAaKg&qyUe7%^-LNq-!#BeB>d0_hmV%apSPb1G=XB%CJ81uBNdr{h9C#xzLoF$yGnrfHM^Zt>qM{P!yVy~cl^rN0aH;rESqpCM8*mN-1k#lL&7L@r>z9k?Rj)4}*?LfI52H0}(i)b7#wSqM&JHq!w?UkF~K<bpBH`yin<BBuh;P7=~}+>%Ultv=l!wciF*$@y1x_AWoy5$!AcGnJa9L}L`!g<`zBvObb_E_f~$$79}L$uNadOi$C9^Z6d{Ix^*jtGkec*HyxH=WfB6wwH#*y0oxnC*-+I?L^QZFg>ek*+R`VKfXSp?8;LH;@6jD<pL#A!Ef7iTE5~eiq~#lvU+Y8OqM7k$jUJX_k*xmJ9}qO*IGUzvlZP0uTFwljk*>fHrXxgnj=^;d-LU<Il{=%z6~6Si1S^2hkT_Jvu0?GVsDs9@&qM(Eu>Z@Nr50_BK;uzlyVeN&K=tMc2g?=+Q!sW`)JOgE*l${cfODex|a;)N0i9o(r%u2j1D}Qx}30Nvy>j1ED~#eOq8jCvvH=Swry$wpKQF&<0!UdVx-P!NL}h-pp$Ia09vDC9ZV@zO_!<RM7?0f1CR}*?YzDJ@ZIC!!|#3gjp_bd+2w8gzwy@KWV5w(y5anL^|aae{8MiYnqDqn?J5I4v!kwRP|W`g!ZGJCZU(hOAv59U1{<H7Y<%vwkc5`e!p{kQMpU}nx{q$U9zEhdWgc=To<NKY_?UY-;WG@3M3j#X9O5t`(JIpfm`=bX5<jzzhMt+M8f4Ra{qtj+g&$mse!Hm1hp6PWzn6x5K6bz-kT?gOR^YWA#91HPF`816qw*o<a}x3rhvo>SSk(SSL+Y|z!XM<!E+vT;&AaK^G-SU48)@sTOYV<e4;f72bxL@)k11T&-NgYK+EkhHD%mz$ojG~POW@Wk7{2+A2ei&0>{gQvu`#;3LoGx6p)|Qubf<ACxYWwY)Ot3j<knwc*f0R%EB|G%c-}1t)Nu10)V3BqVPi}WsIi?d;OzRa*k(S%lvbg^T~OU>dfc!yfV=E&f0H8{@;T>c;mN&w>vr(Ut%u*a9sK5-H*SX)2wo(tlxZQf<Z1UUXVOi7d3c@%De|Ea(M2DOJaYid&zQn<s+fi=ouDZ)0`jgx0oi`dw##R<xJmLq>vO6dIeKX}JF^tm{Al<}zR$Jf#&7<+8-cDJ+bNtqjvgebD`1{;ldIzCpAshJfZj95-rVX$*#U()66ZcVJZw8}oc+p13cg~bhIJtWVFp#{(r#F?%-XvCvaAC6WH`Ks1o#J3wXkWTX?7?q@iS{@no-qWMPtgD?C6Tzsft6n(?Jy3C>j<B9d9oThfC0yc5HE$+C9pxBj2s9tEip|&f6MVfJ8Gq?AER5*`s2zvI1$aU|-(!kVc`LNG#a4E6s75ejzkFrVBW7?{*mh8fEo7<X%YLTbU{87R{<zFL}fCX;b58!U{;5ybXpgB*T!c(t9VOP2ao6N|@&5GTrMz?@dkWEc3Fei&{^|V$RL5;O7i)MNJM#HLX#?Y918{OttIVtREnCUDAQg7~oF$wC-?t7TR{B(TLmrc0h8(GMVX54`4=i$&B;HAu||6QjabWIqsr=9=fbXu+b)e_iETBYz20NBsoZrtdwS#6hqgB!_r33{t_Sbugc)tPvqnasr~2@+)D9AQ85xhrQCgJ_fJThH0@bgYA_PBv{*%z>h`d=Fk0W!DSp-7_A9z;*R`s_(dMtzUQ|Clh78c`m=rM;RO9>;XR*yeF+v4{UqxT4#m5~o@w9b0UkvgeeKr5&-$tAI9QhbZ1CoV1FB7WJ3Iu@1fSyP<xVU~$YsGtFJ;CG;H{v~|;!S6+U?1qv1RmW(!V>G-2lJW$Y&yE{X%2JI8GNSfl7dF~$$$Ho$KU*?kH7QRzx?_i{hxpRue6@}>Q8_FlOMfHubR!OILo3WyH*Hqw~gxfN}^bmVX2G-MHWo|w0>P&)FH=}tjw~p^D`^^T+c0&32v)n#%5fV`Ko7WpOY9z<AwL0e%Bo7D{CoM0&}(o5Nfg?HkCn^4F#}AP3R16z3T8LU*+@qtG8ZS&lQN5r~BB2qafPLcsg3v(){{z?^P6-M)#6^$<&pSGl#szxlu~*0VCTCE`KqXs%$iNHrJm=UM2jw+}!wl*s9=gR-uqgT8mMQ8N?WuO%~_|Cs@i0%gq90+|H#vO}^))LUEYn5L}YL?R<e?buXRJKc|;=H@;xhd}pZyej9YRk3?B83zo!9@}xC*jihYTqO|_@P+CSbpYSnlJE*Xq$W4ts`jXq!xZG2ELE1iyHS>n+J1?25SQD>=D?8Aeb$+#77Kct&Xa&Bk@x;t@BuO^bt?9s>a_LCrl$!zObFO4#B^g;U=;92Tjj%$H++R0^N_JLL`7E=dDd^E%Q`z!M?Xme#4{k6^suvv}LUT7?s#+V3pAJG0CBThEgkh)oZB?k@CCppcBre17rPpq46LKV{E>OheFX;PDctbmi{cifa>si-NEFaKbtX9+*d{HU$)hI*uv^htaVgr3<^Ac1|4$M8m@XKj<L%IB_x}to`YU^sJcqec37H-*>v?ANQV5Y8Dz6WoB_-MB|w!Q$lj>(zvJNfhNlvI~nJD=vpWRj_*N1%3ThAY1Dvhv5L-F-rOow)&Vx{nSmd+<ds!Kaj#l`dlC%TZZup)86d{>rVKJqsYDj2Q6TX;>sZ^5I`xgG*No^qNjGYd6)!bI#_9@+6PAVYMxN%3S=<uer?OO8Rk%&a|D$rU(fi;qu=3v6JTmSCyc)Kd3gn<&@$A&-mdRChMW$jP##$s0>5i3lxIapg&F$z<eycKAc8{e|vr7;ACVjj&WzNSr7Tg+q5NUx|Dt{dN&Bd6+AG_d~nhcV)w$&z5ddd)>YOn<N_6e+5uKR18MJ)ogi>y=trOHOSutM)09aEX|_2+BcxV1LJAMQKszVFj)-JVkLVTOI?6PfZqAWl8{vxdl`k_I40o$U<DB48r|d-*ED|b17KP(%Qn(*y&6@C$0!fr9U>b;8o;^swP+SP2XO4Sw6=aRcXD+ua$k2FPT;3XGLOIH+Tc&E2BznP|dDVin06W`T-nmyTXHLEbXI0o<TbYg%E&{&PsOKHh_R-SZJ@+~bvTT~iao%x6F<DF`W88mf?_Nd-0u*Bjd7zsaC@mP|XZ$k`e&dT<S2nj~^?^q>kJAKqC-=p|ub}j9QNB_bGO@meqq2+t@4R*Ct#J4Ajh(j+*LR<3hQ<AZnzuX|K&x-NF|$w5icIY0vsa$Gx@*re^Ubdu%dijRi^HbdzPWZ54JmLdhUSq+L)#q|xU5`4ajd33+w%7JXyT}?{k;SF=V+ZCoYi&OLz&IzI#)VEQ3*kybI+INEVgO}GzNs1r<&F=Kc49{zFnWmvjo;mi>b@O9w~y~Ju{$0V!kO?)4s$tG%43Bv)RFNUX_xo)K`0WE^#i=m5G{;zzCMJzTq%r>}=ng8VhLM=dx;0Vz_OeZo96Xybo-QV5y`%?e8V+>3GtfCeT1;_neRvb|A4g%-pMv$*iAh^>R@_yRWQLHNL%3ZeMNWYtk%4(N`rz<JvLH>T*WAYVPy4J!?f`eQ%Y;?d>Q^CVKV~<{UfxRqgsen^jCff%~Ca&SmFzh4OjeY3OoD{fzJL?MX;MH`Gf#`0i}OU%F+<vGPB@{%JrQLPD+bd#vR>UqWZDSwS8Ja|M`tQcdoqaW1dccp34i^5B>zXtH(bJ%^$h&X_Yi!17pQvx+O`h3Ly_?ZBDT3YEY@5=Efp2>texzA3g+LI^TO<pje=0)pqgIDOI$hs0P@+=rc;Zzu|uyw-f!buS~av}n<hs|48LKzT=F4k6^6?79Cq52us$G56l}_*g-TgIwb=9q8m}@fB2SrAN3GoZFSNny}=1xyEmPetJ^3|CQ?+GT<U*RemrowWB3=<$5EYMt$R76vp2SY0=7VcI@K0vL{{>jv<r)B0?l$5a%b}F?;{(XR?Gc-tMAKtqzqd`|WlRnn9AML})NU7t1Hbt1>~c;mruL#TMsSH&z%Bf3(Vq?|3%IXn)bhz;qDkb-{*w7fQF}Ldo4Fl`&owt`pV><)l=-vEB}n1Sm&=Z~~B<&cJ0p3hpy;2*_2LLvxJ|Afgt%bagrYph+X7&@<XBC*mw*7xPh{+Ue$mT@3ZPO}}e!SByGa;VO<+GK_R-_NfB^Y3%mYKj*9TeHGLyvq)E*YenQxzgiW7?w+IL;#25akAgun-=ARKiZ{Vo#wt(2XGKy}YLO}?V5#c1w{4FCd6x>ZJ=A;c-c84jFzX?xD#0ED;W0GR8+vVEsg7IP-rk`#JlR-^>y>sr8jf>JEDz$Rd(q;<un;HaSI!S^QS6^`)>7bhT^-7Y!ENZwy0`3Y+gRC_HGJc}Np;i;3IY?KFj+IV@n!h|03sRI7bZ&T+kDdl^#1Uz<&~?QPDuHA`t%1co*xcjh~cnslab1pggD#PL{CppG#Q&02ny;9INcr{E5)f1J?Rds%R&OE_(u6QosgZ6BE1SahhMHpX`g5enF$2s8kwo#C5N&X51X=_e057tv!UC#_?3Wd;)oNMi|JPMDLW2q!}P(tBzW8bCT$%x9lXI2{=tZ@HVJTH>G2CNBP5y0T+YWDlD(9}9mf!CmSiDdRzXJHv~6|>O}{f<_~F}IImXzUSd6cYdeoZez?|+S*Z0Gn55Mv7uRi?V!yi8U=|}H<^pg*N_u=n;_=gX_^YD*9diSIEcTLc9HQ;-R2iT<$ZZSxgAP`94;-vz@^#jQSx3MxYrD@&j@Qm4tPS0fquV}_6a-mza&4qOpAXl36=}Ro>8X>mO`=+|f6<nnq-REsy`D#l!AA|$)JlF9~vE0B-lCzXk1Jc!okh#sxI7ZM05^&137c{6JSI*$Z8?j5~zEgM}r^~`2meO=OlT|&Z-Z<Q8q*A5WvNNZk5OQXwV5@WX<hem-rru^{QO9dx3SKSFaXz^;*#X71>fXXxSKKW`=N<U~C5bM!a7m3>F`-kKIK9t~NK91IG6C*EPqfSx4n)5a421|#*)MuqUAl?A-sx=C0Z>gbTaty|X*pL`ech6or_%9`rQ;(d<ejUQkdG=UE%m0hFGi!K?JsAWe1-^m2nmYM4yYQ618E0D>}Pv?1eOmJk)1W0%no9`Hn}Btg44x28ft(O(lO_jaif%l&L}*j1q(m^=Wjp$(H}hi+0P$;?>mqG{s+JK*LR67JpSfC|Kh=)e*B$3`Sks7fAWXF_sJjs1Hb))2cP`$Uq1fxzy0KcpFjTQcR&5<&mRAmzy0+6zoKs+fB!#!{QbZD^t<mr{-dvR_{{(M-X}l$v&ZlM>!;uPVOhoufAO=QJpS&VKK`@6ef;;|XVCxsyN|#1&!7C`_dosVPapr@Hy{7~&*<%ke@I7Bg6Zhd2S1+UV9KKh-w5^|{@oN8V=eKD4R<$34}Xjo*~9m+7<-RC_`xig;G46^w2m@lvsk=#`750d-aA+7mThkaS4YYVQpO|0j;{yptJmrv(1k8ikdjR#GY^rOlllzHuPtRjh?#*_ps?$FAWBdN2wsfdl+FqFX8qJhJt?T3XA9UGJEbgLbO^(gs%bmB6`QGA{;lm%UHS45KU>bWf>T`wqgqYEtQU55H!C>@cS8ahSJ3_1`2rx_<XYi8Q{}0BJVgaXuz4=_g6mzcoi#I##_bFFMc^9qRkSL5Fu4pfj2@+EfCf>4xSjx)hOZ`l=-;#9;ua`kiu4FCAI-;b!ld^n*=(Ipnex*@9<M+3)(+ZAhWD@Yix=--2g1CllAii)>c%|dL$*ZBiHt1g*y54FT}je6r*riwUj-TdcH{HgNRoOVxk>MnR)forC2Q#Yx5=Ow!J|y<>^@bQvXe{)J-Cm&2M1)*F_)OeQ!*suJtgEPI)98w#cBN6A$HOIU%vN0e)$)F^^1S`?&BYS9~4`EfBfe^{P^4d`|-Daa(_WtM1q6S!=DnTdGz3qKp7u>@LqD?yn6U|6EK-aAN=)kz-60Zd8WPbsWxR)1)Vkt#{}At-%(QNI19z{<T3S}4<wJ4xK_JnKr-Cm?dV=4T$6$6q}>1I5C7oP_rLzhyI;TGkLK9%k-vDgt0yDC|4?NNF=*e27xq@`T=}&2OI8Clqo=Bwg@{fzU{%1Tt;p54*ZgcKqO!g%mwo%eJq7J{+vZOxDB9ZsnGC|%uGT?4Eg0F2<SQcYw$(23tjmOe%AvMh8H*0LyX<S*u)RVH9;b2v5sy>ngGtrBZPLiIbfN(zng_bsH1REx*9k5rBnn0zc)1!SWv`8{te67G19JH?mkBjZpnrc#@m3T&ry-#ONq*u*Sb4L9=YrGRHQJ@N<Z40>)M`?`e4a9q<4)O8ew_@b;P<|}5i+64w{I%Kg}+Og%Pb&DRfe4U&T%9+iN4LB&rr{sL|=Ms6E>o@rdv0yIw{u8gOg2PzH$e;YOp;nCkWGA;l_;PC+w+69bS%6;3{s~n@U(t%i>t`owcS;ArpO2WiqRoAhIjfd~!`s?p3gNOu`dUyk-#SQ$C-wF#-$VAPADu0#Uie(9cPR%o#9J;Yc)Vd!(&oKP;K&Tq~JQgrPa5q+O+qNB-C}txKK?S98JHI}g0G+4XsqE1@C)1$q9MBKEvYb6w=>DrfOMMGz%koQ9g^y}4F~O!t1PFQuy_;oNCQ?Og~0*vTwtk_vKDoK<jm>4ZrDA{BQ}IgQ{YpAgvFs_B(W<luVP&y=Q(XZt<5d?!zkMxQy)jD89!aZJYwK|C$?W7BB)lp2A~=~{r-y)fG1b?G^HD{MseqI3cl?}OAS!IAqjnN(5d+V(<Bu<~R`cL8?grZ0D{%r{N3Xu8j_dqiDKIniyo6z**%bGNCvqu=bE539!W1{V#OT7o%(6Aw)HHO1g;ZVtZEP4U}>`Q{tM^*V8}m%49=sh_On?9We!4Cdlro>jO%9^W^wwLAHD_a1qCywbtP5_C7+GYUQ5aUnPvLKVQNNp2+UO}P~X_Q+F%@A={N(|mvYY{q~iotOQI(j^;^kh<V&-$TBib4ItCef8x#Z`^wAHt|=nZ&+rZFR%^#WXC?5q&#$a`{iRI&U_a77Sg@oy(qsPy<AyR37R5a9?Hv?4e!p^XPE3)0GS^{=bHBNaZ(JY{xYsgNmoHu0?tw+Wz5eim{4>zT<~PR?L?&7s77soq6lpOVIvo0sI}t=XnrUuh5XPA4O(oTp1KE1baRPRCBv1CXwOFwH;<1Hv6pF3VZMVomX>y+S^QNS@212TZ%89%@4XN#`P9rbKBN?FfiNt}sR34~hwLe%2C{EV=}em0s}9W-S;ln1Y~@4tY&c;vd|8#`DB!D5F?E%cr7fF|J-H?r$w;A|M?^dyDlF!%AZh{4b9P4b!VM^dF2k3$=kvarQLK4+fJ|+>8oor7MfW&lBVV4pN;v@;6_&PW{cOW*>y&RS<k*k=tL`>zZ<+H^T9N>pEuUO!ht(YsLx8=Pr)g~>x<c9`=i;84IXJURbcdB%kloId{kOWTMmfYBC}K$#`8)g%?s5f6jNq6<OpMmB42&rg%)l_P9XbK;7G3mdXWK0(D5QzUT7j?c#aaHw_#bbwr8gz<rRs6b#%<pn5njKJ{2(tQ8hwc^)}S^iStO;x<4))g3Ef6XM^j$%t!;$m^r4->q(&*Lj50<^VT;<ByAyP<e)?G?8;Xxw3Y+hBm@X$VC*O^T_KNQT=VGLBjH{T{l%RJNKwE<03RmvwRPHbm8*z7+?y24^<h$@^nkood7Ibb2NHM;;6_CWZVB}apDnS%o-Iz|x2VuQF(@u0`baVV!^`;!StL;d#V+lF$)E$uJeP*kRH^v6ts9RVTQsZ)r1*0d?+qE~VudsWZITz(^&B?GDOwR|)$vi`vLQb<v3Ok6CRSQBs5@ZRQbDyf!!_RR@G`9wLi_nt$NaS_Je4h(+$%s>nY8L`$EI&0<+oE|r7eJXw2avolSdh=gDO4sL41jdW0qz>KN_R&uvymL)DB?74`QUS|KGO|4fguL;i;qbtEj<?+SVGaGNC5;P-7}dkw+)F2DUmZpXAjIV4Q=28xM0q42xSh71Op#S)cQ%5&-;B^F`F=+K`tb1gq<Yq+faGs3@BaNY4t8jrzfXvj}}mrAXuiXT0+R;dm<XS!3<}OUwebI%TvCxdr<5$GAu$%;&&vROBRiUE*0Usn-SOw76PVD$2i8v0bfH&Vj7#V@eBtruR-x3Gb@M89U%Qkl6Jn!BFs4zOPqYXD2$%jlY>MM76-0Rm|~##6%s8XUmg2a3Kk;$&VnLNL=~?|*Uvn=4$sinSsWiUI-ZrvmEPHvgY;8ZwhCcL8tFO7?)B5f*$LI998UlTe`=p7>F|vBxPE`-E4|xaeC6g>hyhk*R@bD|wK=IpB)S8c0IH-4MK{#Uq{sdO&Z5krHo%Y<U>sw{z?M^p=VB7!c=HL#;N?8hWf4l}V%VQ9&T|Ja6~l56YRWvc^xZ43-TX@L<^PDhl+N$lU;H;a6Q$uDC-Hc&`d;)Jw7UrET@0Vzyj_V;l~-<@u{3YqZrZb_{R287R<vG8fQ}(NmP9$KQC>qq<y6i9`OJ(<pi(p=RmIg8sU_S>q9Ybm$|I3+k3KVH^8c6J>(B{hG;v)$x$l=G*F5PYnU44{FGKThK&n(}sY0zzJFijWQg3bMrTviRnrjZ9#S=$ul`rSkTknY1=+;Xcxb%FL#+nwj)hg(RJy-=Ps3vvQ)>fcD&4N{4Okrwkc&)o0+jR-&)qTVc9g1I4myYrH__{OWq<Dmo$c9yXzguTzqj%<Qc<JUWO_y_4tzo#*r8&J+trNI|X-9)WLkC-$Wl==fh$68FGQiY?=9c_}It<sLc&>G))M2tK>eigLhqVL9CwsgHyCxMq*Vua;6e($rUi|XzniBwD56X4Ycp9M8=cA&MiHi-;e2EaHeaiPN!?X1~87g&^K7Y0Z@J?|`Z3wq6w9dyki6rOK&Pl<iJ0hd*tdIM@S$^g-#0~t==z6x?am2AS=w{yMBLoAA#R@ie&m0aqWY5p_OpAQ^hP0{hX~4xnv2N5XO^EpQl`Jy-H8dIEmwcd4$#N_2P_tYbgk`=23hBz*I_5=16`&3*O$gbGZRv|3+EX5)Rlv71m5)HzPzvuI_vkQzZ`G98ETmAF&6i_wLJ}*G=~bQycPr{i!~q5T>vFnB1G0Klmlg!o<vF()kq_QBqKwaLYo}O42a(@vpy0RdNWy$5ltIA$;9UG;F>X-UM35_)sz9dM3f<A!rE>JYeiu8_uo7Z^>_GsYhr*l}wxhx_`7qW)x7RKxbD_myoa~Q;bWxv<cHHUQ#q~URlVc&!4!`ZR?&hMCK{nukSAujzyW)f;O7qB$v>0Xi)<#F2IG5cDGSPJHs_de2)|ZbfFhVt}RvHb{dlMDnmDj@#Ql=Ey%C@1cqa$53YpXlMQs-}t3b-zMQS<pQTOOuxLK;J4%3w5L>1>SBf@k*2MlGEOq!v9Bs8ln%{9GtXfGQvrO)VazVSymguOa`cwWd|I;e}L{yQ@*OUffkHHzzl%P`45RIKtJkJK=3+#g&kks~KG23+{S_9bgd!yUyJaB`9Xw_dIZbV(EeMZe9mt(lZ-|vy#JPvzit(^E?;j4lwWLI8+?SA0*LK1K~KU2wW-ruAx-IND@pj&Ife<{56>qs<XovmiFUh+-N_44MdI4Fez->|C_t#YyoN;aHwZ@ouMc%<hC>w3CPh&#x`<$CK=z>!a=$+H>j#lr#0ax23JLiZfgburGDmQ!JQc32g`m!p_I}ooC%4M0tM%;5K?+13L_H3^SQ!c4og{};RKmEfkSs|ar+XlsafAmB6fyp6+>_QDt4@S7fbU8-`8A&My^+NhEw_e-G(VU*h;QC&-o-Gz?lL}@m2%?OU#8CPawn?XRZZPp`uXN>BRnMme9R-wrJJ0O*D2Z-{3YYAutCTm?H533E%rdiZF$R7(FnLcq|#Sy&J1stzX(JSo!eHc)1|x@}W+OSnEopi{Yj)a%8<uZ!zm_nJ+LTPucq6f2fsHXkWwH+=G*FFQ#p((E$SmDEJen`V%OHh(kBEp-8||tfhk~oP-RY4!-i}gP&6pbsQfNyOiFc_SNKz$?VlzclZz}3}BbmgPWsA557Bv*6U#AX#o4d3)nIb-{mmszh3BfM&r{Lg8svQq?vs5;4kpOxF@Z20W-;^!W2+TU%q?y^#GEDTg6HjyMVe#AO0AJUXGSdi#ZFS0*@a210S3Dvnd5%px9gv)0qBS`d{6Hn~y&Dn`v+aL>N<SxZ4q>Xw;1<;3vUgi8l5gJ@`+s;*<+3<F6U%-A51pCz^Qh`+g@uK*%nGqnJP7$>V9aKx-J=?sj|YbJfk<9zFc03C-Hi^kx<_3(7n(mV1vr_z~37<k5roWg&w;;l!<%g8fGizSl2-4H@bWoa7(lhwtW>hhgPMAN?ePWf@n)QZ)U~g9E_+2K9p8K6>yszzxLPw}Y=p-2EJdevn|yQjDEBP-nu5;k*veAHBzQ9=<ovXQ*H*6bxfz&`HMms2?p&>_5;0s|Aq-Unpe*P^OQ#XLOZ7&JdDANUiGZrB*q8aCkrX5AhynmcLIc`6mEM%XU9_mD{-=d^z72RK&$v{Jvpk?mq3Z`@#D9dcc3TSN{{{N~$YGI0tTNgWEitm+l8AM>(NB4dl10y_k17E}1vEeqdiYMj7)e*L(2hl0#+kWS*SF@oCHfCY`SldH7zU#3lTl-}xQo+K#K!qPs5=@3z>@IbC$Ono>1vRc3Y$xV@X8JhHbI!T6(}JbLhh>HhheEga^rZq`_Y%5mo*?vr`t*4^Lg-Ff--*Y4bHPna*+{26jCykys_5pn2X(U62--@=cJfkQ<0aS%x%-d}C|rhHW(VztwtEXlD)RKxU&wf)$<vZ1Kf8dPQTsj|~lN_U9rGtP~of4p?ZrLL!U$0b{gjC@)2>$C6KcSKt!jd*-rKMfqdH`ACe$>yj&4O$dGnFOhUs}j4Wti}`z#Zg1~70@L2No;_gR=xF-^&-ogR>>KpiwhqUI$Gf)!Wi*f;clpM1Q<8AIb)^+(ac&lLebcXEB#N}_4C9x=S4p&SC0V9>ODR*7n1{wq#>zRGbg@)b5T>?R=JST?NY18L*|@d8gcqmx|@K$=VO!PtFlEnzqmJbMu3#w&@%F|0jceznQi4L=SA+@w>>Y)xLXlpTt`rfbj19&3+AkoUeR3@h3HHG-vUso)Utju1)>IYzSEv8Q)*b>&Yf9S$!Ivf$IMCAQHw(7c0Zb6aL%|YiC|PMaz~(^y=HrVJlku8mv#9|d0%|@otJQz0wZOtVu{y-DC|mL7mk}}rs~K~h0hKXnt^sG!ndoE$*lI-si)Hn)bt_unH6ta;bRyDUmJO}JkQ!%wU3U^Ke<C)?Rl#RXH|!r_Ltf%Ue97~16KFn>hVZvO}bn`7Rb4g&^7D_6)|1K4bP-qAeJV;JL6Thfwrsxbo=}l$bV*sxXQ_dDi3tk7pp$pRh6S?#i^?b*Y3ahTu*hwym;=XDrfi;<bpb@Lshw<%8AL&I{sFV31=RKD+YFVbFH|1M$hMX+-v4~d0OsO$Im#JPMEKM=xkLGp@|>lTR|YC83rd=8Tj&At#>2cXZcq0!Zzr7A>zM~_qsiH!uE{kWslr<T#EeuNSm=IzcQ+XG^>;%6<8Ry>?wDL+RWL-w8u528uC403{h&%XwUguq}^$R>+3=uLwhu<OIb*x4Za++r$RUY;Oxwz76u)=s_sy^Ib_(QgXWX~J)r~Ardc|HSX(LPyW-20jJ2{{C80ZKq5XK(M)7#w4R29a*^JKRA$e)G$i?SyC9v7N0u~NvDl7{n;9ElF&h%P{X$()|Is~}TK)OS3Y~Tl$ky9PuHXCNUk_BL0$skg%4k!{&uXH9&(6)p?q@GSsh9d#fepfM|9!1!E_gO5i9*n47-`b^;vus)i;d7B&t6hr}_H<iSYgcL~p2f-4mBlbB%kq-EqY!BXKO&2k)9@K>=&8kZb1<=HgxBhasDMaZIs*!wSUeL7q=R)m|5y#>`Lm&@XJefXN?yDxI7+h)9nqQfdbD+;B<pckDD;ycZ@EQ*wX!W?F=r4{-Ph;ZG8gTjbU?MCl+QLPyWoEacGj@USm6wGzCSlPEe**)g(7A%a)C}o0cYn35bu^A>PRf|w>Gw6D%gY3=2-L#0XGJj3HyYu_PS@N7#k&M#5UFyV=<@JS)}RpIR8;kMH-6y%@zaF*NJ`8$w?q-S(#fiscRd}wf_rt`U$l',
 'diagnostics_snapshot.py': 'c-qZe>u==Nb^orv;zmZn;d(@^w%NoOJ%G*b8rjB=^?F^SQ3aYqa%O08$O#{k#_9kq{2}Ov0$sF05d=oPF$~v9gC=#0qN@-6ASeHa`j_-~&b^nsBxgplV;B{%EAl?hJ@=e*&+8s)b93{47J5Y}JdsbmIPQydG>W4n6hY*TlQb_PU;OydV=>CoDF*qOcN&YVO!^{9{J0FFWGrT0Rzw~*=g#Kl=0-G~rCA}oY>civ+~CQ68pokuL}`)>HQ;_)CPkR_MG%g>GA@G1FZ5(TKaukSpjSlGP*2kDeL??!kS3v?GAX99evZ=WiGRh>(3ysXhuNO)c|T7RxyH;ZCK#cYc#Ky)Xcl|LD9xt&J)h_Lk=EDGA4IcJ6o(rd8`z5gO#_#}_Xu)doO)RTO6B%oL*S>#=JJ7lval$#gpqOL)C+Q+<1mOKJS%!(;-^ICeXA@+Td!EQ&2z&uKb#e!_vYIVvn&M>|J;kqke_Yg<<ws<q7iVs*(^+g-pD%VRT6I+E(YQpgBO<F#_4>~-cP)IL(xbiX~AT4!m}tZ@}Aw<;o%XtWa()x_VwFoR7~W&6o_^AN;}C5tUs;cfgK_#r$dq)menW`AsWIAqS2G1`(nqyK;cM(9rQ=DUaPai74wHM)`T<IM9^%zGehyE=0=AZ=FfV@oXc&tIHc%FXes{Su(-o{5#~K=?2Azp#(`#zBnU_zrOw;fd@_FYW-kd(3IBad!92|(oJ0%)DCYsr;k(>WQIjDOQu)3F-p8Jb%*#DK()(5fZIOiyXb%6HKz0B~b`a};B5{l`H3Za{fu7#^aX25yLlKnIS>8J#dScNeFSF3~azBdpU-jZVw28A(9wm9<C4Q&}f<1Ww&4R7iwg@p$aL-aMRAPly`*NH@(E&bFNHZ;hPHL6=Uo?0j$>fZ~f+zQEhb1}eNjcuiqcKP$nFX~09=LfqoqE~4mzF>?Nl!b6Dm)33f{u!D`Y>2*3oA~?xkV3Aa+3NUD^-U_SOyryeQ0Msa71^5{t7(yMcV}CJ&T)g=Jxl6#q%uTB|+VfBbxDIY7AQg>&R3Aa=FggKevWn9_7|RSY_g!cv0*P;}FPNm=sd;k0zJ@0bAl<ej%_sQB1BbK8nOR@)F@+U3}&d^l+G_$6|VQ@r!6z9LouLITx8HimQufzWDOl)y2>0`PU*xgE&pIV@Q>?s7V1zNE{Ehb`<6n4Rzn4L02~l)DCA^I?lp;87<dvNY7fqo|cBHNZmw$$z3-|e4sAA2469&#nl@u1z@yV&DK2hQ@K2~6Dbd^F!pA77`UoWQH@UCvUxp-yx6L&xkGk1%2P76UeQx*@Cxh5ww)p^yja6kngnJ!%!!+tz=Q-LTCs`G{JCVkpH63SScCy4$eD8QRl8eEqTH?b4MyEo2gsj<K^b?ZR_$6VlN+^fth8P27FgBIM0?$FAZ|-E-yk=}JTDc+>VqQ}clpp6n7XI%+gzd{9j7KU2L>nmHAto|_6+h4gh6V|A4N%&V`J#RVT*gjUK{Og<_;h~CG3@Y?~6aRjRk75dW9Q%z*=(!6RS>ut%*3O32gu>vp9F5bzG>-k`;wqmWh2!eD5|b%(6}m6)DZVN(31Qpj8ldz#p1)EdV%;on248vmTbNMPsYpm0bbesR1+7tI<);x{?-Jx}?^LPfevZ3DbsFr03uHN;$@V`0drjCx!CQ$5$^tOGGlcdhuzoD>dOHz5Ez+s6_;r2#uLt{wk5WHoAK8KZ#t&5f7i<6Fy-7t%xr_76kVfzJi~O9ZRBFJBIL&ec0p7pF}5N55^MiFw9zCs3v915EeWJ|HJGA#3*poeLZ&y-6T#YaGu4xQ4)}MILxAO6wiS)3@vCU*r%jW1EB~<7}{QY1+7@H4)&69*n7or>^crzg`K0;?r9_&!{iy<XgPF`!wd>4$4Of4)q&m64L~aEp%3zX3LlF`<7g6(ZRR_W?cIgk-N8QypRo<7-tTw)#LSEGrr*LEGmpb?*2BT*-F9x@#c8-BZr#$LpuU_}Z0&Lk%P0;iCb7rSWRKYdB}M@d4a3Rj#|D{5aqriP8?QYbi8*EAD-Tdvx!Tj8dPy{rD!UGkl(p1-Y$Urbj7LXnsUsx2R@m0?Y=(f7756!;9jrP``9QSnr3A51+SgAr(q>uslM?ty^!s`XG>g1*o|cjd?P)~ROv`r0aXRc-x8zprMi@jx8ryFLQyYdHk)X&KO@V+S!bvdvSW+=OPFZs@;Y!LX0?=-aF{B4=@tyr@z@hAw6oM(I0Y}YsuZbWZJ=lKd$!j7RL(fET(O&$ztX#C>@;AvuoLqjKHdT#fuiqJ|RD>7AK8^;AE#TeO4m+Qx1|-@>pN$9o3eu?;kyMd!gCC)qi>QzuC=6gx=JYfvikVx#{-pHcsF+tR%{6rIDE7uY+@E-v>r+aD-Z(n)3qJam)&U(e+HV-+)(YAda~DL+J9UZV`Z=F?ckX_Vw}Qp0*CvcG8AAopAkX(<e~s;F8DG}aphOd5HHJ6k9z$n0rdE3$QiKPqTZ7jvEjsIFHM3gIUbIEwP&4z0k5Mhrs+pDPa-nXH>jGBzD+Ut$82Jcu4ZTB)^ntw;L=a)ZdKhxZ8X(}TOrbCUrC;<`Dj7Jk9r&x1a+0xWV-Y>bZaJ5Riqpqft!p=uD;zJ#xMtP%2Q7;!`!v*Zi=9uC0eu)Jd5jtQ@>8sjtQ2@<%9fl=DJk}%3c}@{NGB7}{n(oh1CKO3t9yxo-HZg_ALk9r&<yApq7*>d5;-5;SW8(Z8EYvhO7}L_QRV}hV$9~K8$a@Lji62Kmi4sOCh(JmoF&J!G?W}m2Yp6E+m6JtnNoY%jXpyJZSqw)%_6rjoV)S}kq1yGZF?${+ez|B0@cD&%|EPGtJSk*!}R1~%%>NWe0g8&bh1X0QUl}Qlrl?l9q87K!#YSYmof$*r_)<aX`qBu#xcAfs1pN#O6!nh)eikQOgK5ILav1JN>!{&;T2NnE?H7^;=r>}HR6FNaM?<@oO93NiD6vjGbyv*8Dh5~q%*K}Er7-dyAeext_)bENwyN=H4z7fvg43Z9UIl5-j+p)I@VNBIauO!vYgTkTtR)QUnv?hWv|!Z*K$5BktEy1sLAD<tg~J&u3OJgYd2ZU)Pi!Q1gib!+meU9wE{IR(?!(8eWaMz=?$`D=4wjK;WYXs1$ViU0M%{|<$wygP|`SL%V;PLP>;1~!xCyWv}~&ez)A@K(7Ak4+m+;%q>>$dtvhOOzM8JnzV^ZZE)%zae0>I&9U^%7x&f2)qbbflGO85lzn#D>oIvCec)$3Z{NwcUIS^D1^bD<^M8d;$8*aJ0e1eehbep_Byv@bg<#V5-{of=E03p7P*F^$HI}$-@hVHCVIf+dU#<$@te_eOE>6CFp!^P^Xa*cnz)TdT5c4H6QHVODLQScjgvcC;K+g3^Kuqen{8CNYzu9sc5(6w)XGrfODPM0Tv^5$~Rp*|$i+@>c;_(UP)3l3|s-}??!LouI)d}DjCNB<+Wid(<UlW6l`kM3y&iANq5`<r}$f>sv6gZt(Ik?F3zZP{he(Avw3IhuxPFh3t5t{m*#ot@F$Esph88SMgBuxxPWyR)-hx(C~ugwc3{Vb0w}L01#!L#UQ8+oD@*5~+cH*gcJcVlvpdefv)q2;Pf8jU4RI0%;rt;-w%6c4uBd#%u6>tU%JU1@RvzgE$-&yPyj0)xB7?!R#&ohln+b)6=cf%$p6oWPUmcvv2_pog+O~gj+d0^>Bc99<fj)KljTF>T57dBhu%Ky=}R7gB&AkZ?dzDbmD*i`X9wP(ccMk-<xr6h`kRRa4@F{WOkqx)u1`6<<Qs*NBq!8IG&rH!D9Q|a3L1ExbpdT)>{`J$${EaSU7vzvnE$(2M@I)%-P8=pIv^!D)0GRR2pPgXb+gjK=Fv<=<4FbDH4KwQS>{Re}GEl(gwP6!kyC33z;e9+=a3&+aMAT{t({oupCz!nb7?lz<85G%Fkl>tjw38{N>+Y{&LQ!<XX*xEYBmI&HeJ3!VZloVYEvb+JD1s_0`2sOGb}IekhYCGupwYKANEU3ALX^Kq`{_#0~D3&pe4AXwt=Ia_7KIDz8o(gZp<5_8@BqmUvm1yfzetJnPr!2^;e*Ad-b$R5Heumpu!oB4)B|<M+is(Qn#1^4lLVGL6Z6Mxhfj{F@v+dH@|T;m4;LU{KC!a``-}THm@~J>DldQ%h8hNL|V2Z9r5D9koLb<-gRS2VeE-D>xnHc0ADj>Ts>U<TBb4e<dvEeH`ZNoG#F{XYC5&kMa@^mE}CLo7aC*-^*lxr7%)SzB(+@x?0EZ7EzagTM8X0fFd|(oO@bV-j)NU%}txM2F6%Sw(1&P(n-ky0ks>(0Tm+!EqF-1lCJ?QKosW(drHS0JjDTc2DTnw{!0AW+i$)>^84a5c>q3x##6(j!1f#RDlHs&WKszCwkzn{Y=3lSfE>0drI|;TWxXVps7q))tsij#^7_@qKa2OOT@bAI5b&0K9LEv6Y{}*4iBcdW;5fQJ!!E$j#M#&q?<u@FSNz_fksU>OlPSu5@z(u5-)y5r`>)2O_cmYKTyq!1YwBbP3O%r-u8C4}p$GpiLNZrrRyu!^&_8C|(oufk8$(nFLqX(|MHzIW%61mM&w(SHZSqowph_afq_5Q3#wE|yi~k~hMw>6mvt^*l3Mil|z3fl^@LR8imLN8)Ql)CgDmx`%wZ^Jm!nh!(=Z8<9J{IzXQqEc_Bffrn^gzK=NjwG6pOicXzvlp%hzZ}CG3)ACVR8~>X+jZ^=J(@yF~P;G075^N`20;NCNL?SEr;75(G{JfSplHpgiL}I9jY$b28_z4C<D-!VFH!saA6t&s%!ajWgmje{|0SY1lbPf1d7Co=;DdHOBl5+2|@78Zpg%Q+;kmZ{>zkjhk%2)h)KIF@e;h-xZsgQOowUFT6X6)0sfs9G5nUJoZ<`{n)3P-CYn>k>(LJwpq69;sh8`Maru;J`mc(n!%GN;FaP@j2LHu>CcCXhX>9%9e-1w4<W~fyV~spb)56lqRO>;j(p{=g*B?Ph(u)e9D8yY-Ogx)tH-(Te&$;NeHY5C!8{~EY({?sC*iYe0S#|inF^5Gw!3+t4H!n3*GQRvJwhq=#h$}#AcS9n?+E-m=$!Tq7K-8O2*n3cADKWFwJR3tVg=)I^hbeLB6XpH^*Dt(nzA%sBT0o{rOqIH;u9H}Hp;^C>K2W3bhqu>IfDY5HrULOm`We-MQJtofSd*z4ZWH(Ptrjg_t08B@@OTF|c6kSFpLYYshs}|?b9GQcODbb}45ibPFZuq{jEmtYa&^;VepLaPzT$=*(-5e}lCz8Y9+xc`nPaDz&XhGa)Dz_{tr_bk%E!!e2R4o_zA34v%3;x=A07R`nM}ke$KIGTIp>Roac#=YGRr@kdvHlk&2W~3?99XIhT_dUdY#Ru<t!L+Pt36|BAwXvN27>d;<G8ICMY$TMCH_4ck^%T?yyGj=ukb<O-r{<SZ}+hM%nF>fF)BN%b{<i#j8{)!leRV8I|h7rm+t$l~r}d&X1$nt91Ee6kJ$DP|B%NoekAwR%0JMb{{-^^|gCXA3orM+95DAv+|so=Qde@Np79><l(&suRnCAM0x2NHESBJR_$vCK*KX4PW@$~JW1VSe?^F<y99vJv-^rL3?)S5stafJ)xWBpy#|co^EAMy_M3t+yqpFYo%&vpp0lstW$p5`wkbP0kZ*Y9X17aalq*!%c~rM3lQa+Mo=B18aBz3ofVJ0fa>!yC+?TJrMQdHhRf`>cQ;0})9&~q3sl#CzAofBZtmiHKcjvq6SF>J-zoCGoB(seq8ltr(eb?Q5t*I+%6{X;Lv{`R#!3St_T`6kq;;P_G-{{tX?yXyVk-J!TW4z+ByIJtMic!l8<~vm)O)_x)7PPn#1v)T%FAv%JwKuDydDW=ij=FhoHK5gc)L&V4u26KfC;)=Wpl}Mh-Mm;z#Cq!vt%&k-_yu|NK`8e2P2pQkQd4{M3|LTNn>svDgxv1X&|c0Z5c06IHL`8ws&v1ZXcP&r1e6~~GZ#`$>Yrfgb~48yYCLyb_@CC=y6U<%V#T)}d=Pcjg}#Q;B^R))X^aZG<(uYWPfCOWN4PO?wA>=CE5y80%YY!SIn{Vs$1zwbBpMzaa?CKnQ$nH!>UgMhP4nn%1&WoxD0nz<rVd(-fx!z?Dpzu}=B%PhS1xNcxqjKEczUOOUG|r6Q+i78#a;`<(8*i-OAuN+*(8yJyngRyYkoa?tvU%^sNLgGZBiYHB}E`NSY^!$`ob0$QV_g63PQfSm3ltS=H2C#m8|NG0Q*|s#YnHToEOzodoT3a07nR^aXWdfd$cSof%)bgm_+-U-1b-WJ=-au=8{J5AGIuh9!CKl$66}Nb#)rUna$bJqa;zgQKpRAtw#2!*|l0s@zqO|W@G?S_dg8V5M}DYMKUGnVg~h9V3xc`ZFQByDNbQH3JZVI?u?OQz@=f?XkVgJ8x8`|X_>uXQ+kbJZs;XxFD=f>kyhe6v<_QV6%TWkt6MvKh-=29!Aku8SW~ZXR+%`$x~k}1CsFNFBSj21^2sO~SB-gEX814woa7i(H*3cT1^A$WYF;BqZ@qKxwMS3?!hQ1a@taSc%4k#zZ!;RS?!EKi(bMKY{moeWq_0`9nnIBhv<+9VPy*68ARP)cWR`VM-PM%Y7S6TMuIhkoN0FV@h23^t`d;_67-7ti#S~-Y*Onj<s;{$^@O{e%r^?>0K0Jm_llr5zQPqkds^CKPiUUnu)R?tK5XiSgo?_nRc2!5wRp)gv2EHMg$x4gsm(r7<5c!jeHZUYQ@KHYM6RroP3s73(RF?Qc3}MwMI@1+Sy6fGCKX4yDdGh9yxB1YlrHCp$Imum}b<=WrjAY^1J@HaEC{v$X|DTD_OX5fPUWTq@a9J6Rsk%wX!ktFR8};WTX+z52h4ignfj^2M)9K2a$(>a}x{O}CICC)_zGlJ&Jxg#!N*$mY+%}pj8LU1pSx$}yUDY}_i+GpPoI{hZYhqTgrw%30C4xwb*B_9y!bseDaLh#snU<~O4{Iw++4g<viIg9z<*cnhCU6_WAcG5^;cNb}tVg3Z*Sx#oF8gSu8Lw6wd|f$i{rLkc8!rEo2OY0kes7+K;j#W0aT&k1Om^A!jrhf$Dqm?eJc7Pnld?=b+x%M**M0)13g&R)Vk4hje~nirsKLTUbZ}u_Xy^~@r9xBRvylSRB;`LblJRwi{BtADy==^dr;oX(7oZRTfAWF<Oo2;Psjh1qGbx}0@^VVg+S-EDSn>-nnE${)WyN^4%jI)0kc3!G<zF;ef5lu^i#I>fqalv^D389mG6@NLhJ}s5;oQ<F3>w2*<=It<o>BPTDATS6=%%9ue?bUYcKQCub@>x=*CqP9t~F2}_7O8S{tu{G4(I',
 'export_snapshot.py': 'c-oCtOK%%D5WWl0e_+)Exv;c0j3h8r=un_8(4t5U)X7B^2x>WNG1-?%u5H!ufA0**)x%2M?m^;m-ro#A?qo8#e(GB5nPNp-cdGDH_S-MB&VmE`V46GJwq5UjVqXC|X5qdE=b5*v0LM&&JE)-DIdjk{t2|WOd@`AgN2ZQ`?H%qr*llc-+f-(~d;j9vW;`lwTeD7i60sz?!PoI<r`T<kwe2HQ4i#VT<`@K1^Few~R^yR|64O&!Qiw{B_iR2MF?@bx&fDB|mGPBnz=@2h%C#))IQHr6^5XL9{L|UR#npSP(isixK~+Yx-~amaBN-DkxPxfq(dDD@n@~aCcF+iDidLi2g7?0hUGa=ELpxA)%7GpWzPAlqQigfeD(wX2ld-b3j8+73v|^?VJ2?gKM7G)&Su4|sp|pUuWs4p1`Oo&Quc7fbK_WD`0zG)sHVY|rTS%E5n&wJtsgjN0vsv4FJ`Jrm7WCrQ_K*xv%VDHs7yif+;f!*$L^1VI&~}dRN#fex7J$No29fqT<8k0}RTR*9(0LH@WCu?j6u66)&vwa;=Bqig0WKS`3VvUrG9uQreuzbgQe%>bjQy|(8&RAcc10Nn>@#)u8i&gYeqDEdn~nlG(mf7JfzytM#SOWY1}aT>2+C{9cyBBED@~&+nO;0j9E1ft5v*Ld)IM6_1|BQlT6*dFb!FTJuobAb&n-UOsz*t~Myj-S8+G~q!{GPeb(d-Aw9$?>Rqr=3jD-gSj~HdW4H<VDRYS~F6rL?AY5DvWwup3o2VUSZPAD!2^jP@IVHYfVNytt~ZBeOutrfHABesnOIG}dix`Q@g`%w43$8+bm(s|{3$5+%gJ`4w^W>Ak&F#x(6j>r5MveTA+Y@6BOhP*&%aip`M_>@JWtURJB_OZet2a#p@2A;IJLuwRZu9C{oht+>(bBo#`4y%o;l`l3!4zAcht->kG<612D-l!K-5}C$-c!$9;UFymLK|+)nR2T@wI7?oNz21@H>5S~`BGaM%KIBQ`qiB^>yjoWf)4io-PSh^DEoC5Rnx?6wMf@AM=RSY522%+3BK%I-F(o99$3dmL3a=<f9LE1~e%G;AB$BveaE%}f&hsxV@@1f0-PfHHcFLe}y+!hJg)xg?@h(6PK4bjvhVKWIlFvx>6|<*<0z_q0_<bHoEf(_a3bNd{118yytda|w9}M*-A>E|w&bED{XEgFBgJa6?lWW4|e}$qO?~k_vVI%sAi2fPLtO(bJ<MY6+2{If->9-x=E)N43NGKgX84o_qolM^yc(CV;?wM2KO9;x4kEDF$16YHTxU*qAl=rquX)2&BpI=_~Jk+Q<+_>Txr|D_`4iH1eq~n<oCKp*8fc4hH?kd~^wM?;BOVV8h+=6PtjQkO1i96=6=_kk!3FqZ_^dWrE$yzn0M0u#;_y`=1en@GGMQd!^;`00R-60Wt!<3jIe5933N-9Xoz$sGFwI(G`NX2iI@#tR>HBWf',
 'longchau_snapshot.py': 'c-p;PUvC`AaewEh81o)x_PiWY(%t1k-5RG*I-g>lB#S)90FM~V?#}LxIXkmAGfRp`6EKjcATKeTm%tD4#W0KnF=7J&f=CGbAj98-`6T&ORd-L%za@1xBG}8_p6=@E>Z<Dc*K^z3+fTA|?i?(_wKK`mGD=HloUPMon8#7!gz3}?og(^jjR$d<pktmbqn&uUit-{$!!pL`PmhkCJ9)M)BY%5)duxU+0G1@tq>Qt)aL_x;OXmP^%P8McKPTDh!hA2zWY{z;qcUDbdW`-(bm;S|ERCdJv<&0KUzc%Wz^}r*h^T*1W`Q23CzyW~uk`mtSS*rwtbfLt{`MoRNPl03<wB-f;TyW^X`a&q^*uN6?YY)zJjtenN+=N4c>(}FBdEcktym>-xg}kam&-&yT9nIVYwHJ(UK~Dt_S6|UZh|eEV25_j?zmf@K07=*MC+xSE`Z89o9M^>BwPAtsXKJs^Ye4R@A1{vgBM4S51u><4j(;vba3?S1<i3x=<bw@2)i34_)N>uc9@s(B#E|9<Y_SplW1oKHaT&(9zH&N^x*JO@X6yxPaYCNo;!_-Ngl6g(+P><V!0}_C4CLC8CjCe!Bg}zkBd|KJB`D6nibK5bb6FWVR=D~Nqh#TLeuj+IwQ6_i`>E1!$+Sy`22}l`FWO~g8$ro=MF)MlJJ7Q`+xC1%~h5WCZ?zN@RN21tchR8clP|D^WOXZ37W_2={%y=J0GHT%d!5s6?aGI;7*w58SkjfFM=6ZFPa8f8st&2UQ(Z<d>vhF{ne9a2j2^h9(-4@T*(s8m=+=nvOfA;+~Lo={CST*-{a5s`SSz%EOYuCC)wom%XL;V3bS}V2@Q#-C4Ww_saspqXy(L4u*izi1A7hjc~!u9ou`hc8IB+M)F}-iXIbvJ?!Zs7^C<TQPMkW-U<zEC<;yUMzlyvvdRZ!n^c4%V*xaxBxlcbi08ZRT!Oo*q5>D{w3trC`DYnyslSh8B9(%ca{6*<}vEJJY_fEb|9UTQ{V6VgU{vy4+H@p&nU!)4K|D!BUy(CW2s{|`zMKfSQ4YV|Z<_i@Vjln>7Q|p~(%YZt0DORSIyNaU;RN=@uKH=Zg?>tOSJtZcp8~uxkV$di>oEBx6V$G_LUxxFcgSiI#Y9VF}o&dJhPw#~dl{x|cOIHMpIjX!!m?Mg_IsJ)ec_>^8i88&gdKSR(x-7vmSFPzQ;1>6SWQ-Q{!Z-(e3km#iwTjZICj_tXt3Nd>sW`y&q`jCd;$%wmsgC<K*<-?TmqD$)J_mOIt>UCPgH5Y)%0jfDRX0}P{MhLn^o?l1I5Wf8jB1I5VoQNA(7cBIniBzR(NbgS1n~9qfDQ;^({%0`qJ9Lu@|}-%=@W~bkn)TI95AG|S?B0k==2KWDe2Mj!aEC-wdU(Kv+~nAn>tVF$mJd2<CWxID-r>wVafAKv5@J|Xfs-d=T(}fJ(K#a2GatRl)AfSS(9~6Yo&8O^vXN}+Lg+qy&!?5cK;;-z|dV0isO&t>1aC$o<92FQSk7=(Ssn^K5@+|k|^b^EC$ZEM$R3B+~w8KIW`Azu#5|kZ60Ve$Z~K+J>Z0>Sb7y+Bw0vQqaHqeF9iPa-bts3tipMO?rPw%yUMdw;hx|N9z2&HrZcUcJI^B@rx{LQG`Is=NL}44BH}zR&SNIWI7$~FC!`QIi3K1csRBz~UJY!J8&n*17&_B<QWDpMG2s1xIO9>CXSwGd2#FkJOE}s1M>q=|ML`SlBuq0LDKIjdI|ucBdi-%7C8KTAY5QdN<3%3LEC4x~sYS_5T4#HrfxQAxmQWp)v<vkie5^46iewEmnIFsyHQhx2=#T;H81#gF+F(rw?0+5PpsonJ`A)+j284w=Sy;@VM#y%kvnqNyiB_fKeJ{G;?HD>o7pq8pt#-+pyJy}M!X8hX{4TNxE?8ta?4jg>d3)$2;TU{1@&*bp*`UPLWb*sdC|c1MOJypi;%<7|{oUR0n70_>Y2@&e5d6BO&k~uLR7q>7YpqGT`BO+wK%C`iMP0<xHS-&64bjm8hXgZF?vOW77K1SsTxb9O&{vEXCZyDe<t};E{v<4-S(Z$_W_Phfs1G45B)Vdq!(T-bFI|{&HFHOBw7}A6HLF&JNfeSXI)_#0<ylq^oh9U$I6+YXZrO`ynVm(>`67c53Lnj4AP%o!g*B&TvCEcbF?3Sc-MIrJds;Z`oh+j%3nr9NAwC|Xh?!cON{yPlG~pEINj!%sz0Q>pWtfxbK|H|Zh#`qI*4o2&k3VL&Bblsnk|g<M@a3jS5*7u2i>Kfdd(+D@FYK+QsaT><hoo_N`cs-)bR18^)^u$aYT-%P#+qV^F8eG@FT654jZ)qmo|(X#CeKysZ)X}800*Ch=PKf|*%YozFzXITJ&Vb}YpI$||KOn{umq+%YC6Umpr-3dxsyj&6)XxYklb=Mj>0i;%4R!?*=rQEHc^{BmzW?V*(bNCMk4cLU)E2+Aw~FE8A7}Rb=ewR)$Kg)(iBuBAmn2HFj_k0;?2LYQG4{}-<QrJzJC3;>#EW8pwItAbOIp??z{+>&gtU%)h{#Ed2lN(T-9ize3m*>^{9(FgLLc2+ml=EY(R&h?46L8{@jL$d>kZX>XH+bE8PUZgNAOoEbIm%PhlynI=(5boT)c*w0u#US4W8SyXhGhtnA6$V<XW#@s$zoxofgO+$@2Vas<jIp6=bCoJ~D+A?h99ntV44*=;~sj9x9N!R;r5)>KvJt}WZRr|&pVqIo#Ea1K5@tkf(d(8=TZqTB&C^AtK>bPD!T9aiIuR+l1M=aYzbCh`^R7qt%h3?RENUrS_fm%?m_66P==q28k$)JI$ZdZ&PlVWagi1S=@cY3I2cyu)uGdTb;Ji;1H{>{>7%?Q`rhPS=s?&F-<qm0smA+aU=@H8gd)lJm{F%SMfvK!~-^lyzxUf(7xggdQvl{ONps{rYE!NmP1i-)%7UDvv>;Y7wB`S)8ql27j=>Wh2Yun%J_F#sF;zWUIq<Tkrxo{zv>ntF$_D93u<V<lbH9qmj5zX7qrXJ`fR360gP6)QS9{xL8Bqkz=U4PS6%6t3~LsX+9xZZtuAN`;&j#mLLK(&~Q;u^o?4qsq47D;~d2$PC8O-FnJ)kEna3WCbCrKitii|6VKNnz?1edTwXMw!>pgiBn;%WhVwjJEsFb2Tq;uv5z2@JMaz)TmPmiTh;j){Eh6nA`7PZ^TTrtr4I6pxsho0r2w%VY)lz@^Wx8<2Z@$j>=i>VHPtl3~{+;uOpI*QI`*gvv$Upve?Y#M)*0JB--^%c|{irrSvD+`~r=lk-mm9kUM7qAceA@r2DJ#{1ea*<bl!w+z_{6E5R9cC2kicREbs`gtLGQuSacqm~4Gbjfyr`fb@7y`LYJ-(U^_>bEZG{=w3eI_;74SDO7~XLX7m$gmQ`x7^a$Uggo=n!O3vf3K?{f;O=F2!G??AKukl7)u<m^1mQ%V$>9{x8sy>73Py_U{BG8Un8_U0E(K~mzZpnLyy>CCTR|1yRwILtW^onOSxS$zHK_oWh>Wk^Hc{04G`hF8BTZPo3zcdwt*_z|f}j{G?vId}H<_HJ(plG6~>orbsETAT0<noE@I$1FmSNBSa>W%{Ng%l!CkPJbk@Oy4-vY^|bmX6@FpliJ}SDvwq?QE?`9WVBc(Wfg^QEdfM<g4h&`*EXh(aJFm)ZDE)QG9_s^N5d%^&d_jzhVwI&-{E8q=V@w*nOe?<lQf*8*>9X0oDg*=Z%VOdcw$(|d9h2P0O^XBg&aCKBlvvCmF<i}g`-?Xn#6H$9~-r`Fs2q>R~3U=5=R@E#JsUp*DhK$=dd(=1ExioK0r#e>)Ds)RP<zD@j)dPn43*+y|o>pP=!)3QZX2w=HaCD#@Y134jtE`x5T$VOGV>N5T3>$;6ceRxGLpr=R?WI<OqSfN(Je_+vk!bd&s?8akrtsjyT-;oZ{ViUpx+ZOc*dE{+7J+ceRv^k`z_Mj%bN=aoe-0x{xs$od=>=C|%kS@Wb*x3~+uCo-XB0O%D#w=K;){B4)j$64nyFT14R#v}#A;AyO*)N9(ESb3I0f#v8UPDNNmEK>T*6X1q$=Q!R$VQr#H>?t2=iwM>Q`BiD%rohQUsASAIqaid`DR?%T;bJ<KaG4t6vNuE+RPBS18I%<wwAut2-Z60epD9R|S$@Uc{oRMbG(_l;)a6J(Peb!^>kp-TF#3|LZ=D`CnYX^I*S5x+G8^csH53;yGZ4m;~wcZpc({W6SN*0~U9^WM`9-KxORd8@<2MOa8=P@3cY?&392MNlD^_l}Me>^N`>J@ISg`AlS_chy?>|QI;0}%+9<9NP?_Z5gJ*5Fk0-2zw<uCi*lFe9ZI!*~b#2fEui5%DY9f~2FPJtZ5gi9c?YZPW}Tm9bDWsEk*`3h7yjnkG=SLKhWvTsz!UL=AX+?u$EQvjWMh9mKA%g6vt*>;`hoS{k`3^4t>-Qq$J^SSG&eW?Gsq<bwhVuav+>T(HNclnlpTtm+qS0INvRZa9UUmt=~}t9X0IdELsZ>7-o6;{#>Abod}WTV;)|@<;0w7;yriVY#;VjQMa0VGZ!V=0?@T^0<cS$)IoJ)OxE@j)Lnw@P8wzm{6zO`aAKCcaUN;IWQW~jpj0$v<6>Tn#lmEZA2Anim1|!4~AxCRV51gkhbf&j1;NEddtLUE1iHet)&mFsoYu)+vhy7Z<545pBWzIE_bWMs^0U#eG`iK^ysrE4%!j~B);u@^5FZ0djrz&JV4tz(%K>iwaD76*M{vk8J&8m5&%Y%P5Bn@AEO&&-vhg&4hU&WkG=H4+wi{DxEg0(Xy9U{0N!TGek+$N5e20l8}_!Utu8SHuvbprah|Jq6IPf`BXY(SUpkAsDk6WMm5+v6Fmj;;ln`=V+}mBer^9M0e^;C13{;2b5h%;5e$33|oS}+|OJI71r29?h&dDzPrIzhZCrx7}G2uG)lAXDfRIRr+Y-AT3Fer&W=;AKP>fNdwv4fq|EPpyJ+)lk`5u@wOy;*}PX4b^es;biI{T$w{-X%v*OsZgZD|_{D!;yST?JkP@CiPy!s87$|ic(tVYhB+>pZ{NbqU(UEp)?D>B&)A71E|3z!+29Q4s{xGG;MH8Q%<{j7#S23aB{fOgd;T%F3(xWXtw<>9&eUG(g0N?)S>SAu`H6SVJt&ubMG)M&dG!7Gm37#aLSG6Q@5L&JN12s&hNL580BM)?q<tqaZkq_9*IKnDEidZ&sYA9f^JB6a+6YB-0O;A-@yI1m4IIB33T2k!NXSBdl*lZ<EA1fHICB;&#-QctW!!E!TpPPu-Q_py=9;D@_{{wtF@?poD6qY2f~kF>XXw(W`Ag`C}aKW(l)j(?o)(L{PSJ%xtM$MEOAJU(Fq&TB?xgJI6$q-hyn?EQjp+S$F*E`6{~<x<z`-;keFF+F2_Qk8$jUWF-s}e=TLZ5c9z@5)L=(fg;yw~6|@5EQsgRy4#(A0Cp1M}mv_ylk7lI}G#08-&dkX1@Avm^;Wk%3=k%g-pQ-d_e9ZJZVUrmVCN_Aek*LP45^XXRsK{Llwe=k%YlMO(e6xhWKylbOY$h9z7*+(<R8d?}a9rnmZ+<gbfFu6++w0dqpK!k6KWNuKyMFaw6FXn`1kYjn{qgHmm5Gfr6ba?bt$<W86ML7v1@Hz9iEdGI;YoDO9zq3m3co8uXltqsn`QNa=UGB{jf<%mL2JHa9nL!8?WzvWYDYS{j_kCVipy2q3>v0WAR~*$P#z~W4{S_w^l9A1AiQ+VUy>E*P6%8nWQ1Ec!mD`2FsUhuW+u_}QowE#Q<AmF9BwV}O{cD>X*3j37%9G_*+%qXDcF(uMrRwcOz=AbWD_$oxkozU-O~9zk>Qbw3lFOfTB(mJ?QWiF-8&)|c*7(%pYgV$-qdQ_FtQ%@9x0hAa1^Pg0htquoFp$8l^3JB(%Jx@RQ9OSmbG<Pr{c)|(S~NRCfAu$rif9nc@}t{BFk`NuRiF^EY>JXD=NqvdN@0*#})r|Orkkc`j2XzH3ppZQt)w=%A-zp8lRt2@im_>*5T5>sCbDO$Oe${a{hJz1^=K}gm>TnV5BY&`tsA`vi|9MxhgE$K$c)qknSlaaXjMEiGj}&6j2%RB6>NE=P=g1!A3r1TT9c=#LrtDs_juN=86g1w<ICZ=19YN@~4?my?pmweY<Yxy!)<ny4o^+9CbT3ckWJ1Wjn`YT8{_193-~y_O$N>RQFDE7+}^NQ_6TETkj}{qdR`%{GdEdT+C2gP*(HHCBPH*7aDD3VEFIf4i_FGWe#I9=x7cJpOj$t5X}HZ=53Zv3k$ItEzAwX8Zm6lsvd;g?NHNW40foh%TNn|`;i1bnn$@`zsRCz(^nwn#=%{OE9Wn~HZjkbd}ma)wZ{^2#*vhFTY!dUCF&mKiS3_nT7?3EVsjqYMQXh&Yc+Z{`(`FHqQ0EDX>zjOjm=(RqAILuhb*uo0ob>-JvTtBjUH!NvI&`P0EP@S+;{u)*30k|QOpQ23{*Qz^6)&NY##_pkus87>`XS!$|A@(@Kubg(k8i>jbvvqnayIyTw-R&5uQyU4|T(qy1PLY_PE89d{Up*SHG=|Ul&mkjH!qL=RbB7EQuPXl#~qJklm^-E@3G_KwBOJIC_&=!*_>6M%5g?>BB2z;T*|2jdTx<V3;SBbdf&u>I6~A)@Cp!nfbhUtbnW*Ng!9W@DHWkvstCO8O-c;F``9w0G`X`Jm|C$UmvH{44mCsDxRkasHe}DYwA$KGrnS?DNk3!_zsR&htHOLy$P;5VnJl&%l(q8HhNrud`|J<k<281Ik-6(qvVil*7p2C&%h8;(RbCHY;F>Gf0+RqB_TZWPnY3KkFH`jX5W#%^g)$eHD*!UqIgA%Mr+$@V}MQ(=@G52zC3Qsqc<X>KogDOssRe9En~xhBvRn4p;l(9VG3)D+^A7UH6c&0kK_7%G>ICr|EBjB^bckz_2j8ZSHQ%DLN61=Vq6oe`kFgzjeE2)NnM-LVadBo6Y{QVs#`nMim2;_wF*6XmhX(J4sBo^0e;qr8vO(MYjk6Q)F{Pk6+Cu%t@l%){FN#WskkPB%ceHVk)z~3ZD5q=T!M}lLpY*Z{YL1F5iAm96g)z|u@6)%$fDL<N4@kpgr<YWU93~G_6m4%_yrywhcK5TT8$z;6P~$7F%NAW$SySw2*sIOuQkq9pgf+TgN8`x!72=s&Z#@fSig##Z%cKyK<o<Ha<A_Qr}v}?nk($u#EkvEuQ)I_b3cRnpgKwT%?$X8JW6E1u{Q-@L@1-8C+ezXe6I`FUXE;O8;d*tghjPE^DS`}R;li<cW#0j-Lp-#(zM2%*Fb3^#klm1RD?PYjoEfvQuYPX@zRx4VjGskE1Buf1L;v@Rqnr)`1cXHwPpn9B6ycbQ4OsLcb&U~%8YFc>mt9*Psd>Ev>>I)>X@%y;=rE9$Qi)sVDGLv3IvtrTyJ-aPsazTJVCSPNPp>+-8vH+B5zg;6dj28j-=iVW9`Vb2+m?1Ac6K6R7u4se@MY3O(|c2=qc||oxmXFnhGwYTGxAT??b$X<?dhdG4}r69x+B|=Z~BZ_V&7Yk>5Q)c@jZDES48!TkkxWl~NSl!#*TO)mehdGFVf!ml?Ejrc%pUS@Hdu$R_O0cM_?&*wDdpmlScW**?ACbycZdL26y-eg@Tx)tsvuQX7#yzfV5v9-tnA(R|3MV@w}-`e~o)>e|Mk-Xui9^{X&M9X2YDWN~$~TN*hr^+MVn=@#-=@Ob~5%7eH=fY1fNPB*@s4EwP<QWD;yGHc3i!+4LAVlO8Ynfh~}b$IxsSxNkMcxeE)LF{H|2U(g*yz^E-XLO$=={W9cZU@=0Jst-0>xR+F+p%P>caUDzc}~|ZI-IT{y^5>ymG&FcTqzGjNr>%8o0pl;6qPrRz{EIGPJIFOkWi(p_U_?ztjW7ZkTc~g6vYaJ4aCRQhfzlez&?Q_*^TOmeB|ogQW<b6Fo3`ZX>_i<8jexWBlR0)GXY&{yBOUwmsapbfM>7;)LFSJk-Piw;wO0FO@WGLlPZ<m0L-Cia=L;hK-dA1(zBRLoH!D)#${jJw1F3@-!mXf$9DCEGr5x(0>-;G2^3jN%TDw-s-vDZqjY)@LUS9VTDo6<K#ri%oF<&4>MuF#*D9T-A|(zhqe-(Jg{T4wI(tFD--7|aPOPLi?@`gdVvMbhR{_|p!PVh*0~e2LKH#VN2R*4V5IyaPp<G$Jqr)4t4ZHRFa6Gj_8l;@75mj`8Hl-cju)1>WQq%8tu|_(5b`|$1rn*nkgN>qXL)_UBD=tGd3n4%>AlBB~)gk*9Mni*+^p<QEdymIOb#Y#au&8f&<xFwxLJc5lC>A;S1S(}%U2c+L3_!uPUn^u0yV^MXm~CImz+<YbIJsiVZ`<`kHL^ELga}e+oQZd!b^TKYR3-mI$8Mn|N|l#q<D`|Era@nXpl45w=5QWA-2ME;6Gs<y`R%ep8ym4mTrNA*cV^(Entj*(`YyQeKqaVMD4&CF^p;7#a&SG*-XR2!!*gaFEE=a|NS8KB%D;D;^lA@OrU<`O!CVDPDT@<39HhIeZjxNCa!)o0zGI_dY_bz>+zHIR%ViLxsCqX<EzBG!`_)&4@7q=6YWKD;Cf`%2i$Z>N)=W9JMm4%i1ZKu`1o5?FXo)=HB3ioRDvYzMg35T`NKxqv?7F#YTlkH;`*K%by^N~1;kAw$U|V@=5aw*|5b)r#vGaP+peg&dLJLs{8&4EE3W1?P*?Jq9XeJ%EkeN3fuKSiUQ{RsFRlgHEi>Dq|Y$rMvJAj--(MWA*4LG!v-C(OT=%R8X?6<sSszev&&>BUJ3rN^wa}sA-5}9Taw0N#sJ=bJZU^`LlMlEe%*-C@<<-)p-Z;_*>k7ZgNR8S&gmZa9G^{QH<Z6&q>K~<B^-@Q$QqG5D4?F!4h?Ak8H1PbfE_&){inmD#}wbAL{s_FEuFc}Ym*#vt#L^@3;;&jOPhDb9Z>Cj=LL(R%qrBai;spr&Wa$^paj&sTh)VV}U_@XRV0j|mOYa8tzU1_Gl)(w8it?dE5N3Bh{wQjDY1aaj%4aFyd;cpn;IGY=Lp_2GLXLmbH*xv6kwC(9urz@%XPN#obXj3Ev=Jw!E)J#e%8A2&vZzC}6?A0AK(f!I!>hg0a;o56F3d}!)?Rb-Q*fK69<H*%#S~JN;j$Ebg>{r9e6(oAc7Zn7vZAh2uv3r4hXIW94Fm%lqkuQAt60galt8M0qZkQ$EoPNu-%`QdfQi+$;(Gcin?N`-StF%IMyZp);U)4@^Pui@o!q9rn6un=dv)IReV2YKkHeFZOJ#V}AszTbTp0WXURNFNcstcsLr|dJThrPGahq~l*Si9pqtY2&%+SipkXXa2Ju?hVRsHhM>Fd^Pf4}Y*KQ607pgvwkW+Sc~4W;X+H60Q*A4OC5#{XWu+e1gl<eOs5ES}z_Hp1-mO&<kIM5S&)w47#le*`6OFw0*pf*!W5JPzcwCfjbmbuZ^^d{?F@<FG+XeHSJQVNGas0J7f?tZoHn&*!YSw|B&C7@c0?thIY<raUGCn708*k;>@Rf!ryR%CxeGV7JAVRiU#GaFy{roE|pFVQgu@5ZtDpp^C$KMaxzNn*=s)_NM40`r)Au?a3GFl3%*2N4(NNP#7|0OcOWSGIy#JnWtdCYM;{r8s&J{MDCmKDi6WmB6huVnVt#BE#ukp&npW{5;ostWH=I!9)rQkhj#4Ray4?s+hD8(!VtuauQw}u@>o7WiSPHxii6X*)W-IxV!Obx1>*_?XwRFJrDq0d#S*sY?be8u^P;j?0E^mR9R>@u1wDmS4gc~hfXeC#&Z=Rw{AvY;nhf)QkYG`x;VvKN=q-)~VOGD}l<mQfv)dQZr5oc*Y9GJ<kwl><i)^R7&B~iwq7B=<nP`ctk#*zDuuKj!SFKg$2fAWvcDZkw5%QfELgc618*T0)MC7RSVM60q_0qiR4g{7WS5O8#Nz{BeFaz7Q*KwhTfr0SJ_3En+*_U6BV%mrQ_mEk|I0-qzvu3vo}SFZ;c(i^FMw&Z(9_Z{_?H(UjI^?Q0BZ@S+7{IN5+e*NnZZ<&4b3)~yX5#aP&7(5s9I&-`x_2;pJmsNe8Qd-qHrB_{H(a!bje|8QIf3W-4htHlq;aA2v6MWp4i!C(cuP^ux%CDXD`ql52>IGTKF69?#;oAz4j+bkCEmj+cKl}q=tam3^{ZC_lv(}LRn=!lJKNwm!yb3&)yGgtHHZlQEfao++?>4ES(507s-oBF?b^4QMPrrNc>4VRM7muDldvWA1r>*~lL}ArzZ~Y(A#=md',
 'postprocess_snapshot.py': 'c-p;MOOND85x)1YNScG~!Q<{7Ew36C(kwfxMJy~DX1UCctSmdLE4N&>Co|pMwXK%8aYP7l;<BUB3ULDo390A6VG#eL`6Wc;!%jb{x>^hLrTxf^j913@Ww?%yk3U^53tlopo1Ci?c3D?K5nAR%Fqt)s)HGYuGhUv>gl!fq&sk1*Nz{swk3S+!T~stDnV=U1DJ#VmRkcozkB^V|MoU^zxhnX=-F_jf@@Ofl4aimag4h?ofLk|6t{TOQqobpoEkO;Mr#LOd8PV*Bz@K2M5v7)vpaL?)`DR3-XDD{5LRMx~4yw<CMlB!yAc!I>^g?jOI;k-^r$xi8aC}1Mq?4$hDu&8Yv4vOoG`ZkvmC9zh<d=F_Vy?+|NRX&a9So)=W=*Y^);n)DwG8#b;M!6)f~8buoX<a^g@k9k%vq`Ck0KQLc^TlSfjxsI1gi_0F{`yPb;%2sVibJ;<@}6E1(w+GByvR`efN8z5r~#~<|kFn$}kX%AR<%}sldS-BM1MMl^~0v%GQ_`V4?^Mx>@9OW=HCHgzud`dY?QX_%DjdA_$^RLZ?yE)Hziw)bb)f)b{;~U7D*fYV#wjL@teqRih5F5OvmJ29+hG9LqzKK`4Nnkibd}qU5Zo7Gdxp!I(h;bz1i|))`RJ4NG*8Sm;6o!CM5MpW0YWn7UO=Cb%WQwr~)w=rA9gGr^bJRBvH4AtqVXluBA{^>c7Vx0QOf>erCTn+@iWu~X0EOA_jr$r)2&Age}XEZxw$h8#@wqMYSmBA2-L$#c=LUQofy-h8MLDHImH1+<M!-d~7h1<RD4@U6Ku^F{ac<({cowW$lXXRa4qLAhk)8I0f?_Ed;UguxT-#%%Skzff}d?sY~syj+og{_ghepU6)>Jo)U|C*=IyS32Qf{>2H}nFNCvnF2czByhIVh-Mk9wMJw<C#RlPIuG<M2yitD<EWn**-)STP~$)dvUvx=EBO+WXb@&M)p#nevStIdNA^9{={53lx$TQsvu!V;ddI;-+;CxUHvYAeQ=4NPHgpXD#Kbu)7n7i-;#I>`DnJ#;Ai?V{b>KBxQ%t^$qC2Ac)F9q;SQoc9Z%T5u;>2mFL#!8ihzlh_S5R>GYx^M6MBT{5Dvb)-Ct6Rx^wpxOmJqML|FH^4n09p?=?5rT-?j4;74QNkn(*R)_8rC({cgr2T&5^5j4)94nN$?22mDJV|B_*hwIXJ9U6H(c<;A%xPexEQm}-h_uT1Xn`!e_##GL5mjdRF)_3m%w8T`9_`^VaPWJa#gw(FoP_pcfbNQnu)HQUJL^!(>l=`b7^$TRi@>}yi6Gn#E<vZSDUjz)m-1um+q0k29BAl6JsvS1KKMyiFPyo3@5w1y!BbHHQGX?ATqCAOWx=vy|m1?x&G@Kwg7OmUFS2hY<GISd?^yo3QIAh{M6L*>uZ-98OIxV`x+stq3eGyLySI_q6{^`<7p?c3ix#&I{Vw{Uawnv>1#+pjb7{eS=Z_diVfDj6kgfVJMHEVKnRuwHe844~9h%Dsi%=9D+l7OAP32U!2aH`navRrT&G;HtMbU#r8+HKh)sr?q+!n0v3_p+8(b1X_e_49YZk<8;b4H?o7Cmv=(X|I$3r-t89WT--GM#l1m&;Os}l)%}s(_}3Gl!Jso6ZeRBBpz}jaJXc}aw;ql=h!_=@UH<x+Gt~IXmX-DG%|8@*)j(nI?T^F!=gNHC>K+DtNCiz>o@Ab<t|-Q6DMjln4&YdQ-wl~`ruhq}h*~_LZ|RN-%>9EvhZ#C7-3G)!9pv2QWT^qreD?-UhIFgp^AFrr?}L+bMCSYYIa=OYdH1gQ@rq^Zy5i+nJ0fXg6hBPAoh;k8V1mQ_h@9#J^)LWCIN8AD!JvDXJ_wPm978YM%k~NOIv}S;bxTLl1tk3Rms+8i{0h&y_WGx6Yp<SfYxnSp0#3DPl)2ZZO%Sr5zdc`Z*wPKu!rPm_XNYgV1!EVxk#MiUME53OQA;;?2?unp4{F-z2Qe`aIQMFkp%#GYtkxU^4(h0z&qo|7xN-<OvvS%EY(Z~E?CF01hW7cjW9;6V1M1-pmcCZrwNnlb=b^r`XH2gp(mNgQVlBoItam^)vA0C6DN}E*ai1xTpu#1~Aj}<E`wZfWEhR~T@SAoL6S~|&PXpW#<6F~p5^M=4gB1+Zj;n-?D-(}N$ske9^u2`5g<}|A7h6ys31Hwxvl<2mMwMVp`K&o^Ut9H4S<+grDwWh*EYx;7Ts?dUBjKXer$;?T#PL-FN~&#Y1Pn>zfT>`TK^>p{x6t-_s=Bqzr(cDvDmKMYm4KHmd*~M|lQBl1XN&Q)j%={YzZUWO-t~N{?@X!%F6Zu3#;qB>eTl~O7_3v(Bwuhh2J~GuFv)KA-8S8;6!+?Y%jq@_G3k|-IhwVDcuZ57`3T7d5VlL2Pj<#adfW)uQXlN+BW|!`l_@?nUTmuA72P+8Hc@s!G^Wgfn18f;TUZqRfJ*jMU*d^%2#k{fH${JdPd)@JgDVB#3!R%RO41amTAE%1X@dvXzETLgrp>`iS`=R8^HAfCQ*MhF1|H031vtkyrODxoO007(G{o{eMI6=_s)FM$RbW1Jf7m&kjjPgaYU}CPNt5H0uxOFyoSv1S8qefGfDPyYYy>*%iYzK<6jPC{_&Ec=p92ojCBO}##2Sjd0xjV&aa}9j0nA$R+3O#)#`?Gvn@js9+Kcl&@G(2<e7fy-ch6xDON*u#xw_uhu)smKVWCg=ct6oQu*o?XqMhg{u;V9#lgtjtp3L?M-eY9$YugMOOg=)J3hG0Fb)U~k2Qp)S-qm$%h}_Y*+u#n{8ppeKpY^eM-pT$pd1gM)Z5lY^g2+l|6}dp3WS9cEfI49)5$;n3FYwGxy0paeJXBTD)GFDNU6wFRCD@>l4Vv^ZIXx)TM$~_&0$rO$R2S&F7cYCKbN5xp>9^}cPJe-s$)<GR)Sl7Z=fJ_?+S>tU+GNkpS3QBxD8QoZ@Mu5|4&9UHd!>|P7J1c0Vb(gZ$M|N@8!bcBCGyDM40<=0;c47C{b4t55_3C<goT<-C`=MK*4l3S`P>_7xIIQn3pORC?M;FD?LvV?gA}M_(MTx5uuh8*frg0TI{=Mt{t!K2$QV$99-^QnJ%nkcJ$PU|?0GVnwL5%#n)BWa4|MITPF06v@OBJof%{01n(qTM^5U*94KEFovE>Jw5Ig!43`!DgC9~N6LuR|<zwtTKZc+EMfppkObN7+`qU#Vm)$$-o5U-C+G5oN@cVa(-OZXZs9dHw>stwPOy*$=-!RL1K*;=|EF`RR%516eQW006&6F|2F>~zhdBOelWv$Y9Ju;VavaMw20RI-DOZ{$Yjk9kW%P6!Y|5TZX{UngJkIvD-p!5^D+gVPR5CXU66h8OMG+x(~_+0Zro%)#h@t_#~$un6JWK>Vvpfs4f)*MZcAPcZP40obtZL*2WK`U|zcez4M`o%DJ3e{%I5+}XIL=K}rX5p+V7G$s5jXB}?1rk828_A(5%<|v0I+xezpHXQ4>Y^<}6{sZF4s&o',
 'priority_probe_snapshot.py': 'c-qZ<OOxA3(%7B90;>%NKn=-$Xl6Y_iQ3RgT90D2l0wqVtfQwF0g_-7(;&bBKuJBlefHT$-yGra%?BTR`u5<%I{ZK8pExrsvrq*TK(?fHtYTuC0P2~Qm6h+hbLY;RO_ZkIb~jIx#dsTC#9p*m#AWF%^366Xk}}Vx-n_`)mvP~(<7gQdrI%#h>-gtgTvp}C%ePgMXHn|CiPJbscALqaJ9h@FBHut8*<x1|aaK)MyJ}a&P<6A-i^_YuE}*Sfd7eJ~5HEIBUW~k`46)NRuHvPxU&SA)G?`1iY?r3|cEGO|)zvmG<>}&=WGlaag2Bu0O;oMrcV4RIxA4wr`;}3oh~-zg-c?DepLX+Ykt2Zgn=AdZN;bM$MH5W23)R)GNTH_*K(vgdKX~_3o@68Zv5Z$mbRn}&(|CbE)!=%Z?_kaVBPbW`(rTG3s=?s<r?1~Ud-=kfdH%LY@*=6O#xS~hJbw3$Klt|PqbE=RzFrSJ_~$=@h22*E)bkO1N9$^nj-u@r7Cyqxjy~X<FFrKiZqoZd&yFYGjP4%YrC(qBBeTKr^C-L60p?)KzpTfPUyOcPbCo!=Ykar8C;)}V&$1F$y@<zxMp*X*O5F?w1DK<i#nt<~xD4Z>$cykY$(F&Qh~B3NgZRT@IHmTMaTO&g%&@GA0AEki{C!*mL;9MmyrfLBvWl`rjGspyw=pTW+X=jl34hDUn>Wv&VsG3w{wrcY+RXFIGF2n-Kc?Zwd6vO?z6I>aca@|0g(?tMs>DZi@*1$My9dO@R`XGI70_O0Fs|~#qvycSqD>6DHBWcgx|;gP^Dm&_a#<wVD)&}UC0^oN^~<YrkE*<j{h?zB%dBj`5Jt7hVXr480|1&ch6AJV7inHX)BGKV;=NC*HB17#+QPhJqedrSp<bw3s{1yYgr?%^&~{Yek6EGa)v*@F(I<mV?Tp152zH!Z#354afJySmo9D|bZd=GS?EFjF5mWCSE%4<Ce!x_@%p|GeO&JVts3MhxYS`A<^yK*TOYg3Sf6lFqwaD|!1o{VJwXLXlo(4rcDdVVEtb=-fbN@k5B~=>#ZGAHQaensKhv#=6{#f2s4V2zyS61FU_Tns>0}8>H2S-1iz@LYJSJdNB^%#FZf|OLDp8dyk8~lD6RTZ2@YSn+<B}Kga&F|HhAOD5}OY_}wl}1H8{Qcj4!qG+ECf`66EHSx08Qr^qFTU9qpO;HG`>?gYE!Ob@2-bynmG25UI<4x?5&#ESA31*E>)Br)oZlUG!-Q9%h|@e;0yzE{rDZIL832$1LJ9AI_w~s)r?A_X-rQg2iycm9qG}V?Q=@uD_z%klb3e`188A_O)P-?xTA#ORv@jYo_dgwJDV!p5mf<O*zgET7)atq_Vq9A+G{LGs*#Q=6DptXFu=fCI@bL5e2_B>Y)}4I7Q$6sHj<Cwn5gv8H@GQ%#0JmugoW^uoE{bGZ4QZZ@FV$5V56^*uO|a)}V0YStIYOBSf`C3bJ=Vq5&i8`vqjVS3!7@^_ifNdykP|4H2ac<cx5X|tDT8eJ2~gv}U#!NPq)INLN)cmUpMl;CK%@7Ik85UcTzJvmSJqHynIF(rdT8y24!oiA+qeKk-XMO{ArD-M_bx7y6>}7Qb-AzhF12yb0@08OV*xBwT&}|sw@;o0tGI%V22kGtphKXQnb2O~PoKb{6BTf1tz$mP%tK}il*N0$hS%N}34ZYoXY=jbx34^I9XJkvIB~kzB9wZU<U1s}-0eh$VQy9+(=~0DI=mI}@AfJf0*?pGU7){m1=S3kJrxW^k@l&F=WzwBj}#txmqSJ6)iF+Bze2QF#6^XFqg}Pezf}Rt8OGT=pGiO9#0MH6u&B`n2MVLYe(ce3vDPS6t@hxtk)Z{m37y0YmSvo3fFU_a1#CQ91?w)L@{@pQvfD0UQGy1<oMo6f9(i}~Ui;s{3XdNlf&d#@W6O=*f-Doaz>jtr29J~xXxU{5RJw=)oIO^Anx0J!LR#7Q3QlmODY(Rf4|0sxl%!<T;5J7_McFX#BiQ)RChQ|1!!>`cH%|Nxpnx(*v`mvM1_=%JB~T9Tri4ymZG`6mWDHm7GN4*#J_t4b`G_`GSi(-qs`P4xv}{N_m@BRGva)KZnP&NWXc_sU$tDL1ooC5{7ad0p-!{>Q;1pI*#^ilbGhaZ~^UY+JB}j*Y<B>P{YN)3*$>2l?ql(BUPGRI754|yVsl^8B;|p(c;ys+{>8mm|gn85iyb$0CTpG}+b3Bk91N~U$@3W(Mw7ASyD>@9~WeGHD6D1iE6Xp=T`EGe345dNj>%-jYhL#MdYMw1i>Z$<2a2eq1k#|pxpP*KzalEDXL-XvQvqZ=x4FDE@krzwH>X#A9;vnDeDwBqeJZ4u%Ja*VaBa4#jnUq_UYCs^(t8zv&8hNU_aG4Y{U;Qrqnk>Ir1M9W)NDsjSA*_e>jF3N1qKgbLCRvnpb|dwMCbmF$0ZC$D2ro5Bp2rtS)`diRO=n#03!V%E3&$n2Pv8I)zkov)P-iBXwyPFMu!Y;7ot@(Y%njyEPY$@iAqySPd^j1?aIsn?K4P3uat(bSAYO*d8-^=DnYaKUKSL@`pd5NJ{2zrVmY`jo$s%v^C3LA)0;oQKbU|n8s<BGkrkX*hv_d3jgrrw+_o3GJ+Q&{%j^D-^Dn7cKhqVY{dLWVJAu=wZ;(56bCcCUSLgAhkfa$|Q2Lp5reS8d1O#=F<2ZzPmA{K{8vVl5**!FdU=_B|66-lEjnAHoAxvkGFNKRV|(c+{XP0RK>V^km#A|O=dwmB?090m&d!&ZyT*2)fejDLxX?dZJSUt?9&xesfj`UfTfBq<OkM&9+!(A{Q~ls0$}xN;6?5TDc}pTCu`%}<8yfdX`!RKg_u)9z6?;1Kwtnh2!<WVl{i^d7*tj(`ILTE}7^(Rbeg?{{hwTod|FWm_OCrzQpO_!qsyA|`dZ8FVg*5RUT+==)&Ls&lwK1SWi4wA5M%IPF2L=R)CL97S$=S98Zon_=Q6)3F>iEgA$T$E=><Kh9(5Hvedo%*+k*sV(wZIn!sR;KWRRbZ)VrrA%TRdMY}lk@xbAk|05Mj@8`2trrD~I7`sN3xy+X(@Zp~XVb5akIzS*R%C%DEh@_yD&3XJH4IT(7={PHs@CSt_9=KBxRrpBA?-jQ=8|B|ws^3#S)|qm9}8Y<#;HLP1Z)X07lhDVG!F1OQo0B%X($D`jRO_WeKXUYs5!}#vkILSGA87cgf>2CQ$+myse(Yq7(V$3b_09Al;7I`!;|YPeQrH@g09xMxc;H_DakqtDB5c6B!%vRpZbkYtRgScO#d2oEvVX}r1EJ<96bQ5NqtL3>5yJ-dDL0E*dLJ{hH;)L$51H%`DLs5S{>5bA<))tK}N&L<EV_$WR!utmb{CfAX{!p`JKdUT~*sqoh~yuWc2yWP0Uc0(yJZsr);n~TFtKXE@Yw1sxcBtJp4sdioa2t6q7ZDU_Xvlm3nMy^5D6MfF*EnK>(@{)tmsfSETKP7f1R$A9_dL>dv*W%?T{)O?WNvT_zV$N-0!tCKtclX|*mTqblD3`#-;`;xeFq(0l=nOZOwPwocG!3otFI&5^f3Qad{y?pYaWWF|i*VE^6bW%7YUDtKA0qtmZ`H**cVqwOY{R4Yzv^5yZ@Bk#-8Z%F$9i0M7hpC5R?J3bzC?K!*6erh`H+P(h_3VQ${`lOvd2czNTpqsk6wiY9Ph=@P4^wf`eOT6VR;iE<dPxKJ@+L^F)Jsa(l@TIrmCOOamvfqnt%2-wdzwRA_ym9aN6#sV*|M&0le_tybg9VA_LJS;hOS&ll2rLtO2h`jyH*!`ta6ar|?pX^_bNAOiQesPlsB(CeKH=F|Quv^LILJ>enoHfmLC9$u?|RLc0|zH@+Ss%A;29N((oP$gzxDnEN-$Cqk?NifDJ5Cpq9nlmh!Kg*4ocIcR7^=ZxrKzyK&Mt{j|mO}c#3FIF=do;<}N`#K!k0DAF37FB3+_^mH^!2*#d(;=x0r8@Po~pOz$)O%4AGBNFXjqU4C#d*S}IIcFm|28s@3QFl{S6QKPW$ItRNL6RIg($;zko*A0OhAn@<k)~UK*#6{OoU@h$4!H)XT%iKd#p;^={NxMJqq7(**uZJT>{h80bA?r7Ao!el)z6gZ?1gZnOq)pm!h7E00k8Xo?gJ}ZHiZ9mr4xLa-EVXCL6M_;GIY64i=gBehkOT8j)}_h7*;ckp2M2NFqW=h$Mkdr68Q;WzGFS{;Tko*B`8yzyTiRe69n~CH{&ic1zL|R0@Z!d|W^T6(r)E;For>s&_&e(qeM#w!f-mb@xtaH{9pVt(=4^gX>+Btz<phi05v;eY4eAx!8_F05wj)jNYL|w>uGY*D=|@!CfC(k|Ml519F6^@zpcoI9Y5W45^_5rbl$#J+ok4ST`r!j|2=}$A7acIRaJs?~01I{HvC3K-57ba-1j1t@EGF#Wgs`TUs)+Gk)X!3PhE3r)g%p#E0}%a91lB2-K0yy`_DKn{a2lHyJhcgH_kjR(h&)4XN8D$k3@|Y%(Gm#f*9tf~Je@TdtiC3^XE#xCNxnWfz9Y{U=!N#k)O?m`jjygq4XOGLp><*r3V{KNHfrtQ{n=Tm&;y{ziu?20zl_T{%p-oi)X+-(a3gS4!=BJ|ZF^_*j#QL_%IF!OFj-ZxXb5?}VFfyI!DjxOSwjGnDT@R+bP>qDwSfZ!PLJh%)Jp(zX7PJk5Jf%n;%Z_LZ{996BAI9k6AS{iI5ixAMM(2`;NNAuISgIciK6TqM-ih%2A1nG0d^4~iN>t6b8UE5i)Jie-8GA%HFnD;?Iah226IRJe0x!8Vby?yi!~c-`sOrPPHo0(;{^6+VFVwOiUWhTheCDK8^qutp@}tsk!_p0HIh7ga`fHn=iVkM!997gHgTdkCg#d%3<d{)q{S@O=Ib166O$w6sI6{H34lc9<DufUKi{gLq$ZTLrifZ_Vd)OqX;4)IUFtTFaDp39glZu$Ckin-cp;!Ag}$KWh0RvNzL&*;ZDKA6ky6rk<S^x+9s+7|=02(5fgHxs#_pUg8S2vn9dU!E-Ga<g<{WG1y@xF$hv_|GinxdvCJ>c0?4REX6ziy5AC<{3aTiWD#5NOkTJkbk@GQsFPsg6$dasm27Vzp|dlglTSPj7mIlu4aAo3OOaiW?hNMj(b;G_veIVJZCUUx&xBj;Lh!O|wkKzYkdo6@ME-=`8IBGwLRens+Wk}i>IQc<_OX7@@pY%s*Bvs-jeaGNu!056Mds18*UgbiGv9>&^@RDa)Qv}EI`K(i*ap^mIuH^4hvM$@scg><RznuTodYPYcHM174uO;XUH0p`C3J&=L#p$4|fl6$=kb!fm0DTTDtMJ<8`HZWySs;LGksoO%9s9r>&CZ7o3e)_Uk39XjBsdkpu`&7^EtvI3ox7ZE$KBuZ9FTfqg+r3va+q0kTk4@>^A&E5^^u5fLEsGhBgUUD|f)_WW_9vjN*yI|o!B;v4!^;|dLmXs;Pk?Q{4ama~&7{M@Ki5$4G0GS7q|$n4#p2jJ+`ut2U-_Sf6ut#kOE?43v(Il$_i$y--Mivfra12qm9sNbo*Hzit2qG8?TXf?B_DKK2@T9r^;dVA2zTVdwKnEj8<yplTiS~{H^HqH5w&E~&1;C{NX~vD_vvLxP>sT}kusU!8)jh8Ok=M=r#?kVp(E9sr6o~fpHh;bPoUCK!BK$KU{BVXjz^?CIbwuI9Tb~YL9zo!g(ax~7L~HNW_dPVMHL+EjCdY8V1xLF>cLWLIz*KQZW8Y;NuyRk*M<)AZa^AR3m|02%z)psEzzO@l12@~x7u^6L5%T&lK{hAMCwoxOtlzd4$YEIWNc<xIJ$91HY6Og1PL>(af+(hD0$;M8W%?p)(Xqe&2h=F^q^fL8BJWl1pYI%-~e4+lNT<e6;qYY&Tj!^Uo_lC52k-)-4jO1a2;PqzO)0?kk`Hnsj!K`<m)vUkYkWX+TT}2et}zDqwXto@euVJ$h;WYx&d7>(8=@)RPnnFY!KuNwp^2#6Tw^p0K#UDEJIvlFYcKz1chi7hT!U+u14%%3mXjzE_W$z(loz-6BkI?%ob$P?+;b5`ukAU7zYc$&>t|iXP+Qd7uI??4=FElr`TW&o-QW0gTtNZ(c*(KWO}Y!ob^u2Zc@ikBVUOx80!24iv%Y*E(g6kwks<nj&av<%yG~du^{E)CDrP676a#qOY|+2fawX?H!q=3y}$}Pu4sXc<)cXiq|{vQHro;`oC->)m^6XwYmmqn{i4ry<!e|=H3Rq@^(tS~Q-5m=FsP#Z%1wCpTB!Xd%&jJx9}jCv(G*uZ^ksO1JE?@TC5*~Nk|_6mwArV-a&4GJ>SeU?$_8`2m>h1P_G*2)9e5NeqJt-+vXHkAOIu%~5v-wqu=mLqGw-DRUEq19ew=p{Yl8o*MZQgS8rXv_A&LZS_j{UCr?eQ8pt(J#!m#ZxjxcEpDJDs14dHg%ij2eP@vm3HKg3~{xoK4J%bfTP8WEDE`GHiihqS4@qjf!E?nkoev@^p&P2p*7wq<c0BZ{dkO0xjIfYcR0-3}aV12c@KnmBW-=44NIpbDz<4{Ok1B&OZ$g~G7I8*H^@Hgg+#BTT8ii~n8g(1g>Cc+JMCIBO8|jy<A}9pflcw%6uC)JJOC<9yhrmj{eM#r`92wbybA7E^S}L|3M-M(opmGSMz&<?Nmn&o$_ATMtf;q2iZcO@<QfH9PqlUqSd?2)3jdvVJ3XsMnw?7NrbT%9v8uZku;uP8kcHTna&pR!71GH&2H`qG7Ey^IQ<0_pZs`y+gi^v?FBn3m9xQsQL|TKd9oSQ(&Fbpvn4`lUig!tsR4y6m6$C7#yal;~i0g1FiDJ1U8`XL=3EGDr)?Q<MjmYHe&Po$Ilrl)C4#%MTW2Xp)WEMV*)|L&)p0U1nEH!x_<8rr{`HO);RdOENknOsQDu~$Nq@M&fdw?u~-KG4LZJLNun@`F6={ROr#B2+AlN#6^-@4JMQcD2^Dfc4Li-^i>LuP-Z+*pN$C~{L-}hHPo>fWD}1Rv7%A5+fM9Fbr;JE~ENgYzAJv<Hqg9Z2aO{|-AMyjWSb@Ezv3;=m4BLJKuvgLH9u5IcTcmW0kcUB(%XQ=ux<Z7gD-J?(iwHx<x#e@|zUuO(ZiWI77|*PrJx2#($eMaO>d!YLDv|a1XT((cHcX)#D@G5~WM!-5qN^)KFS}eR9NkI&nW^$R5Xyu)VHKS`)uFx;byn74XMhGiVF*eS?w~1udj(m0$YJEFoG>OpHAov$VT?9^kVoz9A-ZqLZlxf`(-+qnaT~DA8MUeZfF_Ux3$FHUuUFiLM{Rh#3Ws`nbgTAz#x~Le<17lwQHk6$ZsD47u=>>0jr2HW9PC<QyE2H^gghyY`z_rn+-vh#yL9z=RkU3l#q-Nj5BE`P1GaiC_}AH#%b!b14W-JO*-Ej%h}kAqfU6G;u11aE0v!0Zl!T*B`xcP($+7m_iXl(C^D|hm?M%=hrC2hn!voZq&am-w7t|n+&el@rJVhx{$K9_umOuOn{FtaMV|jn;y~N{iK_?c^49xt1FBdRr)G>4IDS2-*%fgETgb)@V0K228+I7M{-ss^?IWi0rT&*EHwv1#oG-+sURvspYR%aB*9cbwYnV>`#pD)+KeqpP@i`$PDvu(JQHRq|B@Ae$^CtKZOBK)MQK!?hO)yTEBNDEbU4D2ooQ36mdsmMaJIG~O^BP}G@U`CGglEhkxSj2zHN{_jIj4*vpQwj7HKTIIsBltN7HO`I{W~ekf^iyL?sZGk~-TAGsn(74WnJF&=?%U-^LSN8>0*HXYL$govA0RxgBP8Iro&bOnSz=NW3PvhvAPk55YSV((44e$Kfl3KW$FjJI8;!oj#2Xw1BAHA!zy~Y&Se?ZA1|UZ}uy(hlRYmtoy?gDJX@Ak?>H!+VsS`{K6iL!X6L!)7XDqQsUzYMME8STaXVE%TZTo1;>$P<>9@LCJhtF9R02WlRM#socZph<X<zDU432>4<mY*#FesDNFku)xc3i7q}^0nd_B;^>V(*i>(3=dEb?N~cFuwL!ve?~e_>{EJ$X(cg&hrJWD<wtBcJS#0R&8_X+<`Ww50=b|lYl#tS|LY2C9r_LIo5W}{r;^P<QK*EP`XpP-lS!6tCRwtcT;%V3HxJo1|Di*$Ixcag(C%q*m%a^Ci;XpFku>OB1`q>yIY%n6qsBWDVeFG^udlc{%6(FTSys7Hsuz6=DEW@Z3Wp992WSA|ThSvMm?Ij=aI7_FSoN$Ef_FhF4#0^gs<d18aYOdk$vXB(6UoZ!L$crWAlbCvuk&9^PWh~E;Gb7s>7h!p#t&ywWO`U!raRx2B&srm6$xr$_m}4O?T@lGGf`m%C}ioH4cMu)%n0Ls=Tvugu4Yiwaqlpj$*c~_RdtIrP4%XeUxq)oL-%%U=-p2CDclZ<R=s5=s1ENqSNyAZ5JK)xl|F}BRccfWB{)2pRBt|;TLuvxVxZz>zfEc?ckHD>z?pD6!Y^f`T0&4qz{iL^asazaWS#?24!YwyoFOBmM^7?RhLhVPYC*FyS^+jgI&VN{e>mJjpyenLk>4T}+I18;xS4+ovOq#}18ZjHB-%KI`5K3|2VL`+V;)&&nPGoT?I)ntq~=VC@l-}bwv3BVLw1a898sqG7`a#Ndk2iEb>o5U!qBmjwnLqJT@C-ZLsqVJ{_E2`S?!GtYCDq|6>j5C#sjvalWx`e?OX!8VwV2Jy#kfnoiX^2^t=<#I2#g8?*ZvmO5^GXDz_I$SEm4Zsbe~=3|(l|@JH!xRuynK4yp?uAU~?)g|c@j7v8g*&4th*$4D1+vMFX~11%J@j0(aq9QNE9@>lRwyShU9*7}kINI;iTK4m51#L;3MvTg`Cuu$A_DbsSA`uzq=1yt`3uV4P~=IQJ3?bA1J!#}@#^3)VSaVRLe;V2x8zRGlpKFKIm6^b1NVTc&{f^s3MIo>cNQGs-@@j&y1)PqWx>kQSBusny#uyv3$uxI}<_esavp1~-|O;U_<`}7-}N|*5mpCspFbMSgeaDj=WxG<OEZgkN-*h1Vpm1&7DxLz(TyQZgDfYF^AmIVJy1A+~J0d&cwk!=Tl7MN`yF^<3|xytDQGVG3SsDV0136;|ca!RF>MQx1A#>-v#CF^q3nI>mpUCqKb45-GXlYvdUb){Sc>XSDb%5G(V_0N3%1W0K7?#;dbr}&D0VG1I5;{BFTin+Nl*|h3G;fH$L$WMxhVHVoNVDMaWxb6#;C2b-Q0Zmv8Y91B6%jv+?VSvFZs?HMu?sQ*}vuYEqP)1AodX&~M%~G_!&z1_y1t|y{^_4_{v{XyLv{#2`$LCn9E~xPfKeg~!p@N{Yys(fN-41ohv+>FKne6rRp@>!rp|qL1!RP@W2WscoK0XfT85}3NB)f5BA8H?TjifVw;EO%%<TI`#v<^sW(LrR}Ae<u}Q@c|v0Af!xb>6WfaA>CGb1~G-G((+3Qp&?<4#1tx(<r<2g^_A7m)Zw%+C}WM=pE*_!@P^{qXP4wx5LRAGpTEcmqIPNI+{E0go)yyG|hsEzb*dpKeDyA++F?p_y1IRm+OE0FVIe=zy8~Qzw-XifBwJ!^Vh%sw}qGe`uG2mKpFTl|Hpq#AY;_8|MtJDiG5d(j(|aMHgJ+WJ<u8jkzWPudZ9Whhd+qN^b+(}Xr_rn<q|_l9<k|bTBfO{wG(;<g*kA7##;(9|GoV@N#H(<;w=>Pra<B)f0EIFr$anBbCqN<9O?JSa5gUgF_L(xTXt-8weJaGkCDb@9u9Q{@tFr*9YQ0=(&}~#yhMk>pVm7_U7|}^l!g55@FnWTPQE&B=dfxl8mnAtYl>Q>hN1Q@8tm(fwML$qp1+yGkAkv9GDde$$jD=4fz=n|skXE^&r$b%JZGdScU!D9*(Fl_hj{*kzWo40861$kxnK7uefoXr?|Gi$7UoEP-q3GNh-D+?ja#`S(~B#$eM~3^=0mFrkr|U#xk^BL7l9xf8Vik5EhfZhxMQ@;Z8TTTsl)0cUB*3jDtD@D98Eg$9{5ynsz$?UFzlO$g&Be1vx%`oxH1E-It<#BYwuj$lT>6;U0MNVzeF-nTKRQb$*njgB+r+v%!+Td&MqlI4@RuBurV#tx>#<*#byb9T+F8Z5sSWs@Mgs&ohg2ZUYzp-{z#VAv7+W?G7#r<ABltc?nSx!9IMaTbAvY@G%|>AW3(!q`~<X-AYuSlR;|Wg`+LH!^<Mtiu<Oql;cGAYXG#T&G3t88TIko|>mBjt(KXYpo5?o2sAtPF!<s|9vj>mW%_Z2R0`*X`A)O3QVSd?&Q2@?Is2Ife8p8;1j~i_d-Fv4Qc2Cp+o(#|&6c1nD^?Bc;z!h-eSLouzzmJGzZ^v!-Cm_@_<juqSdQ>yG*?24aU;C;FX-_e`l;IYC^<=HNzHw%y*_$C#Ex4^@L;KiIN}1ZH=QeX--cee`PI1IYi+QJIUvfqbv$9GcVJqCur=3}18wEu=bM^5_Nf0#;{;!h((MbN^owHB{{5zGBAER5ZFLjcu{JQm8y#i3Dpw}hsw8Ad-CjGK<W;bhpb#Ue|$gC~FOKdo`H@801$-p}_;>9sBHTY1_9R_kmCMWgq(02l445gDj0!O_9p5-cG!$xDpiSxyp&>rcK#G38$bfE(6RnZ~OB0ki`4$8ZR)%y0Y(XAuMGZ?yCRFe3Fd<TNU|0`TI+Ht?C`{Gf!A@z{3O=ECMTBJeMFKxX-)1rZg&E%PT0+NTWZmT;B-N5uE$T4o*SXMzPD?bH8O`k27a&xR86-95=jqvLcWy%q;9u1CzI!TyHEd$pU7Ydgg7<6qITD*JYt%TV(;Ng@W_{^AkCpR?b_R`l8r^gN(Hci1?6PF#y6*U7)L{TYA(!qGwZf1-k^}cEhvt(sLU)?Rs-h3A3koB0=O$`%r7kX+%ltzfWXsnN5ImBJsADEmw)M(#7dKo_X?&V{3U4xc@xuu%?OQ-!U82&SSxOAIAc=B%d8cVIlgK=xm{`Bq37yltVU>atn$KQVUD17|wqyN}FXhCkHtZn=0wAGORq1+3n%%qJ$4xPQ)&aO%oik?Gw7Fxcp_0jPRgA&u!V>7Jxm?{DDOjir0$-IC}BS0LHdx6{Q=Mmm8{v4F*AM_=H9(uQxW;WoK-zAI)^)2Mf=#FBdNO7RI#HTc`<-$OBHO?3lE-fz66oQF!&u>gqvDuzw9ZPY%Hj9Ddskv&rhB8x`@$~o($TFt)q5`o5A*t4fZF4@pR#%5>xB^IYIT$CTY`GktOpYgIJ~@UN2K)|QjgPC}eM$GD)Njm?cd0!u5^xnCpSBy}Pp7T6%n!{bv$DJg4M{Wj2o!$A-?qh<)<<~8uT~2e^9_#VL`uQ)iEiS<<(##9=3w>P<5p{bL2(&fe`BlB5Bhc8_Ft9}TITF>>fK4Z-Bt1dOfFFNo-RwT>goOk?@BGO$f+(uuTER7fvp`_X{)Gub!q@!y=Ve_fw2*0MSNjYqsL~YEx6Hw`&ClpE$p#BjcbVB)Q7Y<VQ9QGy()w>Xv3r!Csk=ml&32;?<aS-)h$qtfoqDMJbxZ=6O9C(N^auo+S(29fJ6XmwZZ$1i>X=GlW@#+VA2Hnu3|crx{6zBO)1f!vqu@>{elayU{ktE2qj6<3EYqkET@lD6r)7(ih|Y$s^U9u^oX`PGdh*(a+`gvXc4<j{Mjb!dd~Mf9><aH*4u&f1Tq3Lzu?LVUM+}l>aKX6<ggjR6vGa&@j^nP$YvVvl4U>^+G9u3hTSr6K?h&@&I{&<z{A(R`z7tU^U>y-ex2mI$SIF1-2wu_;WEV#MC`|vYO|g&mH_CCv&L0Ga7Q?7v}~g;d?70VfP&wSyeCip@a)lx@Q<%wzIgle#S?Vx%`kkSP*HpU@aK=iN6(+nd^q9<wniHzY;XeRgD`;Po+Pc7I_%BmNi8lc5Shh^5HUC`;E8<U8LgBM?*vIKjFPXOy?RRTK?*b9zj^!Q<#%rnzwHs(XVL$m4i;8q=8Ia5Bs@|K!O6Rb{ll9DsLl*{kDAv<w%1QZ$?@UM?9nntA_C*Wo5(@$4DtY_sP!YiJzR=`gF<d3gb#{I2z=Q%Z9orkV?lxj^crK26$N>gM)<RH4{&)!-%y!-_-3$n0FrFNqDbdOgvbc5b4=h~0AlC#yn&J?I-U{9^OepC3CH;7pHW*02~ZkDgW0z>*=QUoPQohXxE1T@IpfObZm4MD*4_?hL;x|wp1inW_}u<^=eD@*Ri~Z1sCReVwwKms>fR*xjyqBb14BdRM)Cmxt724;rsT1_pZ-|h4Ssy{#SmK8FtzceD@Sl$yhdS_c!(l0bkEk)gu+FG6Xodxxl};&<#(qO7W@+KfS<c>C2WTwyJJcEcGruX&S$~O8r2Y2aa%e09Ahh#IqFv_b(y?v9JY?l#*ehy$j@>-&hjz89ivPLStB#b4|G>rJs;JYp{6wCkg?9mQ%XIwq$lB&)TNB)J-n5sU(SlIT#-H03(s%J-3mf$569Bcm1R|H%YbCR^$%6vD{lf@dC!Rs57XKYKckGPBX&QjDO*&PJX#ngpcVUe%<Nq=0-qyW`DC4gp$sxkNkXdH4`GHWvbvK{9iqQzJ9>yuhxYgA^#xIFPw;MpWD!f<H|kN1j6F?eb&C^)hA-6@Mr|%P0Vmo}4*39fBBsMZW)kjhe((1eg;d0d!sBK9Zd|T6;~hC@_SY%m4X9tB%6xYL-X|C1ZL-~0M|nStF{AwK<n-&~(KpA(wLe?Ej5UsTwS;BIYRPfiHE$C}??6KrLxdYTzjx~IZyG3A38a6Xf2fCMAym5w^*)q)W7#vrs$>JEc9VgEfZG@7G2qAWw(zZj=u@+Zw$VHx1FnCO1MdfH9zq<AF67v1IVu1h0{&e>-D=+d%zyJVeEjI~w@<@YkKTT({Z|m}*<g-abfUGGEDF{aK{#7{`|Qt8fs0Vo3z#&YBUu$%&jh0>X)pA&MNURr)R_fl<R;#?i{scLPaIyphgd%%2WVlF1d0Yv^Z51CM{l2oFJ6W}JbUrv<qtP4RG4X0P>N0Qw^f61?6!a>GvmJ_S)3XFb$C#Hyhl{a`$fEwl~JhSAiST{8myO)+iMHNx@4X4OGuR99SSC<>i>H#*;p()Rad{GB-=@I(62%z$^+MGP9aC%c(?@LT;rR7#q3zdqXDxfgkX_&><274s&*L&?U$l-XnTwBBn&d&ebXlo9;AIUO@=xx8?L9YpVQ=GT{Ta*yMkz1Y5u+Pr4(H`X__SmNGXYqaO$Ww#|QvzZACRhZD$pn*D+`gEGAzS_W`VrO>>Y=A3t(}*1j#H2V{19NDYnkydNB?*ALJ;>mg5$IHCtf^vZWR?5N9Nk-|5c^ABe0bQQW$6tpJFOp6FK>F1Hmdl&uILh2n?C$?n{j~s2AR)(7cy*56690Ny+YaEnj8g=T|z1|+^@z^(u+YGd|1$&0-sPoy!eGsZBnB7NB28NJ(F6LGVl8r+SP)f0dFr<Mqy^Th0r*A8h5;MoFn>9Am$$S)#Y&)FJPVkK*LG7t4{cAhlD_)5K`wIA!6oNMLl!B^-OVzXt4f}a^-&4$jx+<eYejMd;0-L5~94yys_>;slIz?<F{fkZnYh4-84*!$rkXGW@?pp(l<bP!=uaJ%|ZHh+rbwQnJh59qr8TRAejCFFKV>1_>I^aIb8PUj*HrtcR*j5EuIBGpLkhRTI?WLmP#(4ezg}B^;WphwNdKli0K6ECCk5ydV!TGxlWU!WgPd>u~q8a4h8n8O2dovswS-CV2Jti?%PjFI`9=dF;%VE-RQ9Ud`q!XiVp$4@GrZd3A4!m%HK@^AAm(#{Ix9qFIm)s&6T-K4#<RgT>BkIOqiU~#c)J|iT(_PY_i6|-wpxv5Cx8{<Tx=iDEi#fLfPGE8hETWFF>AO_Dv1a9r!J+}E;?`zJ*!WX#d;b*o(_5&`Zh!yVt+F~g>b4R)I?A{g$qcOSP36&IS=`z9+EnTP0fNpxL;',
 'pubmed_efetch_snapshot.py': 'c-pl*O>g8zcJ~4NAIf6{BsDEc8h^m6aXXM_<lR9$AJE9VYxFh<c6W6*EjHOE+0w{DAA%fm&M7}&54i?NF0l_eILQB)f0Fl6RjlGitH<kju_KABdiCnPSMR&{@ZrNZ@`sJgsw7Uu>&@haoQd<VWi?%j-=ANKo1|Kb*Dszw6W`}aR*E>AiP@%zC#e*Z7)MQ)a(cCfmsJ@)eE9H#2gz!k7nPWn*B?BX7x_xe;!0M@N(%Kx|2+`+?~i#V)j+w5lQi5^Nm}Z$>$oT-PKc^J(i6=ZiyxD<`6J2Auiuw>X8u^k)zbWtmujswJWZ2{UgxzvsHj)q<!HvHNMS-q80l$0Y;r(too>njzAx8lQho5i0tBQjtI~S(c9n**Dx?h0Q@H}VU&8MqF2(sJj+)6i4v8xHw(_%lCWF%t9tijkNc*~wr7W(cSS4?POG%a{nG`sZ*pzQ75zl5xwh-kyo=Tu$RwgqkZt`MQJ^^H^ILX8$&8Jsll5etETqII44VOsf0#_7CDP96o)WY~5Ox+Zj@H~D*AWAV5ci!3?2oFdS<Ih=gjX%<a|1GP-q>?NAV^Vlqo{Jm2iPI}z^7aF3nROx&?V%W{mmzH~FJR?yJ!}pSrpqKn5(qWYf?_ZPKr}3@;zd{xZ+veH=$r(j!;^8C<~OqN1EGmhx-fto^tb`KMl_7qYnjb_?=RV3GB0q4xkG5jgiH|p|E;2To1{#$po4=Uo@1ZqsrULFEx_+Rp$EoA6*B39fGCeokYuzal2G;DL-cSzJf|v>wU4`Hp$0}4@wAExIR$a_`G;mcOyUQ~Dlr@i4>ZwA&d!P|nWoaSh%-&9xJvR2n5(AD6QmC4Ej$iS)oXk2F+)LdP;b>74W6(zfI<ATyrLrj(gkPzC21%_@>Md6=$y3Rp`Ls{U{zA3QbSXCy2xGP2nMS6aSzORQWB}@AdMRn>q#R&gM)dd27Aoh3^{A#5vQqd4YE#Q&n_EsY>){-08mD0)mPP07F|m_1f{hbLls+b2i8ine7Zp`&=iWV^8D(VdS^?7%|ku6#X|%1eR$Y_)@I_qP}^LzXYRvB2bG<C?8519<70d6ZayL-V1vfu8O8*bNkTgs&NpegLWjdIywS<w@5ZBJ_~!t$InJ@IND(JMmp{hoMxGZ%UVs?9`{}QlSS%Cq=NI3IYMH$I>AyGZ)hyrr{MWTe-~Ig0T0V%KHcmBv)re9~QAWY!qsCzd4dDO$7HOcVH2^)FED}Cdnt~RpX`C%K@d7-dE1(mq05vFFNKmNXsMlygy<pIIdwIK7Z)$*xBHyf|8ZFmovURUfX-mc>Y}gB$#x$SZ+61vkQEy}-Qy7D<H16<((bKlToiYVyZjl$a&a9W|>Zy9ZYwj!olM~-YdRhJnJM;}yR8RW%55b^(`||9~o9DlOdH&4fgX1-Asb;ITrgt7Q&pTCIM5jh`<kaGwS0N`(iHC!>8F&8k2vwR6KVfc`t|E;nJl4M=ROH1eQGeUvG{EC7pP`WR<Si;9n?&IK5E;EwIHZpRK=tU{J7iQ;X~VRHO4CLy^N7$09A#pSK5b#GjsseFS0Qw4#aWPt8_D%xX@Bks|4y@E8_<^YpN7Y@-6pFUw9;f2!fKdag~T0bX?sTYxQ^H49ocTuh}6fxg*n!4k^;1h%SaDEEs)*%N*YHj24jM$o*`H*tXd_3+1%if_p&^yiKNu01D*)ud#j`@AykM6Ju{M(AJZ<LPUX6iGtYGs!RH5s3nGL+P)rHHsZjg$%tkee@UGUFTB`9!<E(hR2^>zpz+fA3cy9j*ZzXDXlx2)9*vq8gERu7KUx635X&wWPDh!je2yreguXEyk@!|Q)%jcKhc}P_?Biw+{L1eJgsGupC{_-D!r;DrQFaHUV*Z=&>|Nc9elqnjPe-%{$zZ?4HIWqVBGQ@Ba4`jB`IBK^BL6hdLm91q}5kd$d6$LH8FbZrGu5_l(FD_nP=uinpFua}HP_aJEM703N<%xFwd3sRWVnP62^beaZ`Wxw-;nq5Hq8E|CQ0kYTvc;*m(?GYipec3%Axf|cP4?+g=;(oz%r+2ypx)%yaMZGV6{iWAz`N~$U<9s#R#BrS>Zzl_Oz;UC#Y2&x(7OT2KyM#CO#N`%^qDjswM9V_ffBpP<l8kI3fL2JTCIp+VNn=hq0rR*(;RX&*~WH2i>4*Oup2hth`DscU5}M-@rYY|!bAi-+<zlUb`5W4Y6;?iXB|zDav$IY_isXZf{6KuHH_<{(tE`XBM`re;2K9ujopu)#(-8Q!E3DDkirl$G33n#d!txGh}X$|ST+J6>p+Zf=va}mmUxYQ15YQ(<Ie9>^q$wF<<Se)`W*)C@PwL*{p%REGq5Z`Coy&bG`4^|`~n<)k%6E8^FMCH#CZSHU;Yae5BmL2#Pr=y|CWKv_U`9@TET=ERMt(XlT^yo85Lu=hIZzI2Y)=jc=P<#OEkA@(gKHUaSpFfz<oM@`Rw(p=Pxhu^|Gqg<>}Fpq@-q;O(#hR9$%Ox%W#ojAHi`f<d5q05km>*(?tSso}GVv_U$*9Z-^C?YmLg}N94fC@i7n!(t202Kx&=9UyFE+yvVZ|J_$cX9#tR#>k9w=xM^!KuL%i5<l4z`I1ry5hseFjX0||0E>1o_4uKdi&i))-oWFVf>gAjB=&SE8&k>uG&%^=X@c0w;gNcIXmvjL#4e*#J3!i^zJIa6Ib`B2o<_6;6fS+%Tl|gq}O&!@A9|t->g=Dj!c%~93;)|i^8jR0@Q2Q7^Omx>%#cESlVggB7G`ir9gWm}U=OhdPT<8VecxrC6kZcV=<9V_VTANa$HijdT-U<x(8ALA%hTpw7)FZ?f`{AHx)I<^MyiBU(T594Ld{E6?NeEq5NtUcOD{_B*M=}Q>nP7^2Eokuf@kuat?Jpq?&M9bvY-+X#kU>IySacxf;4@W_k5n>%8U8RSA({o9XO9o8&Ewy&YlT}=ceeeN=WN+@G2{;BZ^JrCK*vYaqNfj3?R=#&^VQP;dHxuYLD#;qzw5nrbxqH&QL2xL($`2{4Sye2YMe7EXQilaa>1Z$hGv`91Vl7!c>KJ@`jk_W27TBFN*+G^UoX$a8X$@;h0sfWN4Xoy&pD}_Bi~Bq1V?|5`5b~0Vu7X>boFqVmsJ)+Dd|I_ds|A?vj{j{?xc*WcYS;D4aOEQi8UCRh$a;XT1dg7e-hNxD4;wwt^8{Qs|le3L-c<5d3`_#d*B@8I-Y*@JUV;*Jo<<8@6e>u-{&vRo`2(HVBqKg1^_LGh&8)T3doLPt^@%!vOhp49^H^`g%17*3ke}VNF5IZ03+UjoQ)ydqTkkaEj2J`jC4qYfJdh%pB^8NZMuM88pU%6SM>335Xo^{p|-V25lcD+4%r+gy!Fvs@o0*I++(6&PzZY*s<4XPT)$@hKw$Qaezj@c_M{u@`knPig{?R=rOhzQZ+u<dgo@@g2*N6_;?$5LXuILv_zlFXYgqtQZd&MC2;x@pf73KBOQo}(g2!SiW5|*yLLx;G@HCwd#1vefRKfzF6$a{*W3&W`fjk>x64;f)bRL>1^cT(J=i0QJ0a#G=Q8f*W{%|0vWhlhs-$;7Zug%=JE~JKUc!Zr0w84c-8P8CKZPv&N@UFjwlIh{u0;4PN4$rbF(&SGNdY6ZM`VW~{M<?Mip29QAdf-@a(8&b_utyN2dE4fi3-Y32pR@_4hd9!DDN;B`I4s19wAYvqTZ|?Wlme!keMPwt(*Y3<_uWMT&XCrV0Ndj**qLu6lG;EZ=LKb1kUo7tNReszn5KCtY1tOzHGO%AGwUW!FvN^!w@R*N@H?i-Y#=vKnv`UMLOL-4nBoQzT7nJT02x-ZhXx{?Faoj4VOpLgQ@@=Ut>>Ys?8qy!8MtI?=$c3<o5RGT3shZ)XQ;a9Y@)s7W(UPn!S2fw(nFxnQ=o6{2i1ikK($PzT>Id4_$T3@#rH#-B!e3Fz@adwvzWNbPmNdWRF+Qh9K$)xg#&~AS%TSpFwx+(l)h`Q%aI2^&`G46B9*@#pmw1QD*G$0^v@=EC|I`fpn&38LEau8;^}}V6fAlJoe8B}4Mv1{o;J@OAHAn6@;d6WP2_-=mK8hSBi}ZUm|(zG5szHSrX9tA5w`6)Zl%sYw{2+9l|`p*j|>W9$NY|Pnw6a}(m8BNC#2IX{W#KY@7Oju*s1}gq`0h9CF!m~59NbX?g7%<(mxrvP%XFq5i~cPHtQNN)H<Z*c)G*nH5X9tJlOIwCtgBegcL<_ITbptts_f#sD4w4ic)0@%Dz|~Bm4rBFmXj`;o792k;EfFA#-FB0cZlYjr}l)4r(`gBXs(A9d9tZKn@8uE&=s6pyf-H>xoBd6zpxWfftmBXq@7rtIpD+ofK~E@%t#Idz2GvAeJCdOH0@lEY;FFZg^5b9z~M+xwh3y9YHvPWP+Fum>&tNQ49p*3A)S6WHyr-krwl<02F-01o9>jF#;fJKZ3%IQmg_>Gf<mv!=^4X$IjsCtObtG<B)$0^!A48=Y2J|I{B7@7Zn){hbrv^Zn}(DLzI936QH2f*O1YYJuEV{7CkQHCy?}*$?hq1wzN-T3HVS~Ixyh~Ln4glhX$^8X2|eA<spZ4*|*2`leiOlZ3ny?v}Nylp-z{Z><TcY3<>M0<L;5%o&}>hS*jMqvPGn}>3^nEN5N@7?`Wk*LyDsb-VR>}9UUD0rD$iYPIrrkQ46rEj^p2bKs&-~y;I{cK$`4D@)i0Bv&~dm38u~?Ywe_EM7}gC=5!zAI<-dwJ<y-FKG}&9v%YRJ>t}^hxvW>&4dBT^pSHV+%4MwmXKe~*$pW0LMs2KXu5!NT9{i!Qr$=`R=;-wEcx(RJhKnD)p7U%k9as4(nMRY_N|wIm)a%s-)N=|7a|I>vvaD;2E9BbnxbOHl8bAXZ)=(9ik3aovsD9;Vgvbz&fTl*z0(b&gF=49d_;`n;*b=YiV2SIK<Ie};lgGash)*%U{OR#y{O4o*=acOYXzkS5A1*Ini#rCreF6lQqKvQW#$uxs`lPO%cn4IwYrA$yu*F$p=25pd+#MBjo5=Ele>tb;V1fq14#%Nt`x@8R(YbZRg6^W3;rd7wC&r|cDOoi4?kIDMrLi!I(4UN=E!0Bc!L~^hh3Qu@tgexJk0sj+^@K$3W5}m~xxC%6LK?3j6=G7mv_0%9_FMYP_(cQpanK78GdSgveZz56C6IU6p4yJ1?GW=3<!u}f$4*9%m221sSFe3jfbr@I$-z~(<MGobI@_Sf)$#k&1tp~lAhZc{P`AE^jny&(*8=>fp|`2#ho5^SaqPA9`Y8C4G4~N*INPk&rB9?rfaRuu(pou9683_qPo~{pTt_Hc{S$<aQ7V3+*R8Ni9mGyQS~ZnGjtd<>Afl?SVL6ZkQfz{<1Jr$k4U9LmhNX_H5~R#Lz;aiub)EO>(Re3d#V+h+Q6ZdXv=s(M`uB$FTtnMOWwYc?3jk}js4KQ|V<JY>em1_>1vUjzoXlPyKM52y!8hAbj*M&#Db;PKpkSS1Q!d$EZgHBsHMM=Qg4f}&FJ|qKiU~;XRig<po?hkiIf&R^XL<nTmV*P~>*Wt2E)F{4Vn=rv$o11fL!l@%x<LKr>OyDl`tn8M%bl6AKDuHkXReZ9a7Mpjoh(mh`Azl&YJ8N?(%CIE*A&qu&>)uQR{REOmLA{D67a^^E#Vdn=4VQlppcZCbn@Xl$?0lM20n*m5Xfoi!5grxRH~zl>;0kO-QHTJ>&)LDdG;PWfM?txPAcLqu-%R`E@g!J7F+b<N^cOrAkl0=VfX6-&MvUOD`CJ;pJ0)zR2cxFAcqjrLu{_?&vhbiLb$d?m%KWyl1X@0IiGU!;OdG`tQ&!Rx3<6}t;>YuwVMioV+6_ZBR)R7<7p#IERLC>51qDfC!^iIu2eVFtA40<irp<T`Z$hxln@{4zcB?CP#M>^vC>=9Db;=Iu=RyG@%-5l)*hi3ldFr3x*B)zX!5S&E7>A1FVC2&<gg(#cT_c!zYe-8#hgzm@8>l5)O|dLIc__U^)LeUk<1f0Gp#(%x$smU8aL>;)(*ShplfDVM^Tw?im5d1LeX>?Q`gIJ-7$i@cdvu!P>c37<gPN_AArQY>1WhkYqjU<)z$kpRz0@1iyR7uHqunI-n+4>bK65WeLh;~QloBV@p~~2oR&#y4u>E_i_XCT(2M$T)baRX1CC1tiQ;@3pbG#U+j8hJ!>~TDjw&2o>v7i})JWW@%6BygqFARIj&)0NKT#AEEl?nE1_bJ?{m89WQ^+}sn(mVtDK*S(o9o8TPV7C#5ZcM=HS3Acf`%NG)sUTfteZjgw!pK|@b^1cqW}%h^w2reAj+^&(C?&ShIJ0rBppaeIF4`^cq<`cO)#FZs1y0-y|#|f%`qO?nf`I#26VjF0Rk>)HbTNqAdM(=$Pp->lFkqn&q`$~UiN}$h45gh7uqQp2N<(<FU=Z^ord2IkKXn4nBnN|3LFr!OILjSuehT`6>e0;#;VgaLI|v%HBd4=yTNw1Pt%Y2*uAqwX9xy2k31r3Us4!!9#Y$q&QrV;YK|iIsIw*;c`XfNm)7(krOzUS0|Rs<i+=|FqYcJGbo|ga+No@Y&AS(@J_`9gbsc-{h3XVd?Pe$*G!)~7X^5vgYf5Dax|QxUsX_B==ykNP)=mT}p<9;OhL1+iv_nxs;xR%qpJ&SDaQZ>;<^;DxA<{Z@C-+E2@B27H?4MFv4~kI?1eGF`IKf}tP}|2ehZnbWX$>0JZbY%kYA-3i!Nv*Ok8J5SY-uX<^ji9j8->vAo0PCOh;*R!4Z3-J2M~hrtjT+Mc8Cx;x3ZkZYe-rGw)Bt-PPy`z^)X^nTW85A9k(CSVF{7kc0q~i{NTQ(MwjqvdG<k%)&)>H!-ho70)y*E;;i}2hZrSm9VdLxDZxykz(5Vm9RwG}AxNq9U5NVg1VDPc#s%d=6-BDMg_KkW`=`_D$Q}wOI1z7ML21O9bpH$~&|#$ZWf`>oExUYhm4l9&xo2E+pcmvAas0k)rKp9=$M=R!?&TCcvHg_>$Hm(jEvUVbEbWR90v%E`u1B;(^IsLrX+-w9*Fe`<c<T;5jDl*+GS7XphSq^lk6`3bwRDmkxV}c6pJQ#dTLr>bGcF|RDDCW9IvA#;w@)ayf;tO&Qv`X-uoaRZtdzsoEB<f^LsbiEk>6Nf;;<H{Dn3zHKMbtqI(3L{Bi4nwbHo)xwI_=}xO$J^)k}9tA5KK$2R~W&cSeDMJEmb#`Kp?^UD&*AOnZSofC^X@LocTfaNM-*2(-Gg;SP?tYpYI&ow{@i86w(KH%+h?6$lJCv`$kL<u43Ftny&X+%&TlO`BuE*ky9Mb=ZYAk1fN)!J}g>cW7I}tV7-2qjC*nIu!0b2;ZxHsqhdN3+jBHKeX$+tz}iExAh!rz7MT0hjf*AIh)8UHEX-9=Q|3!6V=OpU)6$Sr^}Z&nZ;kztVd9T&KSWicj!I2MUEDc+_WaYHZFamrZF0KFUnk%jJCS<XS9_SPkba^!I4Ay;zeXmp@N~h)m4L7-Dm(s0g2-$cna$*H6Ky0v_^nMz`z00I=5kiJN7QHU4=3pl@qqoUY#h_2}$X-6MzPF@YA5vxS0~~&xW&1?2`rWu(BPw@2;eG%V^*0_1~3jn1Oe-peaG#S|-r4q6DYUdg^BKg8>#*BCvvFJO{?3k61&gr>=>GX2^UINgOu(`1n}sLCX}@rq48<_gp<QexpLeX)Q-9<x_oJy=~r`(v!Y$yDHm?O%Lq9C57N{QKR9Ym_d;tN@ttsw%)}{FP0)0?zs9gHVy&he!JPK7T!AFUb2|t^nUMfl@y#7Q(bv67f{hY;qo~iIKM}hq02qZC%$*UqU;#Kektxr><>gUiq)T;;rd6D^qM2d&RRyGUGMQO;j%lk&1&km(W#oCAWZ3^27VjBy0_tXUC%I5k3z!2bKG^ChW1pcPf&2vFytetw96=%YZ;9jmJJimPE?IJ!8~Qik2)cYn`K)zmAu7}11Pp&?!ezpZ~n%*b1(hbBWLw}a?LN*&)7u)Jn?4S-_<%0v`;;SFEkqN4;`L%eh;SAZomnBEXA0VzGpy2@O7jTt_fY|oT<i7(&%@4_1StQ&~>&+J3+YusN}}@LUs042i4oIZi9Abk`gX+H^ubY5ceTb0YX#uXs_l~N43}5&g}%^Uc>0LP`WnV{w{_6K?<!M{hj6QrgpSXJde4r%Gr(%)x_>Jg}s519l@hhrC#i}iQU*6HE|#A_4gCh*L%pZi}k%2DsZT9rjD|Z86kXe0B!PH{T;*DWN0`_tsPy8FwI9>ZtO2<-4|G!T*dn(b=0)(lnxj0YkHTh6d+z1X?>8ZtaR}nV^s#iJ2ieQX_L|2c1+*H?8&+~wL&Z0AMQ#V4LWLno;vxvn=&W^KD5y_eM-(@7*3#FMQsb-xjBYC*Rl(CHo<sZyJ8Yp^|SU!6Lr~Uh%!3*BOg$(Q6DC3S8WozVq=%u>p!jf20U!!Q}(|8Q0r*7B4WPnq09d5a*^4gsS_FXty@#fH(nqGl-4)VkPMhU;m-%bVdAeA{_VTgh>YF4)!YhEs!L-g6N$HiaT20Hx5hfkzjX2oP9yP`fo{kTFFNZ?<cB-e<8iQq6=p@Z_Vu3Y*^~EMPL-~DS8A2JKD3i_W9JgxWHX;bBm)JjJbXjnuX_H<2RAmQ1_7^Wd2uAP6NS$}gZzme_zYpdGJ@<Xhb%6wx$L^}H5I?-EpCH)cFJI{G!J-sxaJm{oY2!FWbj1N{cYcPlU#N`ER%&f(HbhXK{f0fa;dgU+`emBdwndhS~tzpRJCR3Pe?z-zDrr~AaMHds^PB2VCNfICAV$XVE}%2B42bhMN#81zN5b4{acD#D`8Rlgk*^t577?NJX#x`SfBpg%N193bO5`Gp+ctI@VgodPy1@hNmOUJc3Lsz1W5Iq8~Rj)G6(Jvun&-xkk-K3m0<g5>;T0RC}l?0Q?a-YK^Kq%31rjoUv~x*iY`~wtyeo&hlHv{a2q`MKc@e-r~'}

EMBEDDED_SHA256 = {'audit_snapshot.py': 'ae45721c3bdde5fe22803aba7d7cbfd16221de26aa346540c5df52035186cad6',
 'corpus_runner_snapshot.py': 'af71a44fba63996edd28dc8c3374b8d5a053a1203255e6337f68c54d6e7dcb4f',
 'crawler_snapshot.py': 'e05b4d47a9ba09eff94c32bb61d5c99bf9939f157b5d6aa1fa8bb14ceacb7d15',
 'diagnostics_snapshot.py': 'a3d6feb605be2007f75cf1a99629b44d55404861fe154f5ae218c4543823c166',
 'export_snapshot.py': 'e924cfa0a104928aa7d90826118955bbe49bbdca833994bb42746bf7e9c8ba30',
 'longchau_snapshot.py': 'd17d68f8c4664a33a4a76e94cb233ac66653845bbf6f0b6bfac0c95007728d4b',
 'postprocess_snapshot.py': '377d747c79f82e52d4fd9f16e36512af540d985536aa0a70b86855eff76842b8',
 'priority_probe_snapshot.py': 'cb038c065f04385384e81ac2b2ad69c59e2223e95eaf2674e84e9bef895bb073',
 'pubmed_efetch_snapshot.py': '6d107894f08733407536fc1e99302a7882a1db5af5a505b7d50c93555c0a8bd0'}


RUN_DIR.mkdir(parents=True, exist_ok=True)
POSTPROCESS_FILES = {'audit_snapshot.py', 'export_snapshot.py', 'diagnostics_snapshot.py', 'postprocess_snapshot.py', 'longchau_snapshot.py'}
for name, encoded in EMBEDDED_FILES.items():
    target = RUN_DIR / name
    if name in POSTPROCESS_FILES or (not target.exists() and not POSTPROCESS_ONLY):
        body = zlib.decompress(base64.b85decode(encoded))
        if hashlib.sha256(body).hexdigest() != EMBEDDED_SHA256[name]:
            raise ValueError(f'Snapshot bị hỏng: {name}')
        target.write_bytes(body)
CRAWLER = RUN_DIR / 'crawler_snapshot.py'
CORPUS_RUNNER = RUN_DIR / 'corpus_runner_snapshot.py'
AUDITOR = RUN_DIR / 'audit_snapshot.py'
EXPORTER = RUN_DIR / 'export_snapshot.py'
DIAGNOSTICS = RUN_DIR / 'diagnostics_snapshot.py'
import importlib.util
postprocess_spec = importlib.util.spec_from_file_location('notebook_postprocess', RUN_DIR / 'postprocess_snapshot.py')
POSTPROCESS = importlib.util.module_from_spec(postprocess_spec)
postprocess_spec.loader.exec_module(POSTPROCESS)
diagnostics_archive = WORKDIR / f'{RUN_DIR.name}_diagnostics.zip'
from contextlib import contextmanager
import traceback
LOG_DIR = RUN_DIR / 'logs'
LOG_DIR.mkdir(exist_ok=True)

def refresh_diagnostics(strict=False):
    try:
        result = subprocess.run(
            [sys.executable, '-u', str(DIAGNOSTICS), '--out', str(RUN_DIR), '--archive', str(diagnostics_archive)],
            check=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace',
        )
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(result.stdout)
    except (OSError, subprocess.CalledProcessError) as exc:
        with (LOG_DIR / 'diagnostics.log').open('a', encoding='utf-8') as stream:
            stream.write(str(exc) + '\n' + str(getattr(exc, 'output', '') or '') + '\n')
        if strict:
            raise
        print(f'Tạo ZIP chẩn đoán thất bại: {exc}', flush=True)

def save_checkpoint_archive(failed_step=None, error=None):
    archive = POSTPROCESS.package_run(
        RUN_DIR, WORKDIR, id_from=ID_FROM, id_to=ID_TO, dataset_revision=DATASET_REVISION,
        source_sha256=SOURCE_SHA256, packages=versions, refresh_diagnostics=refresh_diagnostics,
        failed_step=failed_step, error=error)
    print('ZIP checkpoint đầy đủ:', archive, flush=True)
    print('ZIP chẩn đoán:', diagnostics_archive, flush=True)
    return archive

@contextmanager
def diagnostic_step(name):
    try:
        yield
    except BaseException as exc:
        with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
            stream.write(f'Bước: {name}\n' + traceback.format_exc() + '\n')
        try:
            save_checkpoint_archive(failed_step=name, error=f'{type(exc).__name__}: {exc}')
        except Exception as rescue_error:
            print(f'Đóng gói cứu dữ liệu lỗi: {rescue_error}; checkpoint vẫn ở {RUN_DIR}', flush=True)
            with (LOG_DIR / 'notebook_errors.log').open('a', encoding='utf-8') as stream:
                stream.write('Đóng gói cứu dữ liệu: ' + traceback.format_exc() + '\n')
        raise
    else:
        refresh_diagnostics()
with diagnostic_step('snapshot_validation'):
    if (RUN_DIR / 'crawl_config.json').exists():
        saved_config = json.loads((RUN_DIR / 'crawl_config.json').read_text(encoding='utf-8'))
        if hashlib.sha256(CRAWLER.read_bytes()).hexdigest() != saved_config['script_sha256']:
            raise ValueError('Snapshot không khớp hash của run cũ; không được ép resume.')
    if not requirements_path.exists():
        requirements_path.write_text(''.join(f'{k}=={v}\n' for k, v in versions.items()), encoding='utf-8')
print('Đã chuẩn bị snapshot và ghi phiên bản dependencies.')
refresh_diagnostics()


## 4. Tải hoặc đọc danh sách URL

Nguồn được cố định theo revision và kiểm tra SHA-256. Chỉ đọc corpus URL;
không dùng query làm nội dung hay nhãn. File Input có thể ở vị trí khác nhưng
được copy vào đường dẫn ổn định trong `/kaggle/working` để dùng lại khi resume.


In [ ]:
with diagnostic_step('source'):
    import requests

    def sha256_file(path):
        digest = hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        return digest.hexdigest()

    SOURCE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SOURCE_PARQUET:
        supplied = Path(SOURCE_PARQUET)
        if not supplied.is_file():
            raise FileNotFoundError(supplied)
        if sha256_file(supplied) != SOURCE_SHA256:
            raise ValueError(f'Parquet Input không khớp {SOURCE_FILENAME} của revision đã chốt; cache hiện tại được giữ nguyên.')
        if supplied.resolve() != SOURCE_PATH.resolve():
            shutil.copyfile(supplied, SOURCE_PATH)
    elif not SOURCE_PATH.exists():
        temporary = SOURCE_PATH.with_suffix('.download')
        try:
            with requests.get(SOURCE_URL, stream=True, timeout=(15, 90)) as response:
                response.raise_for_status()
                with temporary.open('wb') as stream:
                    for block in response.iter_content(1024 * 1024):
                        if block:
                            stream.write(block)
            if sha256_file(temporary) != SOURCE_SHA256:
                raise ValueError('Hash file tải về không khớp nguồn đã chốt.')
            temporary.replace(SOURCE_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    if sha256_file(SOURCE_PATH) != SOURCE_SHA256:
        raise ValueError('Parquet không khớp nguồn đã chốt. Kiểm tra file Input.')
    print(f'Nguồn hợp lệ: {SOURCE_PATH} ({SOURCE_PATH.stat().st_size:,} byte)')


## 5. Chọn ID và tạo manifest

Nếu khoảng chọn có ID không tồn tại trong nguồn, notebook báo số lượng thiếu,
rồi lấy đúng các hàng thực sự có trong khoảng; không tạo ID hoặc URL thay thế.
Bước chuẩn bị hiện quét toàn file để kiểm tra nguồn và thống kê hostname, nên có thể mất một lúc.


In [ ]:
with diagnostic_step('prepare'):
    import pyarrow.parquet as pq
    from collections import Counter
    from urllib.parse import urlsplit
    from IPython.display import display
    from datetime import datetime, timezone
    import time
    import uuid

    selected = pq.read_table(SOURCE_PATH, filters=[('id', '>=', ID_FROM), ('id', '<=', ID_TO)]).to_pylist()
    selected.sort(key=lambda row: row['id'])
    if not selected:
        raise ValueError('Không có URL nào trong khoảng ID đã chọn.')
    if len({row['id'] for row in selected}) != len(selected):
        raise ValueError('Nguồn có ID trùng trong khoảng chọn.')
    missing_count = ID_TO - ID_FROM + 1 - len(selected)
    print(f'Chọn {len(selected):,} URL; có {missing_count:,} ID không tồn tại trong khoảng.')
    print('Phân bố hostname:', dict(Counter(urlsplit(row['url']).hostname for row in selected)))
    display(selected[:5])

    LOG_DIR = RUN_DIR / 'logs'
    LOG_DIR.mkdir(exist_ok=True)

    def log_event(name, event):
        with (LOG_DIR / name).open('a', encoding='utf-8') as stream:
            stream.write(json.dumps(event, ensure_ascii=False) + '\n')

    def run_script(script, *arguments):
        # Output hiển thị trực tiếp để theo dõi tiến độ; không dùng shell.
        invocation_id = uuid.uuid4().hex
        started = time.monotonic()
        operation = str(arguments[0]) if script in (CRAWLER, CORPUS_RUNNER) else Path(script).stem
        log_event('invocations.jsonl', {
            'event': 'start', 'invocation_id': invocation_id, 'operation': operation,
            'at': datetime.now(timezone.utc).isoformat(), 'arguments': list(map(str, arguments)),
        })
        process = subprocess.Popen(
            [sys.executable, '-u', str(script), *map(str, arguments)],
            cwd=str(WORKDIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1,
        )
        returncode = None
        try:
            with (LOG_DIR / (Path(script).stem + '.log')).open('a', encoding='utf-8') as log:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                    if operation == 'crawl':
                        try:
                            progress = json.loads(line)
                        except ValueError:
                            progress = {}
                        if isinstance(progress, dict) and 'completed_this_invocation' in progress:
                            log_event('progress.jsonl', {
                                **progress, 'invocation_id': invocation_id,
                                'at': datetime.now(timezone.utc).isoformat(),
                                'elapsed_seconds': time.monotonic() - started,
                            })
                            refresh_diagnostics()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        finally:
            process.stdout.close()
            log_event('invocations.jsonl', {
                'event': 'end', 'invocation_id': invocation_id, 'operation': operation,
                'at': datetime.now(timezone.utc).isoformat(),
                'elapsed_seconds': time.monotonic() - started,
                'returncode': process.returncode,
            })
            refresh_diagnostics()
        if returncode:
            raise subprocess.CalledProcessError(returncode, process.args)

    manifest_path = RUN_DIR / 'manifest.json'
    if not manifest_path.exists():
        run_script(CRAWLER, 'prepare', '--source', SOURCE_PATH, '--out', RUN_DIR,
                   '--size', len(selected), '--id-start', ID_FROM, '--id-stop', ID_TO + 1)
    else:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        if manifest['source_sha256'] != SOURCE_SHA256 or manifest['selected'] != len(selected):
            raise ValueError('Manifest không khớp nguồn hoặc số URL đã chọn.')
        if not POSTPROCESS_ONLY and Path(manifest['source']).resolve() != SOURCE_PATH.resolve():
            raise ValueError('Đường dẫn nguồn của run cũ khác notebook này; dùng run mới.')
        sample_bytes = (RUN_DIR / 'sample.jsonl').read_bytes()
        if hashlib.sha256(sample_bytes).hexdigest() != manifest['sample_sha256']:
            raise ValueError('Hash sample của run cũ đã thay đổi.')
        with (RUN_DIR / 'sample.jsonl').open(encoding='utf-8') as stream:
            sample = [json.loads(line) for line in stream if line.strip()]
        if {r['id']: r['url'] for r in sample} != {r['id']: r['url'] for r in selected} or len(sample) != len(selected):
            raise ValueError('ID/URL của run cũ không khớp khoảng chọn.')
        print('Manifest hợp lệ; giữ checkpoint để resume.')


## 6. Crawl — chạy lại ô này để tiếp tục

PubMed mặc định dùng **EFetch GET**, **100 PMID/batch**, **1 worker**, nghỉ ít nhất **0,5 giây giữa request**.
`PUBMED_USE_EFETCH = True` hoạt động cả khi `BROWSER_TEST_MODE = 'off'`; run chỉ có PubMed không khởi động Chrome.
Giữ ID/URL gốc, ghép theo PMID trong XML; `text` có title và toàn bộ abstract cùng nhãn.
Các domain khác dùng Requests rồi Selenium theo cấu hình browser; `RESPECT_ROBOTS` áp dụng lượt crawl website.

Checkpoint XML và batch được lưu trước các record ID. Resume sửa các record còn thiếu mà không GET lại batch đã lưu.
`MAX_NEW` tính số URL corpus mới, gồm cả PubMed. API lỗi sau retry giữ batch pending và dừng để resume;
PMID không được API trả về hoặc không có abstract được ghi riêng, không đưa vào corpus accepted.
Xem `pubmed_results.csv`, `pubmed_missing_pmids.csv`, `pubmed_no_abstract.csv`, `pubmed_review.csv`,
`pubmed_requests.jsonl`, `pubmed_efetch_summary.json` và `raw/efetch/` trong ZIP đầy đủ.

Run mới có hậu tố `_efetch`. Nâng cấp run Selenium cũ bằng `RUN_TAG` mới và `RECOVERY_RUN_PATH`:
reuse raw hợp lệ của domain khác; PubMed HTML cũ được lấy mới qua EFetch.
Muốn thử lại PMID thiếu/no abstract cần run mới; resume giữ những kết quả XML hợp lệ đã hoàn tất.
`POSTPROCESS_ONLY = True` vẫn audit/export run cũ mà không crawl lại.

Tham khảo [NCBI EFetch](https://www.ncbi.nlm.nih.gov/books/NBK25499/).

Long Châu mặc định dùng `curl_cffi`/Chrome, 1 worker, delay 1,75 giây; không chạy Selenium cho domain này. Parser giữ sapo/nội dung chính; `/benh/` lấy đầy đủ các trường HTML y khoa từ JSON đã xác minh slug/canonical. Xem `longchau_results.csv`, `longchau_summary.json`; run mới có hậu tố `_longchau`, dùng recovery để tái trích HTML 200 cũ.


In [ ]:
with diagnostic_step('crawl'):
    if POSTPROCESS_ONLY:
        print('POSTPROCESS_ONLY: bỏ qua Requests/Selenium; giữ nguyên cấu hình và snapshot crawl.')
    else:
        # Pass credentials to the subprocess through its environment; settings/log contain none.
        for name, value in [('NCBI_EMAIL', NCBI_EMAIL), ('NCBI_API_KEY', NCBI_API_KEY)]:
            if value:
                os.environ[name] = value
        POSTPROCESS.invalidate(RUN_DIR)
        corpus_settings = {
            'user_agent': USER_AGENT, 'delay': DELAY, 'attempts': ATTEMPTS,
            'timeout': [PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT], 'budget': PAGE_REQUEST_BUDGET,
            'respect_robots': RESPECT_ROBOTS,
            'robots_options': {'robots_attempts': ROBOTS_ATTEMPTS,
                               'robots_connect_timeout': ROBOTS_CONNECT_TIMEOUT,
                               'robots_read_timeout': ROBOTS_READ_TIMEOUT,
                               'robots_retry_budget': ROBOTS_RETRY_BUDGET,
                               'robots_failure_ttl': ROBOTS_FAILURE_TTL},
            'browser_mode': BROWSER_TEST_MODE, 'browser_user_agent': BROWSER_USER_AGENT,
            'pubmed_selenium_first': PUBMED_SELENIUM_FIRST,
            'pubmed_use_efetch': PUBMED_USE_EFETCH,
            'longchau_use_curl': LONGCHAU_USE_CURL,
            'longchau': {'workers': LONGCHAU_WORKERS, 'delay': LONGCHAU_DELAY, 'attempts': LONGCHAU_ATTEMPTS,
                          'timeout': [LONGCHAU_CONNECT_TIMEOUT, LONGCHAU_READ_TIMEOUT],
                          'budget': LONGCHAU_REQUEST_BUDGET, 'impersonate': 'chrome',
                          'retry_failed_on_resume': LONGCHAU_RETRY_FAILED_ON_RESUME},
            'pubmed_efetch': {'batch_size': PUBMED_EFETCH_BATCH_SIZE, 'workers': PUBMED_EFETCH_WORKERS,
                               'gap_seconds': PUBMED_EFETCH_GAP_SECONDS, 'attempts': PUBMED_EFETCH_ATTEMPTS,
                               'timeout': [PUBMED_EFETCH_CONNECT_TIMEOUT, PUBMED_EFETCH_READ_TIMEOUT],
                               'budget': PUBMED_EFETCH_BUDGET},
            'chrome_binary': str(CHROME_BINARY) if CHROME_BINARY else None,
            'browser_load_timeout': BROWSER_LOAD_TIMEOUT, 'browser_wait_seconds': BROWSER_WAIT_SECONDS,
            'browser_attempts': BROWSER_ATTEMPTS,
        }
        settings_path = RUN_DIR / 'corpus_settings.json'
        settings_path.write_text(json.dumps(corpus_settings, ensure_ascii=False, indent=2), encoding='utf-8')
        arguments = ['crawl', '--out', RUN_DIR, '--workers', WORKERS, '--config', settings_path]
        if '--log-every' in CORPUS_RUNNER.read_text(encoding='utf-8'):
            arguments += ['--log-every', LOG_EVERY]
        if RECOVERY_RUN_PATH:
            arguments += ['--recovery-source', RECOVERY_RUN_PATH]
        if MAX_NEW is not None:
            arguments += ['--max-new', MAX_NEW]
        print('ZIP chẩn đoán tự cập nhật:', diagnostics_archive)
        run_script(CORPUS_RUNNER, *arguments)


## 7. Báo cáo, kiểm tra và xuất corpus

Audit chỉ chạy khi đủ checkpoint; raw rỗng hoặc HTML không parse được được ghi cờ chẩn đoán.
Sai hash, ID/URL hoặc thiếu raw của bản ghi extracted vẫn làm bước này thất bại.
Corpus chỉ xuất sau audit đạt. Thành công được ghi riêng trong postprocess_status.json;
run hết pending chưa đủ điều kiện thành công/upload. Nếu lỗi, notebook tạo ZIP checkpoint
đầy đủ và diagnostics trước khi báo lại lỗi gốc. Accepted là QC tự động, vẫn cần review nội dung.


In [ ]:
with diagnostic_step('report_audit_export'):
    summary = POSTPROCESS.run_postprocess(RUN_DIR, SOURCE_PATH, run_script)
    display({k: summary[k] for k in ['selected', 'completed', 'pending', 'accepted', 'review', 'rejected', 'status_counts']})
    if (RUN_DIR / 'access_summary.json').exists():
        display(json.loads((RUN_DIR / 'access_summary.json').read_text(encoding='utf-8')))
        print('Lỗi theo tổng URL dataset giảm dần:', RUN_DIR / 'WEB_ERRORS.md')
        print('Đối chiếu Requests/Selenium:', RUN_DIR / 'requests_vs_selenium.csv')
    if summary['pending']:
        print('Run chưa hoàn tất; chưa audit/export hoặc đánh dấu thành công. Chạy tiếp ô crawl.')
    elif summary['accepted'] > 0:
        corpus = pq.read_table(RUN_DIR / 'corpus.parquet')
        print(corpus.schema)
        display(corpus.select(['id', 'url', 'title']).slice(0, 5).to_pylist())
    else:
        print('Audit đạt; không có tài liệu accepted để xuất corpus. Xem QUALITY_REPORT.md.')


## 8. ZIP chẩn đoán — tích hợp trong Output

Chạy ô này để tạo lại gói debug từ checkpoint hiện có, kể cả khi crawl chưa xong.
Không cần file Python riêng, không tải lại URL. ZIP chứa:
- `overview.html`, `summary.json`: tiến độ, tốc độ/ETA, trạng thái HTTP và QC.
- CSV từng ID, lỗi/pending, request/retry và redirect.
- Log pip/crawl/Chrome, kết quả preflight, traceback notebook, manifest/cấu hình và phiên bản môi trường.
- Snapshot code, bằng chứng robots và tối đa 10 mẫu record/raw để kiểm tra parser.

Raw mẫu có ngân sách 10 MiB; các mẫu bị bỏ qua được ghi trong `sample_manifest.json`.
Giải nén ZIP và mở `overview.html`. Có thể gửi nguyên ZIP này để chẩn đoán lỗi.


In [ ]:
with diagnostic_step('diagnostics_archive'):
    from IPython.display import FileLink
    import zipfile

    refresh_diagnostics(strict=True)
    with zipfile.ZipFile(diagnostics_archive) as diagnostic_bundle:
        diagnostic_summary = json.loads(diagnostic_bundle.read('summary.json'))
    display({key: diagnostic_summary[key] for key in ['selected', 'completed', 'pending', 'quality_status_counts', 'timing']})
    print('ZIP chẩn đoán trong Output:', diagnostics_archive)
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))


## 9. Đóng gói corpus đầy đủ để tải về

ZIP được tạo cả khi xử lý thất bại; manifest ghi complete = False cho run chưa qua audit/export.
ZIP chứa cả raw, records, snapshot code và corpus để truy vết/resume; có thể khá lớn.
ZIP riêng `<run>_diagnostics.zip` nhỏ hơn: giải nén và mở `overview.html` để xem
tiến độ, tốc độ/ETA, HTTP status, lỗi và cờ chất lượng. Có CSV từng ID và log cài dependencies.
Tải file trong Output hoặc qua link bên dưới. Khi đổi session, cần giữ output hoặc
thêm run đã giải nén làm Kaggle Input và đặt `RESUME_RUN_PATH` trước khi chạy lại.
Resume không tự thử lại ID lỗi đã có checkpoint. Dùng `RECOVERY_RUN_PATH`
cùng `RUN_TAG` mới để thử lại lỗi bằng code/cấu hình mới và tái sử dụng raw thành công.


In [ ]:
with diagnostic_step('checkpoint_archive'):
    from IPython.display import FileLink

    archive = save_checkpoint_archive()
    notebook_manifest = json.loads((RUN_DIR / 'notebook_manifest.json').read_text(encoding='utf-8'))
    print('Run đã qua audit/export:', notebook_manifest['complete'])
    display(FileLink(str(Path(archive).relative_to(WORKDIR))))
    display(FileLink(str(diagnostics_archive.relative_to(WORKDIR))))
    if notebook_manifest['complete'] and notebook_manifest['accepted'] > 0:
        display(FileLink(str((RUN_DIR / 'corpus.parquet').relative_to(WORKDIR))))
        display(FileLink(str((RUN_DIR / 'corpus.jsonl').relative_to(WORKDIR))))


## 10. Tự upload output lên Hugging Face Dataset

Repo đích: **AncientWings/AncientWings_R2AI_STAGE3_ds**. Bật Internet và gắn Kaggle Secret
**HF_TOKEN** có quyền ghi vào repo. Run All sẽ tự chạy ô này sau khi xuất corpus/đóng ZIP.
Không cần đăng nhập qua widget hoặc tải output về máy.

Mỗi run được phân biệt bằng `RUN_DIR.name`: nguồn mới mặc định `ids_<ID_FROM>_<ID_TO>_synced_vi_en_zh`.
Recovery có tên riêng; nếu crawl lại cùng khoảng ID với code/cấu hình khác, đặt **RUN_TAG mới**
ở mục 1. Chạy lại ô upload cùng run sẽ cập nhật đúng các đường dẫn của run đó.

```text
corpus/<run>.parquet                       # Đầu vào RAG: chỉ tài liệu accepted
corpus_jsonl/<run>.jsonl                   # Cùng corpus, định dạng JSONL
archives/<run>.zip                         # Chỉ khi bật HF_UPLOAD_ARCHIVE = True
diagnostics/<run>_diagnostics.zip           # Gói chẩn đoán
manifests/<run>/notebook_manifest.json      # Khoảng ID và số lượng bản ghi
manifests/<run>/upload_manifest.json        # Danh sách file, kích thước và SHA-256
```

Mặc định upload corpus, diagnostics và manifest; không upload ZIP raw/checkpoint (`HF_UPLOAD_ARCHIVE = False`).
Mục 9 vẫn tạo ZIP đầy đủ trong Kaggle Output để tải về/resume; chỉ bật upload archive khi chủ động cần.
Đặt `HF_UPLOAD_ENABLED = False` nếu chỉ muốn giữ output Kaggle.
Run còn pending (ví dụ thử `MAX_NEW = 10`) sẽ **chưa upload**; chạy tiếp mục 6 rồi chạy lại mục 7–10.
Run hoàn tất nhưng không có accepted vẫn upload manifest/ZIP; không xuất corpus cũ còn sót lại.
Upload đọc file từ đĩa, không nạp cả ZIP vào RAM. Nếu mạng lỗi, ô thử lại tối đa 3 lần;
có thể chạy lại riêng mục 10 mà không crawl lại. Kết quả commit được lưu tại
`hf_upload_result.json` trong run. Repo đích phải tồn tại; token/repo lỗi sẽ được báo ở ô này.

Tài liệu: [Hugging Face upload](https://huggingface.co/docs/huggingface_hub/en/guides/upload)
và [Kaggle Secrets](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md#using-secrets-in-kernels).


In [ ]:
# ===== UPLOAD OUTPUT CỦA RUN NÀY LÊN HUGGING FACE =====
HF_UPLOAD_ENABLED = True
HF_REPO_ID = 'AncientWings/AncientWings_R2AI_STAGE3_ds'
HF_SECRET_NAME = 'HF_TOKEN'  # Tạo trong Kaggle: Add-ons > Secrets; không dán token vào code.
HF_UPLOAD_JSONL = True
HF_UPLOAD_ARCHIVE = False  # ZIP raw/checkpoint chỉ giữ trong Kaggle Output.
HF_UPLOAD_DIAGNOSTICS = True
HF_UPLOAD_ATTEMPTS = 3


def upload_current_run_to_hf():
    import hashlib
    import importlib.metadata
    import json
    from datetime import datetime, timezone
    from pathlib import Path
    import re
    import subprocess
    import sys
    import time
    import traceback
    from urllib.parse import quote

    if not HF_UPLOAD_ENABLED:
        print('Đã tắt upload Hugging Face; output vẫn được lưu trong Kaggle Output.')
        return {'status': 'disabled'}

    run_dir = Path(RUN_DIR)
    manifest_path = run_dir / 'notebook_manifest.json'
    if not manifest_path.is_file():
        raise RuntimeError('Chạy mục 7–9 để xuất corpus và đóng ZIP trước khi upload.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (manifest.get('id_from_inclusive'), manifest.get('id_to_inclusive')) != (ID_FROM, ID_TO):
        raise RuntimeError('Manifest không khớp khoảng ID hiện tại; chạy lại mục 7–9.')
    if manifest.get('dataset_revision') != DATASET_REVISION:
        raise RuntimeError('Manifest không khớp revision nguồn; chạy lại mục 7–9.')
    if manifest.get('pending', 1) != 0:
        print(f"Chưa upload: run còn {manifest.get('pending', '?')} URL pending. "
              'Chạy tiếp mục 6, rồi chạy lại mục 7–10.')
        return {'status': 'incomplete', 'run_name': run_dir.name}
    # Completion requires fresh audit/export evidence, even if a legacy manifest says complete.
    import importlib.util
    helper_path = run_dir / 'postprocess_snapshot.py'
    if not helper_path.is_file():
        raise RuntimeError('Thiếu bằng chứng audit/export mới; chạy lại mục 7–9 với POSTPROCESS_ONLY = True.')
    helper_spec = importlib.util.spec_from_file_location('hf_postprocess_gate', helper_path)
    helper = importlib.util.module_from_spec(helper_spec)
    helper_spec.loader.exec_module(helper)
    helper.require_ready(run_dir, manifest)
    if type(HF_UPLOAD_ATTEMPTS) is not int or HF_UPLOAD_ATTEMPTS < 1:
        raise ValueError('HF_UPLOAD_ATTEMPTS phải là số nguyên dương.')

    run_name = run_dir.name
    files = [(manifest_path, f'manifests/{run_name}/notebook_manifest.json')]
    if manifest.get('accepted', 0) > 0:
        files.append((run_dir / 'corpus.parquet', f'corpus/{run_name}.parquet'))
        if HF_UPLOAD_JSONL:
            files.append((run_dir / 'corpus.jsonl', f'corpus_jsonl/{run_name}.jsonl'))
    else:
        print('Run không có tài liệu accepted; chỉ upload manifest và các ZIP đã chọn.')
    if HF_UPLOAD_ARCHIVE:
        files.append((Path(WORKDIR) / f'{run_name}.zip', f'archives/{run_name}.zip'))
    if HF_UPLOAD_DIAGNOSTICS:
        files.append((Path(WORKDIR) / f'{run_name}_diagnostics.zip',
                      f'diagnostics/{run_name}_diagnostics.zip'))
    for local_path, _ in files:
        if not local_path.is_file() or local_path.stat().st_size == 0:
            raise RuntimeError(f'Thiếu output hoặc file rỗng: {local_path.name}. Chạy lại mục 7–9.')

    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(HF_SECRET_NAME)
    except Exception:
        raise RuntimeError(f'Tạo và bật Kaggle Secret {HF_SECRET_NAME!r} cho notebook này; '
                           'token cần quyền ghi vào dataset đích.') from None
    if not isinstance(token, str) or not token.strip():
        raise RuntimeError(f'Kaggle Secret {HF_SECRET_NAME!r} đang rỗng.')
    token = token.strip()

    try:
        from huggingface_hub import HfApi, CommitOperationAdd
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                        'huggingface_hub>=0.34,<3'], check=True)
        from huggingface_hub import HfApi, CommitOperationAdd

    def report_upload_error(error, stage, attempt=None):
        # Giữ thông tin chẩn đoán, nhưng che token/header xác thực và query của URL ký.
        def redact(value):
            value = str(value).replace(token, '[REDACTED_TOKEN]')
            value = re.sub(r'\bhf_[A-Za-z0-9]{8,}\b', '[REDACTED_TOKEN]', value)
            value = re.sub(r'(?i)\bBearer\s+[A-Za-z0-9._~+/=-]+',
                           'Bearer [REDACTED_TOKEN]', value)
            value = re.sub(r'(https?://)[^/\s@]+@', r'\1[REDACTED_AUTH]@', value)
            value = re.sub(r'''(https?://[^\s?<>"'\)\]]+)\?[^\s<>"'\)\]]+''',
                           r'\1?[REDACTED_QUERY]', value)
            value = re.sub(r'''(?i)(\bauthorization\b["']?\s*[:=]\s*["']?)'''
                           r'''(?:(?:Bearer|Basic)\s+)?[^\s"',;}]+''',
                           r'\1[REDACTED_AUTH]', value)
            value = re.sub(r'''(?i)(\b(?:access[_-]?token|token|api[_-]?key)\b'''
                           r'''["']?\s*[:=]\s*["']?)[^\s"',;}&]+''',
                           r'\1[REDACTED_TOKEN]', value)
            return value

        chain, seen, current, status = [], set(), error, None
        while current is not None and id(current) not in seen and len(chain) < 8:
            seen.add(id(current))
            message = redact(str(current))
            chain.append(f'{type(current).__name__}: {message}')
            code = getattr(getattr(current, 'response', None), 'status_code', None)
            if status is None and isinstance(code, int) and 100 <= code <= 599:
                status = code
            if status is None:
                # hf_xet có thể bọc HTTP error trong RuntimeError mà không có response.
                match = re.search(r'\b(?:HTTP(?:\s+(?:status|error|code|client|server)){0,4}'
                                  r'|status(?:\s+code)?)\s*[:=]?\s*\(?([1-5]\d{2})\b',
                                  message, re.IGNORECASE)
                if match:
                    status = int(match.group(1))
            current = current.__cause__ if current.__cause__ is not None else current.__context__
        versions = {}
        for package in ('huggingface-hub', 'hf-xet'):
            try:
                versions[package] = importlib.metadata.version(package)
            except importlib.metadata.PackageNotFoundError:
                versions[package] = 'not installed / unavailable'
        detail = '\nCaused by: '.join(chain)
        heading = (f'Hugging Face error | stage={stage} | '
                   f'attempt={attempt or "-"}/{HF_UPLOAD_ATTEMPTS} | HTTP {status or "?"}')
        print(heading)
        print(detail)
        print('SDK:', json.dumps(versions, ensure_ascii=False))
        diagnostic = {
            'time_utc': datetime.now(timezone.utc).isoformat(), 'stage': stage,
            'attempt': attempt, 'http_status': status, 'repo_id': HF_REPO_ID,
            'run_name': run_name, 'sdk_versions': versions, 'error_chain': chain,
        }
        log_path = run_dir / 'logs' / 'hf_upload_error.log'
        try:
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open('a', encoding='utf-8') as log:
                log.write(redact(json.dumps(diagnostic, ensure_ascii=False)) + '\n')
                log.write(redact(''.join(traceback.format_exception(
                    type(error), error, error.__traceback__))) + '\n')
            print('Log lỗi upload:', log_path)
        except OSError:
            print('Không ghi được file log; thông báo lỗi đã in ở trên.')
        return status, detail

    api = HfApi(token=token)  # Không login(), không lưu token vào file hay biến môi trường.
    try:
        api.repo_info(repo_id=HF_REPO_ID, repo_type='dataset')
    except Exception as error:
        status, detail = report_upload_error(error, 'repo_info')
        raise RuntimeError(f'Không truy cập được dataset {HF_REPO_ID} (HTTP {status or "?"}). '
                           f'\nLỗi gốc: {detail}\n'
                           'Kiểm tra Internet, repo đích và quyền của HF_TOKEN.') from None

    file_records = []
    for local_path, remote_path in files:
        digest = hashlib.sha256()
        with local_path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                digest.update(block)
        file_records.append({'path_in_repo': remote_path, 'bytes': local_path.stat().st_size,
                             'sha256': digest.hexdigest()})
    upload_manifest = {
        'format_version': 1,
        'source_notebook': 'crawl_corpus_kaggle_hf_upload.ipynb',
        'repo_id': HF_REPO_ID, 'run_name': run_name,
        'id_from_inclusive': ID_FROM, 'id_to_inclusive': ID_TO,
        'dataset_revision': manifest['dataset_revision'],
        'source_sha256': manifest['source_sha256'],
        'complete': True, 'corpus_policy': 'accepted_only',
        'counts': {key: manifest[key] for key in ('selected', 'completed', 'pending', 'accepted')},
        'prepared_at_utc': datetime.now(timezone.utc).isoformat(),
        'files': file_records,
    }
    upload_manifest_path = run_dir / 'hf_upload_manifest.json'
    upload_manifest_path.write_text(json.dumps(upload_manifest, ensure_ascii=False, indent=2) + '\n',
                                    encoding='utf-8')
    files.append((upload_manifest_path, f'manifests/{run_name}/upload_manifest.json'))
    print(f'Dataset đích: {HF_REPO_ID} | Run: {run_name} | Accepted: {manifest["accepted"]}')
    for local_path, remote_path in files:
        print(f'  {remote_path} ({local_path.stat().st_size / 1024**2:.2f} MiB)')

    for attempt in range(1, HF_UPLOAD_ATTEMPTS + 1):
        try:
            # Một commit cho corpus, manifest và ZIP của cùng run; chỉ thêm/cập nhật các path này.
            # Tạo lại operations khi retry vì SDK có thể thay đổi chúng trong quá trình upload.
            commit = api.create_commit(
                repo_id=HF_REPO_ID, repo_type='dataset',
                operations=[CommitOperationAdd(path_in_repo=remote, path_or_fileobj=str(local))
                            for local, remote in files],
                commit_message=f'Upload crawl {run_name} ({manifest["accepted"]} accepted)',
            )
            break
        except Exception as error:
            status, detail = report_upload_error(error, 'create_commit', attempt)
            retryable = status is None or status >= 500 or status in (408, 409, 429)
            if not retryable or attempt == HF_UPLOAD_ATTEMPTS:
                raise RuntimeError(f'Upload chưa thành công (HTTP {status or "?"}, '
                                   f'{type(error).__name__}).\nLỗi gốc: {detail}\n'
                                   'Xem logs/hf_upload_error.log để biết chi tiết; '
                                   'output local vẫn còn, có thể chạy lại riêng mục 10.') from None
            print(f'Upload thất bại (lượt {attempt}/{HF_UPLOAD_ATTEMPTS}); đang thử lại...')
            time.sleep(min(2 ** attempt, 30))

    result = {
        'status': 'uploaded', 'repo_id': HF_REPO_ID, 'run_name': run_name,
        'commit_url': commit.commit_url, 'commit_oid': commit.oid,
        'files': [remote for _, remote in files],
    }
    (run_dir / 'hf_upload_result.json').write_text(
        json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('Upload thành công:', commit.commit_url)
    for _, remote in files:
        print(f'https://huggingface.co/datasets/{HF_REPO_ID}/blob/main/{quote(remote, safe="/")}')
    return result


hf_upload_result = upload_current_run_to_hf()
